In [ ]:
from ecmwf.opendata import Client
import os
import xarray as xr
import pandas as pd
import numpy as np
import glob
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from datetime import datetime, timedelta
from shapely.geometry import Point, box
from tqdm import tqdm
import json
import geopandas as gpd
from matplotlib.offsetbox import (OffsetImage, AnnotationBbox)
import matplotlib.image as mpimg
import email.message
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import base64

def enviar_alerta(data_referencia, data_validade, maximo_global, municipio_maximo, passo):

    # Configurações
    email_origem = "nsh.cpdam@gmail.com"
    senha_app = "pxfboabhpydavuim"
    
    destinatarios = [
        #"willie.nascimento@sema.ma.gov.br",
        #"felipe.costa@sema.ma.gov.br" 
        "igor.morim@outlook.com",
    ]

    """
    # Cópia oculta (BCC) - opcional
    copia_oculta = [
        # "backup@example.com"
    ]
    """
    LIMITE_ALERTA = 40

    with open("../assets/images/logo/SEMA/2023/MARCA GOV SEMA BRANCA_RECORTE.png", "rb") as image_file:
        logo_base64 = base64.b64encode(image_file.read()).decode('utf-8')
    
    if maximo_global < LIMITE_ALERTA:
        print(f"    ⚠️ Precipitação ({maximo_global:.2f} mm) abaixo do limite ({LIMITE_ALERTA} mm)")
        print("    ❌ Alerta não enviado.")
        return False
    
    print(f"    ⚠️ Precipitação ({maximo_global:.2f} mm) acima do limite ({LIMITE_ALERTA} mm)")
    
    corpo_html = f"""
    <!DOCTYPE html>
    <html lang="pt-br">

    <head>

    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>Alerta Meteorológico</title>

    <style>

    body {{
        margin: 0;
        padding: 0;
        background-color: #edf2f7;
        font-family: Arial, Helvetica, sans-serif;
    }}

    table {{
        border-spacing: 0;
    }}

    .container {{
        width: 100%;
        background-color: #edf2f7;
        padding: 30px 0;
    }}

    .email-wrapper {{
        width: 100%;
        max-width: 820px;
        margin: auto;
        background: #ffffff;
        border-radius: 22px;
        overflow: hidden;
        box-shadow: 0 10px 40px rgba(15, 23, 42, 0.10);
    }}

    .header {{
        background:
            linear-gradient(
                135deg,
                #0b1220 0%,
                #132238 45%,
                #1d3557 100%
            );

        padding: 36px 40px;
    }}

    .header-table {{
        width: 100%;
    }}

    .header-left {{
        vertical-align: middle;
    }}

    .header-right {{
        width: 40%;
        text-align: right;
        vertical-align: middle;
    }}

    .logo-img {{
        max-width: 80%;
        height: auto;
    }}

    .system-name {{
        color: rgba(255,255,255,0.70);
        font-size: 12px;
        letter-spacing: 1.5px;
        text-transform: uppercase;
        margin-bottom: 8px;
    }}

    .header-title {{
        color: #ffffff;
        font-size: 32px;
        font-weight: 800;
        line-height: 1.2;
        margin-bottom: 10px;
    }}

    .header-subtitle {{
        color: rgba(255,255,255,0.72);
        font-size: 14px;
        line-height: 1.7;
    }}

    .alert-level {{
        display: inline-block;
        margin-top: 24px;
        padding: 12px 22px;
        border-radius: 40px;
        font-size: 13px;
        font-weight: bold;
        color: #ffffff;
        background:
            {'#7f1d1d' if maximo_global > 80 else '#b45309' if maximo_global > 60 else '#ca8a04'};
    }}

    .content {{
        padding: 40px;
    }}

    .alert-banner {{
        background:
            linear-gradient(
                135deg,
                #fff1f2 0%,
                #ffe4e6 100%
            );

        border-left: 6px solid #dc2626;
        border-radius: 18px;
        padding: 28px;
        margin-bottom: 32px;
    }}

    .alert-banner h2 {{
        margin: 0 0 12px 0;
        color: #991b1b;
        font-size: 26px;
        font-weight: 800;
    }}

    .alert-banner p {{
        margin: 0;
        color: #475569;
        font-size: 15px;
        line-height: 1.8;
    }}

    .grid {{
        width: 100%;
    }}

    .card {{
        background: #f8fafc;
        border: 1px solid #e2e8f0;
        border-radius: 18px;
        padding: 24px;
    }}

    .card-label {{
        font-size: 11px;
        color: #64748b;
        text-transform: uppercase;
        letter-spacing: 1px;
        margin-bottom: 10px;
        font-weight: bold;
    }}

    .card-value {{
        font-size: 30px;
        font-weight: 800;
        color: #0f172a;
    }}

    .card-unit {{
        font-size: 14px;
        color: #64748b;
    }}

    .precip {{
        color: #dc2626;
    }}

    .section-title {{
        font-size: 20px;
        font-weight: 800;
        color: #0f172a;
        margin: 42px 0 20px 0;
    }}

    .location-box {{
        background:
            linear-gradient(
                135deg,
                #eff6ff 0%,
                #dbeafe 100%
            );

        border: 1px solid #bfdbfe;
        border-radius: 20px;
        padding: 30px;
    }}

    .location-city {{
        font-size: 36px;
        font-weight: 800;
        color: #1e3a8a;
        margin-bottom: 14px;
    }}

    .coordinates {{
        color: #475569;
        font-size: 14px;
        line-height: 1.8;
    }}

    .map-button {{
        display: inline-block;
        margin-top: 24px;
        background:
            linear-gradient(
                135deg,
                #2563eb 0%,
                #1d4ed8 100%
            );

        color: #ffffff !important;
        text-decoration: none;
        padding: 14px 24px;
        border-radius: 12px;
        font-weight: bold;
        font-size: 14px;
    }}

    .recommendation {{
        margin-top: 36px;
        background: #fff7ed;
        border-left: 5px solid #ea580c;
        border-radius: 16px;
        padding: 24px;
    }}

    .recommendation h3 {{
        margin: 0 0 12px 0;
        color: #c2410c;
        font-size: 18px;
        font-weight: 800;
    }}

    .recommendation p {{
        margin: 0;
        color: #444;
        line-height: 1.8;
        font-size: 14px;
    }}

    .footer {{
        background:
            linear-gradient(
                135deg,
                #0b1220 0%,
                #111827 100%
            );

        padding: 34px;
        text-align: center;
    }}

    .footer-title {{
        color: #ffffff;
        font-size: 18px;
        font-weight: 700;
        margin-bottom: 12px;
    }}

    .footer-text {{
        margin: 5px 0;
        color: rgba(255,255,255,0.72);
        font-size: 12px;
        line-height: 1.8;
    }}

    .footer-divider {{
        margin: 10px auto;
        width: 80%;
        border-top: 1px solid rgba(255,255,255,0.08);
    }}

    .footer-warning {{
        color: rgba(255,255,255,0.48);
        font-size: 11px;
    }}

    @media only screen and (max-width: 600px) {{

        .content {{
            padding: 24px;
        }}

        .header {{
            padding: 28px 24px;
        }}

        .header-title {{
            font-size: 24px;
        }}

        .location-city {{
            font-size: 28px;
        }}

        .card-value {{
            font-size: 24px;
        }}

        .header-right {{
            width: 80px;
        }}

        .logo-img {{
            width: 68px;
        }}
    }}

    </style>
    </head>

    <body>

    <div class="container">

    <div class="email-wrapper">

        <!-- HEADER -->
        <div class="header">

            <table class="header-table">

                <tr>

                    <td class="header-left">

                        <div class="system-name">
                            Sistema de Monitoramento Meteorológico
                        </div>

                        <div class="header-title">
                            ALERTA METEOROLÓGICO
                        </div>

                        <div class="header-subtitle">

                            Núcleo de Segurança Climática - NSC<br>
                            Centro de Prevenção a Desastres Ambientais - CPDAm

                        </div>

                        <div class="alert-level">

                            {'🔴 RISCO CRÍTICO'
                                if maximo_global > 80 else
                            '🟠 RISCO ALTO'
                                if maximo_global > 60 else
                            '🟡 ATENÇÃO MODERADA'}

                        </div>

                    </td>

                    <td class="header-right">

                        <img
                            src="data:image/png;base64,{logo_base64}"
                            class="logo-img"
                            style="border-radius: 8px;"
                            alt="Logo"
                        >

                    </td>

                </tr>

            </table>

        </div>

        <!-- CONTENT -->
        <div class="content">

            <!-- ALERTA -->
            <div class="alert-banner">

                <h2>
                    Previsão de Precipitação Significativa
                </h2>

                <p>

                    Foi identificado cenário meteorológico favorável
                    à ocorrência de precipitação intensa sobre o estado
                    do Maranhão, com potencial para impactos hidrológicos,
                    alagamentos localizados, enxurradas e elevação do nível
                    de corpos hídricos.

                </p>

            </div>

            <!-- CARDS -->
            <table width="100%" class="grid">

                <tr>

                    <td width="48%" style="padding-bottom:15px;">

                        <div class="card">

                            <div class="card-label">
                                DATA DE REFERÊNCIA
                            </div>

                            <div class="card-value">
                                {data_referencia}
                            </div>

                        </div>

                    </td>

                    <td width="4%"></td>

                    <td width="48%" style="padding-bottom:15px;">

                        <div class="card">

                            <div class="card-label">
                                VALIDADE DA PREVISÃO
                            </div>

                            <div class="card-value">
                                {data_validade}
                            </div>

                        </div>

                    </td>

                </tr>

                <tr>

                    <td>

                        <div class="card">

                            <div class="card-label">
                                HORIZONTE DE PREVISÃO
                            </div>

                            <div class="card-value">

                                {passo}

                                <span class="card-unit">
                                    horas
                                </span>

                            </div>

                        </div>

                    </td>

                    <td></td>

                    <td>

                        <div class="card">

                            <div class="card-label">
                                PRECIPITAÇÃO MÁXIMA ESTIMADA
                            </div>

                            <div class="card-value">

                                {maximo_global:.1f}

                                <span class="card-unit">
                                    mm
                                </span>

                            </div>

                        </div>

                    </td>

                </tr>

            </table>

            <!-- LOCALIZAÇÃO -->
            <div class="section-title">
                Município com Maior Intensidade Prevista
            </div>

            <div class="location-box">

                <div class="location-city">

                    {municipio_maximo if municipio_maximo else "Não identificado"}

                </div>

                <div class="coordinates">

                    <strong>Latitude:</strong>
                    {lat_max:.5f}°<br>

                    <strong>Longitude:</strong>
                    {lon_max:.5f}°

                </div>

                <a
                    href="https://www.google.com/maps?q={lat_max:.6f},{lon_max:.6f}"
                    class="map-button"
                    target="_blank"
                >
                    🛰️ Visualizar Localização
                </a>

            </div>

            <!-- RECOMENDAÇÕES -->
            <div class="recommendation">

                <h3>
                    Recomendações Operacionais
                </h3>

                <p>

                    <ul>
                        <li>Evite deslocamentos desnecessários, especialmente em áreas sujeitas a alagamentos, encostas e margens de rios.</li>
                        <li>Não atravesse ruas alagadas, a força da água pode arrastar veículos e pessoas, além do risco de buracos e correnteza oculta.</li>
                        <li>Fique atento a sinais de deslizamento, como rachaduras no solo, inclinação de postes ou árvores, e água barrenta escorrendo pelo terreno.</li>
                        <li>Em caso de elevação rápida do nível de rios ou córregos, busque imediatamente um ponto alto e afastado da margem.</li>
                        <li>Não se abrigue sob árvores ou estruturas metálicas durante tempestades com raios.</li>
                    </ul>

                </p>

            </div>

        </div>

        <!-- FOOTER -->
        <div class="footer">

            <div class="footer-title">
                Secretaria de Estado do Meio Ambiente e Recursos Naturais - SEMA
            </div>

            <p class="footer-text">
                Centro de Prevenção a Desastres Ambientais - CPDAm
            </p>

            <p class="footer-text">
                Núcleo de Segurança Climática - NSC
            </p>

            <div class="footer-divider"></div>

            <p class="footer-text">
                Alerta gerado automaticamente a partir do modelo numérico ECMWF
            </p>

            <p class="footer-text">
                Emissão: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}
            </p>

            <p class="footer-text">
                São Luís • Maranhão • Brasil
            </p>

            <div class="footer-divider"></div>

            <p class="footer-warning">
                ⚠️ Mensagem automática institucional • Não responder este e-mail
            </p>

        </div>

    </div>
    </div>

    </body>
    </html>
    """
    msg = MIMEMultipart('alternative')
    msg['Subject'] = f"🚨 ALERTA METEOROLÓGICO - ({municipio_maximo if municipio_maximo else 'Não identificado'} {maximo_global:.0f} mm)"
    msg['From'] = email_origem
    msg['To'] = ", ".join(destinatarios)
    
    # Anexar corpo HTML
    parte_html = MIMEText(corpo_html, 'html', 'utf-8')
    msg.attach(parte_html)
    
    try:
        # Enviar email
        print("📧 Conectando ao servidor Gmail...")
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(email_origem, senha_app)
        s.send_message(msg)
        s.quit()
        
        print("    ✅ Alerta enviado com sucesso!")
        print(f"        Para: {', '.join(destinatarios)}")
        print(f"        Precipitação: {maximo_global:.2f} mm")
        print(f"        Município: {municipio_maximo if municipio_maximo else 'Não identificado'}")
        return True
        
    except Exception as e:
        print(f"❌ Erro ao enviar alerta: {e}")
        return False

client = Client(model="aifs-ens", source="aws", preserve_request_order=False)

parameters = ['tp']
type = ['cf']
filename = 'medium-rain-acc.grib'
temp_file = 'temp.grib'

tempos = [18, 12, 6, 0]
steps = [6, 12, 18, 24, 30, 36, 42, 48]
data = (datetime.now() - timedelta(hours=4)).strftime("%Y%m%d")
tempo_funcional = None

caminho = '../previsao/Dados/'
for tempo in tempos:
    try:
        print(f"🔍 Testando time={tempo} com step=6...")
        temp_test = f'{caminho}temp_test.grib'
        
        resultado = client.retrieve(
            date=data,
            time=tempo,
            step=6,
            stream="oper",
            type=type,
            levtype="sfc",
            param=parameters,
            target=temp_test)
        
        print(f"    ✅ Tempo {tempo} funcionou!\n")
        tempo_funcional = tempo

        if os.path.exists(temp_test):
            os.remove(temp_test)

        for arquivo in glob.glob(os.path.join(caminho, '*.tif')):
            os.remove(arquivo)
            
        for arquivo in glob.glob(os.path.join(caminho, '*.json')):
            os.remove(arquivo)
    
        break
        
    except Exception as e:
        print(f"    ❌ Tempo {tempo} falhou: {str(e)}")
        continue

if tempo_funcional is None:
    print("    ❌ Nenhum tempo funcionou!")
    exit()

print("Baixando todos os steps...")

if not os.path.exists(caminho):
    os.makedirs(caminho)

arquivos_baixados = []

for step in steps:
    try:
        temp_file = f'temp_step_{step:02d}.grib'

        print(75*"=")
        print(f"Baixando step={step}...")
        print(75*"=")

        resultado = client.retrieve(
            date=data,
            time=tempo_funcional,
            step=step,
            stream="oper",
            type=type,
            levtype="sfc",
            param=parameters,
            target=temp_file)
        
        ds = xr.open_dataset(temp_file, engine='cfgrib')

        data_ref = pd.to_datetime(ds.time.values)
        data_val = pd.to_datetime(ds.valid_time.values)
        passo = int(ds.step.values / 1e9 / 3600)
        ds.close()

        time = -3

        data_ref_string = (data_ref - timedelta(hours=time)).strftime("%d%m%Y_%H%M")
        data_validade_string = (data_val - timedelta(hours=time)).strftime("%d%m%Y_%H%M")

        final_file = f'{caminho}/{data_ref_string}_a_{data_validade_string}_{passo:02d}h.grib'

        if os.path.exists(final_file):
            os.remove(final_file)
        os.rename(temp_file, final_file)

        data_referencia = (data_ref - timedelta(hours=time)).strftime("%d/%m/%Y %H:%M")
        data_validade = (data_val - timedelta(hours=time)).strftime("%d/%m/%Y %H:%M")

        ds = xr.open_dataset(final_file, engine='cfgrib')

        # Criar dicionário com as informações
        chave_principal = f"{data_ref_string}_a_{data_validade_string}_{passo:02d}h.grib"
        caminho_arquivo = f'{caminho}/{data_ref_string}_a_{data_validade_string}_{passo:02d}h.tif'
        nome_arquivo = f"{data_ref_string}_a_{data_validade_string}_{passo:02d}h.tif"

        info_json = {
            nome_arquivo: {
            "data_referencia": (data_ref - timedelta(hours=time)).strftime("%d/%m/%Y %H:%M"),
            "data_validade": (data_val - timedelta(hours=time)).strftime("%d/%m/%Y %H:%M"),
            "passo_previsao_horas": passo,
            "caminho_arquivo": caminho_arquivo
            }
        }

        # Salvar como JSON
        json_path = f'{caminho}/historico_arquivos.json'

        # Carregar dados existentes ou criar lista vazia
        if os.path.exists(json_path):
            with open(json_path, 'r', encoding='utf-8') as f:
                try:
                    dados = json.load(f)
                    if not isinstance(dados, list):
                        dados = [dados]  # Converter para lista se for objeto único
                except:
                    dados = []
        else:
            dados = []

        # Adicionar novo registro
        dados.append(info_json)

        # Salvar lista atualizada
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(dados, f, ensure_ascii=False, indent=4)

        # Remove .idx da pasta atual e subpastas
        for idx_file in glob.glob('**/*.idx', recursive=True):
            try:
                os.remove(idx_file)
            except:
                pass  # Ignora erros se arquivo não existir

        # Limites estaduais
        BR_UF = '../docs/Shapefile/MA_Municipios_2022_limite/Limite_MA_2022.geojson'
        BR_UF = gpd.read_file(BR_UF).to_crs('EPSG:4326')
        maranhao_shape = BR_UF
        delimitacao = maranhao_shape

        # Limites municipais
        limite_municipal_MA = '../docs/Shapefile/MA_Municipios_2022/MA_Municipios_2022.geojson'
        limite_municipal_MA = gpd.read_file(limite_municipal_MA).to_crs('EPSG:4326')

        bounds = delimitacao.total_bounds

        fator_interpolacao = 10

        maranhao = ds.sel(
            latitude=slice(bounds[3] + 0.5, bounds[1] + 0.5),
            longitude=slice(bounds[0] + 0.5, bounds[2] + 0.5)
        )

        # Converter para mm (assumindo que tp está em metros)
        precip_ma = maranhao.tp# * 1000
        precip_ma = precip_ma * 1.3 #Correcao na previsão de chuva

        # Criar grade de pontos
        lons, lats = np.meshgrid(maranhao.longitude.values, maranhao.latitude.values)
        pontos_grib = gpd.GeoDataFrame(
            geometry=[Point(lon, lat) for lon, lat in zip(lons.ravel(), lats.ravel())],
            crs='EPSG:4326'
        )
        valores_precip = precip_ma.values.ravel()

        # Calcular para cada município
        precipitacao_municipios = []
        areas_municipios_km2 = []

        # Para o loop de municípios
        for idx, municipio in tqdm(limite_municipal_MA.iterrows(),
                                    total=len(limite_municipal_MA),
                                    desc="Processando municípios"):
            
            pontos_no_municipio = pontos_grib[pontos_grib.within(municipio.geometry)]

            if len(pontos_no_municipio) > 0:
                indices = pontos_no_municipio.index
                precip_media = valores_precip[indices].max() #mean() #max()
                precipitacao_municipios.append(precip_media)
            else:
                precipitacao_municipios.append(0)
        
        # Adicionar ao GeoDataFrame
        limite_municipal_MA['PRECIP_MEDIA_MM'] = precipitacao_municipios

        # Ordenar por precipitação
        municipios_ordenados = limite_municipal_MA.sort_values('PRECIP_MEDIA_MM', ascending=False)

        # Município com maior precipitação
        municipio_max = municipios_ordenados.iloc[0]
        print(f"\nMunicípio com maior precipitação: {municipio_max['NM_MUN']} ({municipio_max['PRECIP_MEDIA_MM']:.2f} mm)")

        # Criar geometria do Maranhão
        maranhao_geometry = delimitacao.geometry.union_all()

        # Verificar se os dados são 3D
        if precip_ma.ndim == 3:
            precip_ma = precip_ma.isel(time=0)

        # Aumentar resolução
        nova_lat = np.linspace(precip_ma.latitude.min().item(),
                            precip_ma.latitude.max().item(),
                            len(precip_ma.latitude) * fator_interpolacao)
        nova_lon = np.linspace(precip_ma.longitude.min().item(),
                            precip_ma.longitude.max().item(),
                            len(precip_ma.longitude) * fator_interpolacao)

        # Interpolar
        precip_interpolado = precip_ma.interp(latitude=nova_lat, longitude=nova_lon, method='linear')

        # Calcular resolução espacial
        res_lat_original = abs(precip_ma.latitude.values[1] - precip_ma.latitude.values[0])
        res_lat_final = abs(nova_lat[1] - nova_lat[0])
        km_por_grau = 111.32
        res_km_final = res_lat_final * km_por_grau

        # 1. Criar máscara do Maranhão usando shapely.vectorized (já existe)
        from shapely.vectorized import contains

        # Criar grade interpolada
        lons_int, lats_int = np.meshgrid(precip_interpolado.longitude.values, 
                                        precip_interpolado.latitude.values)

        # Criar máscara booleana para o Maranhão
        mascara_maranhao = contains(maranhao_geometry, lons_int, lats_int)

        # Aplicar máscara aos dados interpolados
        precip_mascarado = precip_interpolado.values.copy()
        precip_mascarado[~mascara_maranhao] = np.nan

        # 2. Calcular máximo global (o valor que você vê no QGIS)
        maximo_global = np.nanmax(precip_mascarado)

        # 3. Encontrar coordenadas do máximo
        if not np.isnan(maximo_global):
            idx_max = np.nanargmax(precip_mascarado)
            lat_max = lats_int.flatten()[idx_max]
            lon_max = lons_int.flatten()[idx_max]
            
            # Identificar em qual município está o máximo
            ponto_max = Point(lon_max, lat_max)
            municipio_maximo = None
            for idx, municipio in limite_municipal_MA.iterrows():
                if municipio.geometry.contains(ponto_max):
                    municipio_maximo = municipio['NM_MUN']
                    print(f"   ✅ Máximo GLOBAL do raster: {municipio_maximo} ({maximo_global:.2f} mm)")
                    break
        else:
            print("   ⚠️ Não foi possível encontrar o máximo")
            municipio_maximo = None
            lat_max, lon_max = None, None

        # 4. Comparar com o máximo por município
        max_por_municipio = municipios_ordenados.iloc[0]['PRECIP_MEDIA_MM']
        print("\nComparação:")
        print(f"  Máximo por município (média/max dentro do município): {max_por_municipio:.2f} mm")
        print(f"  Máximo GLOBAL do raster (ponto exato): {maximo_global:.2f} mm")
        print(f"  Diferença: {maximo_global - max_por_municipio:.2f} mm")
        data_validade = (data_val - timedelta(hours=time)).strftime("%d/%m/%Y %H:%M")

        if maximo_global >= 0:
            print("\n⚠️ Alerta: Verificando a intensidade e envio dos alertas...")
            enviar_alerta(data_referencia, data_validade, maximo_global, municipio_maximo, passo)

        for arquivo in glob.glob(os.path.join(caminho, '*.idx')):
            os.remove(arquivo)
        
        final_file = os.path.join(caminho, f'{data_ref_string}_a_{data_validade_string}_{passo:02d}h.grib')
        if os.path.exists(final_file):
            os.remove(final_file)

        try:
            # Adicionar CRS ao DataArray
            precip_interpolado.rio.write_crs("EPSG:4326", inplace=True)
            # Recortar o raster com a máscara
            precip_recortado = precip_interpolado.rio.clip(delimitacao.geometry.values, "EPSG:4326", drop=True)

            # Salvar como GeoTIFF
            raster_saida = f'{caminho}/{data_ref_string}_a_{data_validade_string}_{passo:02d}h.tif'
            precip_recortado.rio.to_raster(raster_saida)
            print("\n✅ Raster recortado e salvo com rioxarray!\n")

        except Exception as e:
            print(f"Erro no método rioxarray: {e}")
            print("\nTentando método alternativo com rasterio...")

    except Exception as e:
        print(f"  ❌ Step {step} falhou: {str(e)}")
        continue

🔍 Testando time=18 com step=6...
    ❌ Tempo 18 falhou: 404 Client Error: Not Found for url: https://ecmwf-forecasts.s3.eu-central-1.amazonaws.com/20260528/18z/aifs-ens/0p25/enfo/20260528180000-6h-enfo-cf.index
🔍 Testando time=12 com step=6...
    ❌ Tempo 12 falhou: 404 Client Error: Not Found for url: https://ecmwf-forecasts.s3.eu-central-1.amazonaws.com/20260528/12z/aifs-ens/0p25/enfo/20260528120000-6h-enfo-cf.index
🔍 Testando time=6 com step=6...


By downloading data from the ECMWF open data dataset, you agree to the terms: Attribution 4.0 International (CC BY 4.0). Please attribute ECMWF when downloading this data.
    ✅ Tempo 6 funcionou!

Baixando todos os steps...
Baixando step=6...


Processando municípios: 100%|██████████| 217/217 [00:00<00:00, 2598.51it/s]       



Município com maior precipitação: Apicum-Açu (5.67 mm)


C:\Users\igor.morim\AppData\Local\Temp\ipykernel_16612\2483595723.py:906: DeprecationWarning: The 'shapely.vectorized.contains' function is deprecated and will be removed a future version. Use 'shapely.contains_xy' instead (available since shapely 2.0.0).
  mascara_maranhao = contains(maranhao_geometry, lons_int, lats_int)


...


Município com maior precipitação: Mirinzal (17.71 mm)
   ✅ Máximo GLOBAL do raster: Mirinzal (17.63 mm)

Comparação:
  Máximo por município (média/max dentro do município): 17.71 mm
  Máximo GLOBAL do raster (ponto exato): 17.63 mm
  Diferença: -0.09 mm

⚠️ Alerta: Verificando a intensidade e envio dos alertas...
    ⚠️ Precipitação (17.63 mm) abaixo do limite (30 mm)
    ❌ Alerta não enviado.

✅ Raster recortado e salvo com rioxarray!

Baixando step=42...


Processando municípios: 100%|██████████| 217/217 [00:00<00:00, 1091.36it/s]        
C:\Users\igor.morim\AppData\Local\Temp\ipykernel_16612\2483595723.py:906: DeprecationWarning: The 'shapely.vectorized.contains' function is deprecated and will be removed a future version. Use 'shapely.contains_xy' instead (available since shapely 2.0.0).
  mascara_maranhao = contains(maranhao_geometry, lons_int, lats_int)



Município com maior precipitação: Presidente Médici (30.03 mm)
   ✅ Máximo GLOBAL do raster: Presidente Médici (29.75 mm)

Comparação:
  Máximo por município (média/max dentro do município): 30.03 mm
  Máximo GLOBAL do raster (ponto exato): 29.75 mm
  Diferença: -0.28 mm

⚠️ Alerta: Verificando a intensidade e envio dos alertas...
    ⚠️ Precipitação (29.75 mm) abaixo do limite (30 mm)
    ❌ Alerta não enviado.

✅ Raster recortado e salvo com rioxarray!

Baixando step=48...


Processando municípios: 100%|██████████| 217/217 [00:00<00:00, 928.36it/s]         
C:\Users\igor.morim\AppData\Local\Temp\ipykernel_16612\2483595723.py:906: DeprecationWarning: The 'shapely.vectorized.contains' function is deprecated and will be removed a future version. Use 'shapely.contains_xy' instead (available since shapely 2.0.0).
  mascara_maranhao = contains(maranhao_geometry, lons_int, lats_int)



Município com maior precipitação: Presidente Médici (34.74 mm)
   ✅ Máximo GLOBAL do raster: Presidente Médici (34.45 mm)

Comparação:
  Máximo por município (média/max dentro do município): 34.74 mm
  Máximo GLOBAL do raster (ponto exato): 34.45 mm
  Diferença: -0.29 mm

⚠️ Alerta: Verificando a intensidade e envio dos alertas...
    ⚠️ Precipitação (34.45 mm) acima do limite (30 mm)
📧 Conectando ao servidor Gmail...
    ✅ Alerta enviado com sucesso!
        Para: igor.morim@outlook.com
        Precipitação: 34.45 mm
        Município: Presidente Médici

✅ Raster recortado e salvo com rioxarray!



In [ ]:
import sys
import re
import requests
import io
import pandas as pd
import os
from pathlib import Path
from datetime import datetime, timedelta

# Configurações unificadas
CONFIG = {
    'arquivo_acidentes_ambientais': 'docs/Monitoramento_Emergencias_Ambientais_2025.xlsx',
    'sheet_url_acidentes_ambientais': "https://docs.google.com/spreadsheets/d/1NhMo7JwBnRK87JnRTDhAKF8ydTqLyl5C/edit?gid=171598562#gid=171598562",
}

def extrair_dados_guia(url):

    # Extrai o ID da planilha da URL
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    if not match:
        raise ValueError("Não foi possível extrair o ID da planilha da URL")
    
    sheet_id = match.group(1)
    
    # Constrói a URL de exportação
    xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    
    
    try:
        response = requests.get(xlsx_url, timeout=30)
        response.raise_for_status()
        
        # Lê o arquivo Excel
        xlsx_file = io.BytesIO(response.content)
        
        try:
            dados_guia = pd.read_excel(xlsx_file)
        except Exception as e:
            print(f"Erro ao ler Excel: {e}")
            print("Tentando com engine específica...")
            dados_guia = pd.read_excel(xlsx_file, engine='openpyxl')

        return dados_guia
        
    except requests.exceptions.RequestException as e:
        raise Exception(f"Erro ao baixar a planilha: {e}")
    except Exception as e:
        raise Exception(f"Erro ao processar a planilha: {e}")
    
def processar_datas(df):
    """Processa os dados para análise mensal"""
    # Converte a coluna de data/hora
    df['Data do acidente'] = pd.to_datetime(df['Data do acidente'], errors='coerce')
    return df


def salvar_planilha_local(df, caminho_arquivo):
    """Salva o DataFrame como arquivo Excel local"""
    try:
        caminho = Path(caminho_arquivo)
        caminho.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(caminho_arquivo, index=False)
        print(f"Planilha salva com sucesso em: {caminho_arquivo}")
        return True
    except Exception as e:
        print(f"Erro ao salvar planilha: {e}")
        return False
    


def main():
    """Função principal do sistema integrado"""
    try:
        url_planilha = CONFIG['sheet_url_acidentes_ambientais']
        caminho_arquivo = CONFIG['arquivo_acidentes_ambientais']

        # Busca os dados da guia específica
        dados_formulario = extrair_dados_guia(url_planilha)

        # Processa dados mensais
        dados_processados = processar_datas(dados_formulario)

        # SALVA A PLANILHA COMPLETA
        salvar_planilha_local(dados_processados, caminho_arquivo)
        
        # Verifica se há dados
        if len(dados_formulario) == 0:
            print("\n⚠️  AVISO: A planilha está vazia!")
        else:
            print("\n✅ Dados carregados com sucesso!")
        
    except Exception as e:
        print(f"\n❌ Erro: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Planilha salva com sucesso em: ../docs/Monitoramento_Emergencias_Ambientais_2025.xlsx

✅ Dados carregados com sucesso!


In [15]:
import sys
import re
import requests
import io
import pandas as pd
import os
from pathlib import Path
from datetime import datetime, timedelta

# Configurações unificadas
CONFIG = {
    'arquivo_acidentes_ambientais': '../docs/Monitoramento_Emergencias_Ambientais_2025.xlsx',
    'sheet_url_acidentes_ambientais': "https://docs.google.com/spreadsheets/d/1NhMo7JwBnRK87JnRTDhAKF8ydTqLyl5C/edit?gid=171598562#gid=171598562",
}

def extrair_dados_guia(url):

    # Extrai o ID da planilha da URL
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    if not match:
        raise ValueError("Não foi possível extrair o ID da planilha da URL")
    
    sheet_id = match.group(1)
    
    # Constrói a URL de exportação
    xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    
    # Faz o download da planilha
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        response = requests.get(xlsx_url, headers=headers, timeout=30)
        response.raise_for_status()
        
        # Lê o arquivo Excel
        xlsx_file = io.BytesIO(response.content)
        
        try:
            dados_guia = pd.read_excel(xlsx_file)
        except Exception as e:
            print(f"Erro ao ler Excel: {e}")
            print("Tentando com engine específica...")
            dados_guia = pd.read_excel(xlsx_file, engine='openpyxl')
        
        print(f"Planilha baixada com sucesso! Shape: {dados_guia.shape}")
        return dados_guia
        
    except requests.exceptions.RequestException as e:
        raise Exception(f"Erro ao baixar a planilha: {e}")
    except Exception as e:
        raise Exception(f"Erro ao processar a planilha: {e}")

def main():
    """Função principal do sistema integrado"""
    try:
        url_planilha = CONFIG['sheet_url_acidentes_ambientais']

        # Busca os dados da guia específica
        dados_formulario = extrair_dados_guia(url_planilha)

        # Informações sobre os dados
        print(f"\n{'='*50}")
        print(f"Total de linhas: {len(dados_formulario)}")
        print(f"Total de colunas: {len(dados_formulario.columns)}")
        print("\nColunas disponíveis:")
        for i, col in enumerate(dados_formulario.columns, 1):
            print(f"  {i:2d}. {col}")
        
        print("\nPrimeiras 5 linhas:")
        print(dados_formulario.head())
        
        print("\nInformações do DataFrame:")
        print(dados_formulario.info())
        
        # Verifica se há dados
        if len(dados_formulario) == 0:
            print("\n⚠️  AVISO: A planilha está vazia!")
        else:
            print("\n✅ Dados carregados com sucesso!")
        
    except Exception as e:
        print(f"\n❌ Erro: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Planilha baixada com sucesso! Shape: (62, 13)

Total de linhas: 62
Total de colunas: 13

Colunas disponíveis:
   1. Nº
   2. Fonte
   3. Data do acidente
   4. Nome da Empresa
   5. Tipo de Sinistro
   6. Descrição do Impacto Ambiental
   7. Localização do acidente
   8. Latitude
   9. Longitude
  10. Nº do SEI
  11. Status do Processo
  12. Observações
  13. ÓRGÃOS COMPETENTES

Primeiras 5 linhas:
    Nº       Fonte     Data do acidente  \
0  1.0       Email  2025-03-10 00:00:00   
1  2.0       Email  2025-03-11 00:00:00   
2  3.0       Email  2025-05-12 00:00:00   
3  4.0  Reportagem  2025-05-22 00:00:00   
4  5.0         NaN  2025-07-08 00:00:00   

                           Nome da Empresa Tipo de Sinistro  \
0                                     VALE        Vazamento   
1                               TRANSPETRO        Vazamento   
2  Graciliano Transportes e Logistica LTDA        Vazamento   
3                      Auto Posto ESA LTDA     Derramamento   
4                       

In [10]:
import sys
import re
import requests
import io
import pandas as pd
import os
from pathlib import Path
from datetime import datetime, timedelta

# Configurações unificadas
CONFIG = {
    'arquivo_acidentes_ambientais': '../docs/Monitoramento_Emergencias_Ambientais_2025.xlsx',
    'sheet_url_acidentes_ambientais': "https://docs.google.com/spreadsheets/d/1NhMo7JwBnRK87JnRTDhAKF8ydTqLyl5C/edit?gid=171598562#gid=171598562",
}

def extrair_dados_guia(url):

    # Extrai o ID da planilha da URL
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    if not match:
        raise ValueError("Não foi possível extrair o ID da planilha da URL")
    
    sheet_id = match.group(1)
    
    # Extrai o GID (ID da guia/aba) se existir
    gid_match = re.search(r'[?&]gid=(\d+)', url)
    gid_param = f"&gid={gid_match.group(1)}" if gid_match else ""
    print(f"ID da planilha: {sheet_id}, GID: {gid_param}")
    
    # Constrói a URL de exportação
    xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx{gid_param}"
    
    print(f"Baixando planilha de: {xlsx_url}")
    
    # Faz o download da planilha
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        response = requests.get(xlsx_url, headers=headers, timeout=30)
        response.raise_for_status()
        
        # Verifica se o conteúdo é realmente um arquivo Excel
        content_type = response.headers.get('Content-Type', '')
        if 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet' not in content_type:
            print(f"Aviso: Content-Type inesperado: {content_type}")
            print("Verificando se é um arquivo Excel válido...")
        
        # Lê o arquivo Excel
        xlsx_file = io.BytesIO(response.content)
        
        # Tenta ler com diferentes engines se necessário
        try:
            dados_guia = pd.read_excel(xlsx_file)
        except Exception as e:
            print(f"Erro ao ler Excel: {e}")
            print("Tentando com engine específica...")
            dados_guia = pd.read_excel(xlsx_file, engine='openpyxl')
        
        print(f"Planilha baixada com sucesso! Shape: {dados_guia.shape}")
        return dados_guia
        
    except requests.exceptions.RequestException as e:
        raise Exception(f"Erro ao baixar a planilha: {e}")
    except Exception as e:
        raise Exception(f"Erro ao processar a planilha: {e}")

def main():
    """Função principal do sistema integrado"""
    try:
        url_planilha = CONFIG['sheet_url_acidentes_ambientais']
        print(f"URL da planilha: {url_planilha}")

        # Busca os dados da guia específica
        print("Baixando dados do Google Sheets...")
        dados_formulario = extrair_dados_guia(url_planilha)

        # Informações sobre os dados
        print(f"\n{'='*50}")
        print(f"Total de linhas: {len(dados_formulario)}")
        print(f"Total de colunas: {len(dados_formulario.columns)}")
        print(f"\nColunas disponíveis:")
        for i, col in enumerate(dados_formulario.columns, 1):
            print(f"  {i:2d}. {col}")
        
        print(f"\nPrimeiras 5 linhas:")
        print(dados_formulario.head())
        
        print(f"\nInformações do DataFrame:")
        print(dados_formulario.info())
        
        # Verifica se há dados
        if len(dados_formulario) == 0:
            print("\n⚠️  AVISO: A planilha está vazia!")
        else:
            print(f"\n✅ Dados carregados com sucesso!")
        
    except Exception as e:
        print(f"\n❌ Erro: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

URL da planilha: https://docs.google.com/spreadsheets/d/1NhMo7JwBnRK87JnRTDhAKF8ydTqLyl5C/edit?gid=171598562#gid=171598562
Baixando dados do Google Sheets...
ID da planilha: 1NhMo7JwBnRK87JnRTDhAKF8ydTqLyl5C, GID: &gid=171598562
Baixando planilha de: https://docs.google.com/spreadsheets/d/1NhMo7JwBnRK87JnRTDhAKF8ydTqLyl5C/export?format=xlsx&gid=171598562
Planilha baixada com sucesso! Shape: (190, 13)

Total de linhas: 190
Total de colunas: 13

Colunas disponíveis:
   1. Nº
   2. Fonte
   3. Data do acidente
   4. Nome da Empresa
   5. Tipo de Sinistro
   6. Descrição do Impacto Ambiental
   7. Localização do acidente
   8. Latitude
   9. Longitude
  10. Nº do SEI
  11. Status do Processo
  12. Observações
  13. ÓRGÃOS COMPETENTES

Primeiras 5 linhas:
    Nº       Fonte     Data do acidente  \
0  1.0       Email  2025-03-10 00:00:00   
1  2.0       Email  2025-03-11 00:00:00   
2  3.0       Email  2025-05-12 00:00:00   
3  4.0  Reportagem  2025-05-22 00:00:00   
4  5.0         NaN  2025

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import ColorInterp
import numpy as np
from PIL import Image
import math
import concurrent.futures
from pathlib import Path
import json

class WebTileGenerator:
    """Gerador de tiles web otimizado"""
    
    def __init__(self, input_path, output_dir, tile_size=256):
        self.input_path = Path(input_path)
        self.output_dir = Path(output_dir)
        self.tile_size = tile_size
        
        # Criar diretórios
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
    def process_image(self):
        """Processa a imagem principal"""
        print("📥 Carregando imagem...")
        
        with rasterio.open(self.input_path) as src:
            # Reprojetar para Web Mercator se necessário
            if src.crs and src.crs.to_epsg() != 3857:
                print("🔄 Reprojetando para EPSG:3857...")
                self.reproject_to_web_mercator(src)
            else:
                self.original_data = src.read()
                self.profile = src.profile.copy()
                self.bounds = src.bounds
                
        print(f"✅ Imagem carregada: {self.original_data.shape}")
        
    def reproject_to_web_mercator(self, src):
        """Reprojeita para Web Mercator (EPSG:3857)"""
        transform, width, height = calculate_default_transform(
            src.crs, 'EPSG:3857', 
            src.width, src.height,
            *src.bounds
        )
        
        # Preparar array de destino
        dst_data = np.zeros((src.count, height, width), dtype=src.dtypes[0])
        
        # Reprojetar
        reproject(
            source=rasterio.band(src, range(1, src.count + 1)),
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs='EPSG:3857',
            resampling=Resampling.cubic
        )
        
        self.original_data = dst_data
        self.profile = src.profile.copy()
        self.profile.update({
            'crs': 'EPSG:3857',
            'transform': transform,
            'width': width,
            'height': height
        })
        self.bounds = rasterio.transform.array_bounds(height, width, transform)
        
    def calculate_zoom_levels(self, max_pixels=4096):
        """Calcula níveis de zoom automaticamente"""
        height, width = self.original_data.shape[1], self.original_data.shape[2]
        
        # Zoom máximo baseado na resolução
        max_dim = max(width, height)
        max_zoom = math.ceil(math.log(max_dim / self.tile_size, 2))
        
        # Zoom mínimo (mostrar toda imagem)
        min_zoom = max(0, max_zoom - 6)  # Mostrar 6 níveis acima
        
        return min_zoom, max_zoom
    
    def create_tile_pyramid(self, min_zoom, max_zoom):
        """Cria pirâmide de tiles"""
        print(f"🏗️ Criando pirâmide de zoom {min_zoom} a {max_zoom}...")
        
        # Processar cada nível de zoom
        for zoom in range(max_zoom, min_zoom - 1, -1):
            print(f"  Zoom {zoom}...", end=' ')
            self.create_zoom_level(zoom)
            print("✅")
    
    def create_zoom_level(self, zoom):
        """Cria todos os tiles para um nível de zoom"""
        zoom_dir = self.output_dir / str(zoom)
        zoom_dir.mkdir(exist_ok=True)
        
        # Calcular dimensões para este zoom
        scale_factor = 2 ** (max_zoom - zoom)
        scaled_height = self.original_data.shape[1] // scale_factor
        scaled_width = self.original_data.shape[2] // scale_factor
        
        if scaled_width == 0 or scaled_height == 0:
            return
        
        # Redimensionar imagem
        if self.original_data.shape[0] >= 3:  # RGB
            img_data = self.original_data[:3]
            img_data = np.transpose(img_data, (1, 2, 0))
            img = Image.fromarray(img_data, 'RGB')
        else:  # Grayscale
            img = Image.fromarray(self.original_data[0], 'L')
        
        img_resized = img.resize((scaled_width, scaled_height), Image.Resampling.LANCZOS)
        
        # Calcular número de tiles
        tiles_x = math.ceil(scaled_width / self.tile_size)
        tiles_y = math.ceil(scaled_height / self.tile_size)
        
        # Criar tiles com processamento paralelo
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            futures = []
            for x in range(tiles_x):
                for y in range(tiles_y):
                    futures.append(
                        executor.submit(
                            self.create_single_tile,
                            img_resized, zoom, x, y, 
                            scaled_width, scaled_height
                        )
                    )
            
            # Aguardar conclusão
            for future in concurrent.futures.as_completed(futures):
                future.result()
    
    def create_single_tile(self, img, zoom, x, y, total_width, total_height):
        """Cria um único tile"""
        # Diretório da coluna
        col_dir = self.output_dir / str(zoom) / str(x)
        col_dir.mkdir(exist_ok=True)
        
        # Calcular coordenadas
        left = x * self.tile_size
        upper = y * self.tile_size
        right = min(left + self.tile_size, total_width)
        lower = min(upper + self.tile_size, total_height)
        
        # Cortar tile
        tile_img = img.crop((left, upper, right, lower))
        
        # Se o tile for menor que 256x256, adicionar padding
        if tile_img.size != (self.tile_size, self.tile_size):
            new_img = Image.new('RGB' if img.mode == 'RGB' else 'L', 
                               (self.tile_size, self.tile_size))
            new_img.paste(tile_img, (0, 0))
            tile_img = new_img
        
        # Salvar com otimização
        tile_path = col_dir / f"{y}.png"
        tile_img.save(tile_path, 'PNG', optimize=True, compress_level=9)
        
        return tile_path
    
    def create_metadata(self, min_zoom, max_zoom):
        """Cria arquivos de metadados"""
        print("📄 Criando metadados...")
        
        # Calcular bounds em WGS84 para Leaflet
        if self.profile['crs'].to_epsg() == 3857:
            # Converter de 3857 para 4326
            import pyproj
            transformer = pyproj.Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True)
            left, bottom = transformer.transform(self.bounds.left, self.bounds.bottom)
            right, top = transformer.transform(self.bounds.right, self.bounds.top)
        else:
            left, bottom, right, top = self.bounds.left, self.bounds.bottom, self.bounds.right, self.bounds.top
        
        # tilejson.json
        tilejson = {
            "tilejson": "3.0.0",
            "name": self.input_path.stem,
            "description": f"Ortofoto {self.input_path.stem}",
            "version": "1.0.0",
            "attribution": "© Dados Municipais",
            "scheme": "xyz",
            "tiles": ["./{z}/{x}/{y}.png"],
            "minzoom": min_zoom,
            "maxzoom": max_zoom,
            "bounds": [left, bottom, right, top],
            "center": [(left + right) / 2, (bottom + top) / 2, min_zoom]
        }
        
        (self.output_dir / 'tilejson.json').write_text(
            json.dumps(tilejson, indent=2)
        )
        
        # leaflet.html
        self.create_leaflet_viewer(left, bottom, right, top, min_zoom)
        
        print("✅ Metadados criados")
    
    def create_leaflet_viewer(self, left, bottom, right, top, min_zoom):
        """Cria visualizador Leaflet"""
        center_lat = (bottom + top) / 2
        center_lng = (left + right) / 2
        
        html = f"""<!DOCTYPE html>
<html>
<head>
    <title>{self.input_path.stem}</title>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <style>
        #map {{ height: 100vh; width: 100%; }}
    </style>
</head>
<body>
    <div id="map"></div>
    
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <script>
        var map = L.map('map').setView([{center_lat}, {center_lng}], {min_zoom});
        
        L.tileLayer('./{{z}}/{{x}}/{{y}}.png', {{
            minZoom: {min_zoom},
            maxZoom: {max_zoom},
            bounds: [[{bottom}, {left}], [{top}, {right}]],
            noWrap: true,
            attribution: 'Ortofoto {self.input_path.stem}'
        }}).addTo(map);
    </script>
</body>
</html>"""
        
        (self.output_dir / 'index.html').write_text(html)

def main():
    """Função principal"""
    # Configurações
    INPUT_TIF = "../docs/DEM/Bacabal/Bacabal (MA)_Orto.tif"
    OUTPUT_DIR = "web_tiles_auto"
    
    # Criar gerador
    generator = WebTileGenerator(INPUT_TIF, OUTPUT_DIR)
    
    # Processar
    generator.process_image()
    
    # Calcular zoom levels automaticamente
    min_zoom, max_zoom = generator.calculate_zoom_levels()
    print(f"📐 Níveis de zoom: {min_zoom} a {max_zoom}")
    
    # Criar tiles
    generator.create_tile_pyramid(min_zoom, max_zoom)
    
    # Criar metadados
    generator.create_metadata(min_zoom, max_zoom)
    
    print(f"\n🎉 Tiles web criados com sucesso!")
    print(f"📁 Pasta: {OUTPUT_DIR}")
    print(f"\n🌐 Para visualizar:")
    print(f"cd {OUTPUT_DIR} && python -m http.server 8000")
    print(f"Acesse: http://localhost:8000")

if __name__ == "__main__":
    main()

NameError: name 'math' is not defined

In [1]:
# Funciona para imagem georeferenciadas

import rasterio
from rasterio.enums import Resampling

# Abrir o TIFF
with rasterio.open('Orto.tif') as src:
    # Ler dados
    data = src.read()
    
    # Criar perfil para PNG
    profile = src.profile
    profile.update(driver='PNG', dtype='uint8')
    
    # Salvar como PNG
    with rasterio.open('output.png', 'w', **profile) as dst:
        dst.write(data)

In [ ]:
import os
import re
import pdfplumber

# Configurações
pasta_pdfs = "../database/alertas/ano/2025/"

for arquivo in os.listdir(pasta_pdfs):

    condicoes = [
        arquivo.endswith(".pdf"),
        "ALERTA HIDROLÓGICO" in arquivo,
        "_2025" not in arquivo,
    ]
    if all(condicoes):

        caminho_arquivo = os.path.join(pasta_pdfs, arquivo)
        with pdfplumber.open(caminho_arquivo) as pdf:
            texto_completo = ""
            for pagina in pdf.pages:
                texto_completo += pagina.extract_text() + "\n"
                print(f"{arquivo}")
        
        # Extrair informações usando regex
        #padrao = r"Alerta Hidrológico Nº (\d+/\d{4}) - (.+?) - Nível: (.+?)\nData de Emissão: (.+?)\n"
        #correspondencias = re.findall(padrao, texto_completo)
        
        #for correspondencia in correspondencias:
        #    numero_alerta, localidade, nivel, data_emissao = correspondencia
        #    print(f"Número do Alerta: {numero_alerta}")
        #    print(f"Localidade: {localidade}")
        #    print(f"Nível: {nivel}")
        #    print(f"Data de Emissão: {data_emissao}")
        #    print("-" * 40)

In [3]:
import os
import re
import pdfplumber

def extrair_cota_pdf(caminho_pdf):
    """Extrai apenas a cota/nível atual do rio do PDF"""
    try:
        with pdfplumber.open(caminho_pdf) as pdf:
            texto_completo = ""
            for pagina in pdf.pages:
                texto = pagina.extract_text()
                if texto:
                    texto_completo += texto + "\n"
        
        # Padrões para encontrar a cota (nível atual do rio)
        padroes_cota = [
            # Padrão 1: "chegou a X metros" ou "atingiu X metros"
            r'chegou a (\d+[.,]\d+) metros',
            r'atingiu (\d+[.,]\d+) metros',
            r'chegou a (\d+[.,]\d+) m',
            r'atingiu (\d+[.,]\d+) m',
            
            # Padrão 2: "nível do Rio ... X metros"
            r'nível do (?:Rio|rio)[^0-9]*(\d+[.,]\d+) metros',
            
            # Padrão 3: "X metros as XX:XXh"
            r'(\d+[.,]\d+) metros as \d{2}:\d{2}h'
        ]
        
        cota_encontrada = None
        
        for padrao in padroes_cota:
            match = re.search(padrao, texto_completo, re.IGNORECASE)
            if match:
                cota_encontrada = match.group(1)
                # Substituir vírgula por ponto para padronizar
                cota_encontrada = cota_encontrada.replace(',', '.')
                break
        
        return cota_encontrada
        
    except Exception as e:
        print(f"Erro ao ler {caminho_pdf}: {e}")
        return None

def processar_cotas_pdfs(pasta_pdfs):
    """Processa todos os PDFs e extrai as cotas"""
    resultados = []
    
    for arquivo in os.listdir(pasta_pdfs):
        if arquivo.endswith(".pdf") and "ALERTA HIDROLÓGICO" in arquivo:
            caminho_completo = os.path.join(pasta_pdfs, arquivo)
            
            # Extrair cota do PDF
            cota = extrair_cota_pdf(caminho_completo)
            
            resultado = {
                'arquivo': arquivo,
                'cota_metros': cota
            }
            
            resultados.append(resultado)
            
            # Mostrar resultado
            if cota:
                print(f"{arquivo}: Cota = {cota} metros")
            else:
                print(f"{arquivo}: Cota NÃO ENCONTRADA")
    
    return resultados

# Configurações
pasta_pdfs = "../database/alertas/ano/2025/"

# Verificar se pdfplumber está instalado
try:
    import pdfplumber
except ImportError:
    print("Instalando pdfplumber...")
    import subprocess
    subprocess.check_call(["pip", "install", "pdfplumber"])
    import pdfplumber

# Processar os PDFs
print("Extraindo cotas dos PDFs...\n")
resultados_cotas = processar_cotas_pdfs(pasta_pdfs)

# Resumo
print("\n" + "="*60)
print(f"Total de PDFs processados: {len(resultados_cotas)}")

cotas_encontradas = sum(1 for r in resultados_cotas if r['cota_metros'])
print(f"Cotas extraídas com sucesso: {cotas_encontradas}")
print(f"Cotas não encontradas: {len(resultados_cotas) - cotas_encontradas}")

# Salvar em CSV simples
with open("cotas_extraidas.csv", "w", encoding="utf-8") as f:
    f.write("Arquivo;Cota (metros)\n")
    for resultado in resultados_cotas:
        cota = resultado['cota_metros'] if resultado['cota_metros'] else "NÃO ENCONTRADA"
        f.write(f"{resultado['arquivo']};{cota}\n")

print("Resultados salvos em 'cotas_extraidas.csv'")

Extraindo cotas dos PDFs...

N° 01_25 - ALERTA HIDROLÓGICO - ATENÇÃO DE SECA - Luzilândia - rio Parnaíba - 01_01_2025_18_06.pdf: Cota NÃO ENCONTRADA
N° 111_25 - ALERTA HIDROLÓGICO  - EMERGÊNCIA DE SECA - Vargem Grande - rio Munim - 30_09_2025_13_33.pdf: Cota = 0.70 metros
N° 12_25 - ALERTA HIDROLÓGICO - ALERTA DE CHEIA - Barra do Corda - rio Mearim - 16_01_2025_09_59.pdf: Cota NÃO ENCONTRADA
N° 13_25 - ALERTA HIDROLÓGICO - ALERTA DE CHEIA - Balsas - rio Balsas - 16_01_2025_10_16.pdf: Cota NÃO ENCONTRADA
N° 14_25 - ALERTA HIDROLÓGICO - ATENÇÃO DE CHEIA - Grajaú - rio Grajaú - 16_01_2025_12_08.pdf: Cota NÃO ENCONTRADA
N° 15_25 - ALERTA HIDROLÓGICO - ALERTA DE CHEIA - Grajaú - rio Grajaú - 17_01_2025_00_24.pdf: Cota NÃO ENCONTRADA
N° 16_25 - ALERTA HIDROLÓGICO - ATENÇÃO DE CHEIA - Mirador - rio Itapecuru - 17_01_2025_09_55.pdf: Cota NÃO ENCONTRADA
N° 17_25 - ALERTA HIDROLÓGICO - EMERGÊNCIA DE CHEIA - Barra do Corda - rio Mearim - 19_01_2025_05_07.pdf: Cota NÃO ENCONTRADA
N° 21_25 - ALERTA

In [ ]:
import os

def extrair_nomes_alertas(pasta_pdfs):
    resultados = []
    
    for arquivo in os.listdir(pasta_pdfs):
        if arquivo.endswith(".pdf"):
            # Remover a extensão .pdf
            nome_sem_ext = arquivo[:-4]
            
            # Verificar se contém "ALERTA HIDROLÓGICO"
            if "ALERTA HIDROLÓGICO" in nome_sem_ext:
                # Formatar exatamente como no exemplo
                texto_formatado = nome_sem_ext  # Já está no formato correto
                resultados.append(texto_formatado)
    
    return resultados

# Configurar o caminho da pasta
pasta_pdfs = "../database/alertas/ano/2025/"

# Processar os arquivos
alertas = extrair_nomes_alertas(pasta_pdfs)

# Printar os resultados
for alerta in alertas:
    print(alerta)

# Salvar em arquivo de texto
#with open("alertas_hidrologicos.txt", "w", encoding="utf-8") as f:
#    for alerta in alertas:
#        f.write(alerta + "\n")

print(f"\nTotal de alertas encontrados: {len(alertas)}")
#print("Resultados salvos em 'alertas_hidrologicos.txt'")

N° 01_25 - ALERTA HIDROLÓGICO - ATENÇÃO DE SECA (LUZILÂNDIA - RIO PARNAÍBA)
N° 111_25 - ALERTA HIDROLÓGICO  - EMERGÊNCIA DE SECA - Vargem Grande - rio Munim - 30_09_2025_13_33
N° 12_25 - ALERTA HIDROLÓGICO - ALERTA DE CHEIA (BARRA DO CORDA - RIO MEARIM)
N° 13_25 - ALERTA HIDROLÓGICO - ALERTA DE CHEIA (BALSAS - RIO BALSAS)
N° 14_25 - ALERTA HIDROLÓGICO - ATENÇÃO DE CHEIA (GRAJAÚ - RIO GRAJAÚ)
N° 15_25 - ALERTA HIDROLÓGICO - ALERTA DE CHEIA (GRAJAÚ - RIO GRAJAÚ)
N° 16_25 - ALERTA HIDROLÓGICO - ATENÇÃO DE CHEIA (MIRADOR - RIO ITAPECURU)
N° 17_25 - ALERTA HIDROLÓGICO - EMERGÊNCIA DE CHEIA (BARRA DO CORDA - RIO MEARIM)
N° 21_25 - ALERTA HIDROLÓGICO  - ATENÇÃO DE CHEIA (MIRADOR - RIO ITAPECURU)
N° 31_25 - ALERTA HIDROLÓGICO  - ATENÇÃO DE CHEIA (CAROLINA - RIO MANOEL ALVES GRANDE)
N° 35_25 - ALERTA HIDROLÓGICO  - ATENÇÃO DE CHEIA (BACABAL - RIO MEARIM)
N° 37_25 - ALERTA HIDROLÓGICO - ATENÇÃO DE CHEIA - São Luiz Gonzaga - rio Mearim - 06_03_2025_16_43
N° 40_25 - ALERTA HIDROLÓGICO - ATENÇÃO DE

In [ ]:
import sys
import re
import requests
import io
import pandas as pd
import os
from pathlib import Path
from datetime import datetime, timedelta

# Configurações unificadas
CONFIG = {
    'arquivo_observa': '../monitor_seca/rede_observadores/Observadores.xlsx',
    'sheet_url_observa': "https://docs.google.com/spreadsheets/d/1X1b8U2gsiSqIcGIcDO1kvlPAoD1xg3c4tY7v4l-1N6Y/edit?rtpof=true&sd=true&gid=1996667901#gid=1996667901",
    'geojson_ma': '../docs/dados.geojson',  # Arquivo GeoJSON com os municípios do MA
    'intervalo_inicio': '12/2024',  # DD/MM/AAAA ou MM/AAAA
    'intervalo_fim': '12/2026'      # DD/MM/AAAA ou MM/AAAA
}

def extrair_dados_guia(url, nome_guia="Respostas ao formulário 2"):
    """Extrai dados de uma guia específica da planilha Google Sheets"""
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    
    if not match:
        raise ValueError("Não foi possível extrair o ID da planilha da URL")
    
    sheet_id = match.group(1)
    xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    response = requests.get(xlsx_url, timeout=30)
    response.raise_for_status()

    xlsx_file = io.BytesIO(response.content)
    
    try:
        dados_guia = pd.read_excel(xlsx_file, sheet_name=nome_guia)
        print(f"Guia '{nome_guia}' carregada com sucesso: {len(dados_guia)} linhas")
        return dados_guia
    except ValueError as e:
        todas_guias = pd.read_excel(xlsx_file, sheet_name=None)
        guias_disponiveis = list(todas_guias.keys())
        raise ValueError(f"Guia '{nome_guia}' não encontrada. Guias disponíveis: {guias_disponiveis}") from e

def processar_dados_mensais(df):
    """Processa os dados para análise mensal"""
    # Converte a coluna de data/hora
    df['Carimbo de data/hora'] = pd.to_datetime(df['Carimbo de data/hora'], errors='coerce')
    
    # Extrai mês e ano
    df['Ano'] = df['Carimbo de data/hora'].dt.year
    df['Mes'] = df['Carimbo de data/hora'].dt.month
    df['Mes_Ano'] = df['Carimbo de data/hora'].dt.to_period('M')
    
    # Remove linhas com datas inválidas
    df = df.dropna(subset=['Carimbo de data/hora'])
    
    return df

def converter_para_periodo(data_str):
    """Converte string de data para período (MM/AAAA)"""
    try:
        # Tenta converter como MM/AAAA
        if len(data_str.split('/')) == 2:
            mes, ano = data_str.split('/')
            return pd.Period(year=int(ano), month=int(mes), freq='M')
        # Tenta converter como DD/MM/AAAA
        elif len(data_str.split('/')) == 3:
            dia, mes, ano = data_str.split('/')
            return pd.Period(year=int(ano), month=int(mes), freq='M')
    except Exception as e:
        raise ValueError(f"Formato de data inválido: {data_str}. Use MM/AAAA ou DD/MM/AAAA") from e

def filtrar_por_intervalo(df, inicio, fim):
    """Filtra o DataFrame pelo intervalo de datas especificado"""
    # Converte as strings para períodos
    periodo_inicio = converter_para_periodo(inicio)
    periodo_fim = converter_para_periodo(fim)
    
    print(f"Filtrando dados de {periodo_inicio} a {periodo_fim}")
    
    # Filtra os dados pelo intervalo
    df_filtrado = df[(df['Mes_Ano'] >= periodo_inicio) & (df['Mes_Ano'] <= periodo_fim)]
    
    print(f"Dados após filtro: {len(df_filtrado)} linhas")
    print(f"Períodos encontrados no intervalo: {sorted(df_filtrado['Mes_Ano'].unique())}")
    
    return df_filtrado

def gerar_dados_mapa_mensal(df, intervalo_inicio, intervalo_fim):
    """Gera dados para o mapa mensal com agrupamento por mês"""
    # Filtra os dados pelo intervalo
    df_filtrado = filtrar_por_intervalo(df, intervalo_inicio, intervalo_fim)
    
    if len(df_filtrado) == 0:
        print("⚠️  Nenhum dado encontrado no intervalo especificado!")
        return {
            'intervalo_inicio': intervalo_inicio,
            'intervalo_fim': intervalo_fim,
            'grupos_mensais': {},
            'total_respostas': 0
        }
    
    # Agrupa por mês e município
    grupos_mensais = {}
    
    # Pega todos os períodos únicos no intervalo filtrado
    periodos_unicos = sorted(df_filtrado['Mes_Ano'].unique())
    
    for periodo in periodos_unicos:
        # Filtra dados do mês específico
        df_mes = df_filtrado[df_filtrado['Mes_Ano'] == periodo]
        
        # Lista de municípios que responderam neste mês
        municipios_mes = df_mes['Município'].unique().tolist()
        
        # Contagem de respostas por município neste mês
        contagem_municipios = df_mes.groupby('Município').size().to_dict()
        
        # Contagem de respostas por instituição neste mês
        contagem_instituicoes = df_mes.groupby('Instituição').size().to_dict()
        
        # Lista de instituições únicas que responderam neste mês
        instituicoes_mes = df_mes['Instituição'].unique().tolist()
        
        # MUNICÍPIOS POR INSTITUIÇÃO COM CONTAGEM DE VOTOS
        municipios_por_instituicao = {}
        contagem_municipios_por_instituicao = {}
        
        for instituicao in instituicoes_mes:
            # Filtra os dados da instituição específica
            df_instituicao = df_mes[df_mes['Instituição'] == instituicao]
            
            # Lista de municípios únicos que responderam para esta instituição
            municipios_instituicao = df_instituicao['Município'].unique().tolist()
            municipios_por_instituicao[instituicao] = municipios_instituicao
            
            # CONTAGEM DE VOTOS POR MUNICÍPIO PARA ESTA INSTITUIÇÃO
            contagem_por_municipio_inst = df_instituicao.groupby('Município').size().to_dict()
            contagem_municipios_por_instituicao[instituicao] = contagem_por_municipio_inst
        
        # Adiciona ao grupo mensal
        grupos_mensais[str(periodo)] = {
            'municipios_responderam': municipios_mes,
            'total_respostas': len(df_mes),
            'contagem_por_municipio': contagem_municipios,
            'total_municipios': len(municipios_mes),
            'instituicoes_responderam': instituicoes_mes,
            'contagem_por_instituicao': contagem_instituicoes,
            'total_instituicoes': len(instituicoes_mes),
            'municipios_por_instituicao': municipios_por_instituicao,
            'contagem_municipios_por_instituicao': contagem_municipios_por_instituicao  # NOVO CAMPO
        }
        
        print(f"📅 {periodo}: {len(municipios_mes)} municípios, {len(instituicoes_mes)} instituições, {len(df_mes)} respostas")
        
        # Mostra detalhes dos municípios por instituição com contagem de votos
        for instituicao, municipios in municipios_por_instituicao.items():
            contagens = contagem_municipios_por_instituicao[instituicao]
            print(f"   🏛️ {instituicao}: {len(municipios)} municípios")
            for municipio in municipios:
                votos = contagens.get(municipio, 0)
                print(f"      📍 {municipio}: {votos} resposta(s)")
    
    # Estatísticas gerais do período
    total_municipios_unicos = df_filtrado['Município'].nunique()
    total_instituicoes_unicas = df_filtrado['Instituição'].nunique()
    
    print(f"\n📊 Total de períodos com dados: {len(grupos_mensais)}")
    print(f"🏙️ Municípios únicos no período: {total_municipios_unicos}")
    print(f"🏛️ Instituições únicas no período: {total_instituicoes_unicas}")
    print(f"📝 Total de respostas no intervalo: {len(df_filtrado)}")
    
    return {
        'intervalo_inicio': intervalo_inicio,
        'intervalo_fim': intervalo_fim,
        'grupos_mensais': grupos_mensais,
        'total_respostas': len(df_filtrado),
        'total_municipios_unicos': total_municipios_unicos,
        'total_instituicoes_unicas': total_instituicoes_unicas,
        'periodos_com_dados': [str(p) for p in periodos_unicos]
    }

def salvar_dados_mapa(dados_mapa):
    """Salva os dados do mapa em arquivo JSON para uso no frontend"""
    caminho_json = '../monitor_seca/rede_observadores/dados_mapa_observadores.json'
    
    caminho = Path(caminho_json)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    
    import json
    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(dados_mapa, f, ensure_ascii=False, indent=2)
    
    print(f"Dados do mapa salvos em: {caminho_json}")

def main():
    """Função principal do sistema integrado"""
    try:
        url_planilha = CONFIG['sheet_url_observa']
        caminho_salvar = CONFIG['arquivo_observa']
        intervalo_inicio = CONFIG['intervalo_inicio']
        intervalo_fim = CONFIG['intervalo_fim']
        
        # Busca os dados da guia específica
        dados_formulario = extrair_dados_guia(url_planilha, "Respostas ao formulário 2")

        print("\nDados da guia 'Respostas ao formulário 2':")
        print(f"Total de linhas: {len(dados_formulario)}")
        
        # Filtra apenas as colunas desejadas
        dados_filtrados = dados_formulario.loc[:, ['Carimbo de data/hora', 'Município', 'Instituição']]
        
        # Processa dados mensais
        dados_processados = processar_dados_mensais(dados_filtrados)
        
        # Gera dados para o mapa com o intervalo especificado
        dados_mapa = gerar_dados_mapa_mensal(dados_processados, intervalo_inicio, intervalo_fim)
        
        # Salva dados do mapa
        salvar_dados_mapa(dados_mapa)
        
        # SALVA A PLANILHA COMPLETA
        sucesso = salvar_planilha_local(dados_filtrados, caminho_salvar)
        
        if sucesso:
            print("\n✅ Processo concluído!")
            print(f"📊 Intervalo de referência: {intervalo_inicio} a {intervalo_fim}")
            print(f"📅 Períodos com dados: {dados_mapa['periodos_com_dados']}")
            print(f"🏙️ Municípios únicos no período: {dados_mapa['total_municipios_unicos']}")
            print(f"🏛️ Instituições únicas no período: {dados_mapa['total_instituicoes_unicas']}")
            print(f"📝 Total de respostas: {dados_mapa['total_respostas']}")
            
            # Mostra resumo por mês
            for periodo, dados in dados_mapa['grupos_mensais'].items():
                print(f"  • {periodo}: {dados['total_municipios']} municípios, {dados['total_instituicoes']} instituições, {dados['total_respostas']} respostas")
                
                # Mostra detalhes dos municípios por instituição no resumo final
                for instituicao, municipios in dados['municipios_por_instituicao'].items():
                    contagens = dados['contagem_municipios_por_instituicao'][instituicao]
                    print(f"    🏛️ {instituicao}: {len(municipios)} municípios")
                    for municipio in municipios:
                        votos = contagens.get(municipio, 0)
                        print(f"       📍 {municipio}: {votos} resposta(s)")
        else:
            print("\n❌ Erro ao salvar o arquivo")
        
    except Exception as e:
        print(f"Erro: {e}")
        sys.exit(1)

def salvar_planilha_local(df, caminho_arquivo):
    """Salva o DataFrame como arquivo Excel local"""
    try:
        caminho = Path(caminho_arquivo)
        caminho.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(caminho_arquivo, index=False)
        print(f"Planilha salva com sucesso em: {caminho_arquivo}")
        return True
    except Exception as e:
        print(f"Erro ao salvar planilha: {e}")
        return False

if __name__ == "__main__":
    main()

Guia 'Respostas ao formulário 2' carregada com sucesso: 937 linhas

Dados da guia 'Respostas ao formulário 2':
Total de linhas: 937
Filtrando dados de 2024-12 a 2026-12
Dados após filtro: 421 linhas
Períodos encontrados no intervalo: [Period('2024-12', 'M'), Period('2025-01', 'M'), Period('2025-02', 'M'), Period('2025-04', 'M'), Period('2025-05', 'M'), Period('2025-06', 'M'), Period('2025-07', 'M'), Period('2025-08', 'M'), Period('2025-09', 'M'), Period('2025-10', 'M'), Period('2025-11', 'M')]
📅 2024-12: 38 municípios, 4 instituições, 60 respostas
   🏛️ SENAR: 22 municípios
      📍 São José de Ribamar: 1 resposta(s)
      📍 Rosário: 1 resposta(s)
      📍 Humberto de Campos: 2 resposta(s)
      📍 Lima Campos: 2 resposta(s)
      📍 Pedreiras: 1 resposta(s)
      📍 Igarapé Grande: 2 resposta(s)
      📍 São Mateus do Maranhão: 1 resposta(s)
      📍 Peritoró: 1 resposta(s)
      📍 Balsas: 4 resposta(s)
      📍 São João do Carú: 2 resposta(s)
      📍 Igarapé do Meio: 1 resposta(s)
      📍 Bo

In [4]:
import sys
import re
import requests
import io
import pandas as pd
import os
from pathlib import Path
from datetime import datetime, timedelta

# Configurações unificadas
CONFIG = {
    'arquivo_observa': '../monitor_seca/rede_observadores/Observadores.xlsx',
    'sheet_url_observa': "https://docs.google.com/spreadsheets/d/1X1b8U2gsiSqIcGIcDO1kvlPAoD1xg3c4tY7v4l-1N6Y/edit?rtpof=true&sd=true&gid=1996667901#gid=1996667901",
    'geojson_ma': '../docs/dados.geojson',  # Arquivo GeoJSON com os municípios do MA
    'intervalo_inicio': '03/2025',  # DD/MM/AAAA ou MM/AAAA
    'intervalo_fim': '12/2025'      # DD/MM/AAAA ou MM/AAAA
}

def extrair_dados_guia(url, nome_guia="Respostas ao formulário 2"):
    """Extrai dados de uma guia específica da planilha Google Sheets"""
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    
    if not match:
        raise ValueError("Não foi possível extrair o ID da planilha da URL")
    
    sheet_id = match.group(1)
    xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    response = requests.get(xlsx_url, timeout=30)
    response.raise_for_status()

    xlsx_file = io.BytesIO(response.content)
    
    try:
        dados_guia = pd.read_excel(xlsx_file, sheet_name=nome_guia)
        print(f"Guia '{nome_guia}' carregada com sucesso: {len(dados_guia)} linhas")
        return dados_guia
    except ValueError as e:
        todas_guias = pd.read_excel(xlsx_file, sheet_name=None)
        guias_disponiveis = list(todas_guias.keys())
        raise ValueError(f"Guia '{nome_guia}' não encontrada. Guias disponíveis: {guias_disponiveis}") from e

def processar_dados_mensais(df):
    """Processa os dados para análise mensal"""
    # Converte a coluna de data/hora
    df['Carimbo de data/hora'] = pd.to_datetime(df['Carimbo de data/hora'], errors='coerce')
    
    # Extrai mês e ano
    df['Ano'] = df['Carimbo de data/hora'].dt.year
    df['Mes'] = df['Carimbo de data/hora'].dt.month
    df['Mes_Ano'] = df['Carimbo de data/hora'].dt.to_period('M')
    
    # Remove linhas com datas inválidas
    df = df.dropna(subset=['Carimbo de data/hora'])
    
    return df

def converter_para_periodo(data_str):
    """Converte string de data para período (MM/AAAA)"""
    try:
        # Tenta converter como MM/AAAA
        if len(data_str.split('/')) == 2:
            mes, ano = data_str.split('/')
            return pd.Period(year=int(ano), month=int(mes), freq='M')
        # Tenta converter como DD/MM/AAAA
        elif len(data_str.split('/')) == 3:
            dia, mes, ano = data_str.split('/')
            return pd.Period(year=int(ano), month=int(mes), freq='M')
    except Exception as e:
        raise ValueError(f"Formato de data inválido: {data_str}. Use MM/AAAA ou DD/MM/AAAA") from e

def filtrar_por_intervalo(df, inicio, fim):
    """Filtra o DataFrame pelo intervalo de datas especificado"""
    # Converte as strings para períodos
    periodo_inicio = converter_para_periodo(inicio)
    periodo_fim = converter_para_periodo(fim)
    
    print(f"Filtrando dados de {periodo_inicio} a {periodo_fim}")
    
    # Filtra os dados pelo intervalo
    df_filtrado = df[(df['Mes_Ano'] >= periodo_inicio) & (df['Mes_Ano'] <= periodo_fim)]
    
    print(f"Dados após filtro: {len(df_filtrado)} linhas")
    print(f"Períodos encontrados no intervalo: {sorted(df_filtrado['Mes_Ano'].unique())}")
    
    return df_filtrado

def gerar_dados_mapa_mensal(df, intervalo_inicio, intervalo_fim):
    """Gera dados para o mapa mensal com agrupamento por mês"""
    # Filtra os dados pelo intervalo
    df_filtrado = filtrar_por_intervalo(df, intervalo_inicio, intervalo_fim)
    
    if len(df_filtrado) == 0:
        print("⚠️  Nenhum dado encontrado no intervalo especificado!")
        return {
            'intervalo_inicio': intervalo_inicio,
            'intervalo_fim': intervalo_fim,
            'grupos_mensais': {},
            'total_respostas': 0
        }
    
    # Agrupa por mês e município
    grupos_mensais = {}
    
    # Pega todos os períodos únicos no intervalo filtrado
    periodos_unicos = sorted(df_filtrado['Mes_Ano'].unique())
    
    for periodo in periodos_unicos:
        # Filtra dados do mês específico
        df_mes = df_filtrado[df_filtrado['Mes_Ano'] == periodo]
        
        # Lista de municípios que responderam neste mês
        municipios_mes = df_mes['Município'].unique().tolist()
        
        # Contagem de respostas por município neste mês
        contagem_municipios = df_mes.groupby('Município').size().to_dict()
        
        # Contagem de respostas por instituição neste mês
        contagem_instituicoes = df_mes.groupby('Instituição').size().to_dict()
        
        # Lista de instituições únicas que responderam neste mês
        instituicoes_mes = df_mes['Instituição'].unique().tolist()
        
        # NOVO: Municípios que responderam por instituição neste mês
        municipios_por_instituicao = {}
        for instituicao in instituicoes_mes:
            # Filtra os dados da instituição específica
            df_instituicao = df_mes[df_mes['Instituição'] == instituicao]
            # Pega os municípios únicos que responderam para esta instituição
            municipios_instituicao = df_instituicao['Município'].unique().tolist()
            municipios_por_instituicao[instituicao] = municipios_instituicao
        
        # Adiciona ao grupo mensal
        grupos_mensais[str(periodo)] = {
            'municipios_responderam': municipios_mes,
            'total_respostas': len(df_mes),
            'contagem_por_municipio': contagem_municipios,
            'total_municipios': len(municipios_mes),
            'instituicoes_responderam': instituicoes_mes,
            'contagem_por_instituicao': contagem_instituicoes,
            'total_instituicoes': len(instituicoes_mes),
            'municipios_por_instituicao': municipios_por_instituicao  # NOVO CAMPO ADICIONADO
        }
        
        print(f"📅 {periodo}: {len(municipios_mes)} municípios, {len(instituicoes_mes)} instituições, {len(df_mes)} respostas")
        
        # Mostra detalhes dos municípios por instituição
        for instituicao, municipios in municipios_por_instituicao.items():
            print(f"   🏛️ {instituicao}: {len(municipios)} municípios - {', '.join(municipios)}")
    
    # Estatísticas gerais do período
    total_municipios_unicos = df_filtrado['Município'].nunique()
    total_instituicoes_unicas = df_filtrado['Instituição'].nunique()
    
    print(f"\n📊 Total de períodos com dados: {len(grupos_mensais)}")
    print(f"🏙️ Municípios únicos no período: {total_municipios_unicos}")
    print(f"🏛️ Instituições únicas no período: {total_instituicoes_unicas}")
    print(f"📝 Total de respostas no intervalo: {len(df_filtrado)}")
    
    return {
        'intervalo_inicio': intervalo_inicio,
        'intervalo_fim': intervalo_fim,
        'grupos_mensais': grupos_mensais,
        'total_respostas': len(df_filtrado),
        'total_municipios_unicos': total_municipios_unicos,
        'total_instituicoes_unicas': total_instituicoes_unicas,
        'periodos_com_dados': [str(p) for p in periodos_unicos]
    }

def salvar_dados_mapa(dados_mapa):
    """Salva os dados do mapa em arquivo JSON para uso no frontend"""
    caminho_json = '../monitor_seca/rede_observadores/dados_mapa_observadores.json'
    
    caminho = Path(caminho_json)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    
    import json
    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(dados_mapa, f, ensure_ascii=False, indent=2)
    
    print(f"Dados do mapa salvos em: {caminho_json}")

def main():
    """Função principal do sistema integrado"""
    try:
        url_planilha = CONFIG['sheet_url_observa']
        caminho_salvar = CONFIG['arquivo_observa']
        intervalo_inicio = CONFIG['intervalo_inicio']
        intervalo_fim = CONFIG['intervalo_fim']
        
        # Busca os dados da guia específica
        dados_formulario = extrair_dados_guia(url_planilha, "Respostas ao formulário 2")

        print("\nDados da guia 'Respostas ao formulário 2':")
        print(f"Total de linhas: {len(dados_formulario)}")
        
        # Filtra apenas as colunas desejadas
        dados_filtrados = dados_formulario.loc[:, ['Carimbo de data/hora', 'Município', 'Instituição']]
        
        # Processa dados mensais
        dados_processados = processar_dados_mensais(dados_filtrados)
        
        # Gera dados para o mapa com o intervalo especificado
        dados_mapa = gerar_dados_mapa_mensal(dados_processados, intervalo_inicio, intervalo_fim)
        
        # Salva dados do mapa
        salvar_dados_mapa(dados_mapa)
        
        # SALVA A PLANILHA COMPLETA
        sucesso = salvar_planilha_local(dados_filtrados, caminho_salvar)
        
        if sucesso:
            print("\n✅ Processo concluído!")
            print(f"📊 Intervalo de referência: {intervalo_inicio} a {intervalo_fim}")
            print(f"📅 Períodos com dados: {dados_mapa['periodos_com_dados']}")
            print(f"🏙️ Municípios únicos no período: {dados_mapa['total_municipios_unicos']}")
            print(f"🏛️ Instituições únicas no período: {dados_mapa['total_instituicoes_unicas']}")
            print(f"📝 Total de respostas: {dados_mapa['total_respostas']}")
            
            # Mostra resumo por mês
            for periodo, dados in dados_mapa['grupos_mensais'].items():
                print(f"  • {periodo}: {dados['total_municipios']} municípios, {dados['total_instituicoes']} instituições, {dados['total_respostas']} respostas")
                
                # Mostra detalhes dos municípios por instituição no resumo final
                for instituicao, municipios in dados['municipios_por_instituicao'].items():
                    print(f"    🏛️ {instituicao}: {len(municipios)} municípios")
        else:
            print("\n❌ Erro ao salvar o arquivo")
        
    except Exception as e:
        print(f"Erro: {e}")
        sys.exit(1)

def salvar_planilha_local(df, caminho_arquivo):
    """Salva o DataFrame como arquivo Excel local"""
    try:
        caminho = Path(caminho_arquivo)
        caminho.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(caminho_arquivo, index=False)
        print(f"Planilha salva com sucesso em: {caminho_arquivo}")
        return True
    except Exception as e:
        print(f"Erro ao salvar planilha: {e}")
        return False

if __name__ == "__main__":
    main()

Guia 'Respostas ao formulário 2' carregada com sucesso: 937 linhas

Dados da guia 'Respostas ao formulário 2':
Total de linhas: 937
Filtrando dados de 2025-03 a 2025-12
Dados após filtro: 288 linhas
Períodos encontrados no intervalo: [Period('2025-04', 'M'), Period('2025-05', 'M'), Period('2025-06', 'M'), Period('2025-07', 'M'), Period('2025-08', 'M'), Period('2025-09', 'M'), Period('2025-10', 'M'), Period('2025-11', 'M')]
📅 2025-04: 27 municípios, 3 instituições, 37 respostas
   🏛️ AGERP: 13 municípios - Cantanhede, São João dos Patos, Pinheiro, Viana, Imperatriz, Nina Rodrigues, Bacabal, Alto Alegre do Maranhão, Paraibano, Trizidela do Vale, Santa Inês, Caxias, São João do Soter
   🏛️ SENAR: 14 municípios - São José de Ribamar, Pio XII, Santa Inês, Bom Jardim, Satubinha, Santa Luzia, São João do Carú, Humberto de Campos, Rosário, Lima Campos, Peritoró, Igarapé Grande, São Mateus do Maranhão, Pedreiras
   🏛️ DEFESA CIVIL: 3 municípios - Pinheiro, Bacabal, Açailândia
📅 2025-05: 15 muni

In [3]:
import sys
import re
import requests
import io
import pandas as pd
import os
from pathlib import Path
from datetime import datetime, timedelta

# Configurações unificadas
CONFIG = {
    'arquivo_observa': '../monitor_seca/rede_observadores/Observadores.xlsx',
    'sheet_url_observa': "https://docs.google.com/spreadsheets/d/1X1b8U2gsiSqIcGIcDO1kvlPAoD1xg3c4tY7v4l-1N6Y/edit?rtpof=true&sd=true&gid=1996667901#gid=1996667901",
    'geojson_ma': '../docs/dados.geojson',  # Arquivo GeoJSON com os municípios do MA
    'intervalo_inicio': '03/2025',  # DD/MM/AAAA ou MM/AAAA
    'intervalo_fim': '12/2025'      # DD/MM/AAAA ou MM/AAAA
}

def extrair_dados_guia(url, nome_guia="Respostas ao formulário 2"):
    """Extrai dados de uma guia específica da planilha Google Sheets"""
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    
    if not match:
        raise ValueError("Não foi possível extrair o ID da planilha da URL")
    
    sheet_id = match.group(1)
    xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    response = requests.get(xlsx_url, timeout=30)
    response.raise_for_status()

    xlsx_file = io.BytesIO(response.content)
    
    try:
        dados_guia = pd.read_excel(xlsx_file, sheet_name=nome_guia)
        print(f"Guia '{nome_guia}' carregada com sucesso: {len(dados_guia)} linhas")
        return dados_guia
    except ValueError as e:
        todas_guias = pd.read_excel(xlsx_file, sheet_name=None)
        guias_disponiveis = list(todas_guias.keys())
        raise ValueError(f"Guia '{nome_guia}' não encontrada. Guias disponíveis: {guias_disponiveis}") from e

def processar_dados_mensais(df):
    """Processa os dados para análise mensal"""
    # Converte a coluna de data/hora
    df['Carimbo de data/hora'] = pd.to_datetime(df['Carimbo de data/hora'], errors='coerce')
    
    # Extrai mês e ano
    df['Ano'] = df['Carimbo de data/hora'].dt.year
    df['Mes'] = df['Carimbo de data/hora'].dt.month
    df['Mes_Ano'] = df['Carimbo de data/hora'].dt.to_period('M')
    
    # Remove linhas com datas inválidas
    df = df.dropna(subset=['Carimbo de data/hora'])
    
    return df

def converter_para_periodo(data_str):
    """Converte string de data para período (MM/AAAA)"""
    try:
        # Tenta converter como MM/AAAA
        if len(data_str.split('/')) == 2:
            mes, ano = data_str.split('/')
            return pd.Period(year=int(ano), month=int(mes), freq='M')
        # Tenta converter como DD/MM/AAAA
        elif len(data_str.split('/')) == 3:
            dia, mes, ano = data_str.split('/')
            return pd.Period(year=int(ano), month=int(mes), freq='M')
    except Exception as e:
        raise ValueError(f"Formato de data inválido: {data_str}. Use MM/AAAA ou DD/MM/AAAA") from e

def filtrar_por_intervalo(df, inicio, fim):
    """Filtra o DataFrame pelo intervalo de datas especificado"""
    # Converte as strings para períodos
    periodo_inicio = converter_para_periodo(inicio)
    periodo_fim = converter_para_periodo(fim)
    
    print(f"Filtrando dados de {periodo_inicio} a {periodo_fim}")
    
    # Filtra os dados pelo intervalo
    df_filtrado = df[(df['Mes_Ano'] >= periodo_inicio) & (df['Mes_Ano'] <= periodo_fim)]
    
    print(f"Dados após filtro: {len(df_filtrado)} linhas")
    print(f"Períodos encontrados no intervalo: {sorted(df_filtrado['Mes_Ano'].unique())}")
    
    return df_filtrado

def gerar_dados_mapa_mensal(df, intervalo_inicio, intervalo_fim):
    """Gera dados para o mapa mensal com agrupamento por mês"""
    # Filtra os dados pelo intervalo
    df_filtrado = filtrar_por_intervalo(df, intervalo_inicio, intervalo_fim)
    
    if len(df_filtrado) == 0:
        print("⚠️  Nenhum dado encontrado no intervalo especificado!")
        return {
            'intervalo_inicio': intervalo_inicio,
            'intervalo_fim': intervalo_fim,
            'grupos_mensais': {},
            'total_respostas': 0
        }
    
    # Agrupa por mês e município
    grupos_mensais = {}
    
    # Pega todos os períodos únicos no intervalo filtrado
    periodos_unicos = sorted(df_filtrado['Mes_Ano'].unique())
    
    for periodo in periodos_unicos:
        # Filtra dados do mês específico
        df_mes = df_filtrado[df_filtrado['Mes_Ano'] == periodo]
        
        # Lista de municípios que responderam neste mês
        municipios_mes = df_mes['Município'].unique().tolist()
        
        # Contagem de respostas por município neste mês
        contagem_municipios = df_mes.groupby('Município').size().to_dict()
        
        # Contagem de respostas por instituição neste mês
        contagem_instituicoes = df_mes.groupby('Instituição').size().to_dict()
        
        # Lista de instituições únicas que responderam neste mês
        instituicoes_mes = df_mes['Instituição'].unique().tolist()
        
        # Adiciona ao grupo mensal
        grupos_mensais[str(periodo)] = {
            'municipios_responderam': municipios_mes,
            'total_respostas': len(df_mes),
            'contagem_por_municipio': contagem_municipios,
            'total_municipios': len(municipios_mes),
            'instituicoes_responderam': instituicoes_mes,
            'contagem_por_instituicao': contagem_instituicoes,
            'total_instituicoes': len(instituicoes_mes)
        }
        
        print(f"📅 {periodo}: {len(municipios_mes)} municípios, {len(instituicoes_mes)} instituições, {len(df_mes)} respostas")
    
    # Estatísticas gerais do período
    total_municipios_unicos = df_filtrado['Município'].nunique()
    total_instituicoes_unicas = df_filtrado['Instituição'].nunique()
    
    print(f"\n📊 Total de períodos com dados: {len(grupos_mensais)}")
    print(f"🏙️ Municípios únicos no período: {total_municipios_unicos}")
    print(f"🏛️ Instituições únicas no período: {total_instituicoes_unicas}")
    print(f"📝 Total de respostas no intervalo: {len(df_filtrado)}")
    
    return {
        'intervalo_inicio': intervalo_inicio,
        'intervalo_fim': intervalo_fim,
        'grupos_mensais': grupos_mensais,
        'total_respostas': len(df_filtrado),
        'total_municipios_unicos': total_municipios_unicos,
        'total_instituicoes_unicas': total_instituicoes_unicas,
        'periodos_com_dados': [str(p) for p in periodos_unicos]
    }

def salvar_dados_mapa(dados_mapa):
    """Salva os dados do mapa em arquivo JSON para uso no frontend"""
    caminho_json = '../monitor_seca/rede_observadores/dados_mapa_observadores.json'
    
    caminho = Path(caminho_json)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    
    import json
    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(dados_mapa, f, ensure_ascii=False, indent=2)
    
    print(f"Dados do mapa salvos em: {caminho_json}")

def main():
    """Função principal do sistema integrado"""
    try:
        url_planilha = CONFIG['sheet_url_observa']
        caminho_salvar = CONFIG['arquivo_observa']
        intervalo_inicio = CONFIG['intervalo_inicio']
        intervalo_fim = CONFIG['intervalo_fim']
        
        # Busca os dados da guia específica
        dados_formulario = extrair_dados_guia(url_planilha, "Respostas ao formulário 2")

        print("\nDados da guia 'Respostas ao formulário 2':")
        print(f"Total de linhas: {len(dados_formulario)}")
        
        # Filtra apenas as colunas desejadas
        dados_filtrados = dados_formulario.loc[:, ['Carimbo de data/hora', 'Município', 'Instituição']]
        
        # Processa dados mensais
        dados_processados = processar_dados_mensais(dados_filtrados)
        
        # Gera dados para o mapa com o intervalo especificado
        dados_mapa = gerar_dados_mapa_mensal(dados_processados, intervalo_inicio, intervalo_fim)
        
        # Salva dados do mapa
        salvar_dados_mapa(dados_mapa)
        
        # SALVA A PLANILHA COMPLETA
        sucesso = salvar_planilha_local(dados_filtrados, caminho_salvar)
        
        if sucesso:
            print("\n✅ Processo concluído!")
            print(f"📊 Intervalo de referência: {intervalo_inicio} a {intervalo_fim}")
            print(f"📅 Períodos com dados: {dados_mapa['periodos_com_dados']}")
            print(f"🏙️ Municípios únicos no período: {dados_mapa['total_municipios_unicos']}")
            print(f"🏛️ Instituições únicas no período: {dados_mapa['total_instituicoes_unicas']}")
            print(f"📝 Total de respostas: {dados_mapa['total_respostas']}")
            
            # Mostra resumo por mês
            for periodo, dados in dados_mapa['grupos_mensais'].items():
                print(f"  • {periodo}: {dados['total_municipios']} municípios, {dados['total_instituicoes']} instituições, {dados['total_respostas']} respostas")
        else:
            print("\n❌ Erro ao salvar o arquivo")
        
    except Exception as e:
        print(f"Erro: {e}")
        sys.exit(1)

def salvar_planilha_local(df, caminho_arquivo):
    """Salva o DataFrame como arquivo Excel local"""
    try:
        caminho = Path(caminho_arquivo)
        caminho.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(caminho_arquivo, index=False)
        print(f"Planilha salva com sucesso em: {caminho_arquivo}")
        return True
    except Exception as e:
        print(f"Erro ao salvar planilha: {e}")
        return False

if __name__ == "__main__":
    main()

Guia 'Respostas ao formulário 2' carregada com sucesso: 937 linhas

Dados da guia 'Respostas ao formulário 2':
Total de linhas: 937
Filtrando dados de 2025-03 a 2025-12
Dados após filtro: 288 linhas
Períodos encontrados no intervalo: [Period('2025-04', 'M'), Period('2025-05', 'M'), Period('2025-06', 'M'), Period('2025-07', 'M'), Period('2025-08', 'M'), Period('2025-09', 'M'), Period('2025-10', 'M'), Period('2025-11', 'M')]
📅 2025-04: 27 municípios, 3 instituições, 37 respostas
📅 2025-05: 15 municípios, 2 instituições, 17 respostas
📅 2025-06: 20 municípios, 3 instituições, 40 respostas
📅 2025-07: 21 municípios, 3 instituições, 24 respostas
📅 2025-08: 16 municípios, 4 instituições, 19 respostas
📅 2025-09: 20 municípios, 3 instituições, 23 respostas
📅 2025-10: 11 municípios, 3 instituições, 11 respostas
📅 2025-11: 69 municípios, 4 instituições, 117 respostas

📊 Total de períodos com dados: 8
🏙️ Municípios únicos no período: 75
🏛️ Instituições únicas no período: 5
📝 Total de respostas no i

In [3]:
import sys
import re
import requests
import io
import pandas as pd
import os
from pathlib import Path
from datetime import datetime, timedelta

# Configurações unificadas
CONFIG = {
    'arquivo_observa': 'docs/Observadores.xlsx',
    'sheet_url_observa': "https://docs.google.com/spreadsheets/d/1X1b8U2gsiSqIcGIcDO1kvlPAoD1xg3c4tY7v4l-1N6Y/edit?rtpof=true&sd=true&gid=1996667901#gid=1996667901",
    'geojson_ma': '../docs/Shapefile/MA_Municipios_2022/MA_Municipios_2022.geojson',  # Arquivo GeoJSON com os municípios do MA
    'intervalo_inicio': '03/2025',  # DD/MM/AAAA ou MM/AAAA
    'intervalo_fim': '12/2025'      # DD/MM/AAAA ou MM/AAAA
}

def extrair_dados_guia(url, nome_guia="Respostas ao formulário 2"):
    """Extrai dados de uma guia específica da planilha Google Sheets"""
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    
    if not match:
        raise ValueError("Não foi possível extrair o ID da planilha da URL")
    
    sheet_id = match.group(1)
    xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    response = requests.get(xlsx_url, timeout=30)
    response.raise_for_status()

    xlsx_file = io.BytesIO(response.content)
    
    try:
        dados_guia = pd.read_excel(xlsx_file, sheet_name=nome_guia)
        print(f"Guia '{nome_guia}' carregada com sucesso: {len(dados_guia)} linhas")
        return dados_guia
    except ValueError as e:
        todas_guias = pd.read_excel(xlsx_file, sheet_name=None)
        guias_disponiveis = list(todas_guias.keys())
        raise ValueError(f"Guia '{nome_guia}' não encontrada. Guias disponíveis: {guias_disponiveis}") from e

def processar_dados_mensais(df):
    """Processa os dados para análise mensal"""
    # Converte a coluna de data/hora
    df['Carimbo de data/hora'] = pd.to_datetime(df['Carimbo de data/hora'], errors='coerce')
    
    # Extrai mês e ano
    df['Ano'] = df['Carimbo de data/hora'].dt.year
    df['Mes'] = df['Carimbo de data/hora'].dt.month
    df['Mes_Ano'] = df['Carimbo de data/hora'].dt.to_period('M')
    
    # Remove linhas com datas inválidas
    df = df.dropna(subset=['Carimbo de data/hora'])
    
    return df

def converter_para_periodo(data_str):
    """Converte string de data para período (MM/AAAA)"""
    try:
        # Tenta converter como MM/AAAA
        if len(data_str.split('/')) == 2:
            mes, ano = data_str.split('/')
            return pd.Period(year=int(ano), month=int(mes), freq='M')
        # Tenta converter como DD/MM/AAAA
        elif len(data_str.split('/')) == 3:
            dia, mes, ano = data_str.split('/')
            return pd.Period(year=int(ano), month=int(mes), freq='M')
    except Exception as e:
        raise ValueError(f"Formato de data inválido: {data_str}. Use MM/AAAA ou DD/MM/AAAA") from e

def filtrar_por_intervalo(df, inicio, fim):
    """Filtra o DataFrame pelo intervalo de datas especificado"""
    # Converte as strings para períodos
    periodo_inicio = converter_para_periodo(inicio)
    periodo_fim = converter_para_periodo(fim)
    
    print(f"Filtrando dados de {periodo_inicio} a {periodo_fim}")
    
    # Filtra os dados pelo intervalo
    df_filtrado = df[(df['Mes_Ano'] >= periodo_inicio) & (df['Mes_Ano'] <= periodo_fim)]
    
    print(f"Dados após filtro: {len(df_filtrado)} linhas")
    print(f"Períodos encontrados no intervalo: {sorted(df_filtrado['Mes_Ano'].unique())}")
    
    return df_filtrado

def gerar_dados_mapa_mensal(df, intervalo_inicio, intervalo_fim):
    """Gera dados para o mapa mensal com agrupamento por mês"""
    # Filtra os dados pelo intervalo
    df_filtrado = filtrar_por_intervalo(df, intervalo_inicio, intervalo_fim)
    
    if len(df_filtrado) == 0:
        print("⚠️  Nenhum dado encontrado no intervalo especificado!")
        return {
            'intervalo_inicio': intervalo_inicio,
            'intervalo_fim': intervalo_fim,
            'grupos_mensais': {},
            'total_respostas': 0
        }
    
    # Agrupa por mês e município
    grupos_mensais = {}
    
    # Pega todos os períodos únicos no intervalo filtrado
    periodos_unicos = sorted(df_filtrado['Mes_Ano'].unique())
    
    for periodo in periodos_unicos:
        # Filtra dados do mês específico
        df_mes = df_filtrado[df_filtrado['Mes_Ano'] == periodo]
        
        # Lista de municípios que responderam neste mês
        municipios_mes = df_mes['Município'].unique().tolist()
        
        # Contagem de respostas por município neste mês
        contagem_municipios = df_mes.groupby('Município').size().to_dict()
        
        # Adiciona ao grupo mensal
        grupos_mensais[str(periodo)] = {
            'municipios_responderam': municipios_mes,
            'total_respostas': len(df_mes),
            'contagem_por_municipio': contagem_municipios,
            'total_municipios': len(municipios_mes)
        }
        
        print(f"📅 {periodo}: {len(municipios_mes)} municípios, {len(df_mes)} respostas")
    
    print(f"\n📊 Total de períodos com dados: {len(grupos_mensais)}")
    print(f"🏙️ Municípios únicos no período: {df_filtrado['Município'].nunique()}")
    print(f"📝 Total de respostas no intervalo: {len(df_filtrado)}")
    
    return {
        'intervalo_inicio': intervalo_inicio,
        'intervalo_fim': intervalo_fim,
        'grupos_mensais': grupos_mensais,
        'total_respostas': len(df_filtrado),
        'total_municipios_unicos': df_filtrado['Município'].nunique(),
        'periodos_com_dados': [str(p) for p in periodos_unicos]
    }

def salvar_dados_mapa(dados_mapa):
    """Salva os dados do mapa em arquivo JSON para uso no frontend"""
    caminho_json = 'docs/dados_mapa_observadores.json'
    
    caminho = Path(caminho_json)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    
    import json
    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(dados_mapa, f, ensure_ascii=False, indent=2)
    
    print(f"Dados do mapa salvos em: {caminho_json}")

def main():
    """Função principal do sistema integrado"""
    try:
        url_planilha = CONFIG['sheet_url_observa']
        caminho_salvar = CONFIG['arquivo_observa']
        intervalo_inicio = CONFIG['intervalo_inicio']
        intervalo_fim = CONFIG['intervalo_fim']
        
        # Busca os dados da guia específica
        dados_formulario = extrair_dados_guia(url_planilha, "Respostas ao formulário 2")

        print("\nDados da guia 'Respostas ao formulário 2':")
        print(f"Total de linhas: {len(dados_formulario)}")
        
        # Filtra apenas as colunas desejadas
        dados_filtrados = dados_formulario.loc[:, ['Carimbo de data/hora', 'Município', 'Instituição']]
        
        # Processa dados mensais
        dados_processados = processar_dados_mensais(dados_filtrados)
        
        # Gera dados para o mapa com o intervalo especificado
        dados_mapa = gerar_dados_mapa_mensal(dados_processados, intervalo_inicio, intervalo_fim)
        
        # Salva dados do mapa
        salvar_dados_mapa(dados_mapa)
        
        # SALVA A PLANILHA COMPLETA
        sucesso = salvar_planilha_local(dados_filtrados, caminho_salvar)
        
        if sucesso:
            print("\n✅ Processo concluído!")
            print(f"📊 Intervalo de referência: {intervalo_inicio} a {intervalo_fim}")
            print(f"📅 Períodos com dados: {dados_mapa['periodos_com_dados']}")
            print(f"🏙️ Municípios únicos no período: {dados_mapa['total_municipios_unicos']}")
            print(f"📝 Total de respostas: {dados_mapa['total_respostas']}")
            
            # Mostra resumo por mês
            for periodo, dados in dados_mapa['grupos_mensais'].items():
                print(f"  • {periodo}: {dados['total_municipios']} municípios, {dados['total_respostas']} respostas")
        else:
            print("\n❌ Erro ao salvar o arquivo")
        
    except Exception as e:
        print(f"Erro: {e}")
        sys.exit(1)

def salvar_planilha_local(df, caminho_arquivo):
    """Salva o DataFrame como arquivo Excel local"""
    try:
        caminho = Path(caminho_arquivo)
        caminho.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(caminho_arquivo, index=False)
        print(f"Planilha salva com sucesso em: {caminho_arquivo}")
        return True
    except Exception as e:
        print(f"Erro ao salvar planilha: {e}")
        return False

if __name__ == "__main__":
    main()

Guia 'Respostas ao formulário 2' carregada com sucesso: 937 linhas

Dados da guia 'Respostas ao formulário 2':
Total de linhas: 937
Filtrando dados de 2025-03 a 2025-12
Dados após filtro: 288 linhas
Períodos encontrados no intervalo: [Period('2025-04', 'M'), Period('2025-05', 'M'), Period('2025-06', 'M'), Period('2025-07', 'M'), Period('2025-08', 'M'), Period('2025-09', 'M'), Period('2025-10', 'M'), Period('2025-11', 'M')]
📅 2025-04: 27 municípios, 37 respostas
📅 2025-05: 15 municípios, 17 respostas
📅 2025-06: 20 municípios, 40 respostas
📅 2025-07: 21 municípios, 24 respostas
📅 2025-08: 16 municípios, 19 respostas
📅 2025-09: 20 municípios, 23 respostas
📅 2025-10: 11 municípios, 11 respostas
📅 2025-11: 69 municípios, 117 respostas

📊 Total de períodos com dados: 8
🏙️ Municípios únicos no período: 75
📝 Total de respostas no intervalo: 288
Dados do mapa salvos em: docs/dados_mapa_observadores.json
Planilha salva com sucesso em: docs/Observadores.xlsx

✅ Processo concluído!
📊 Intervalo de 

In [ ]:
import sys
import re
import requests
import io
import pandas as pd
import os
from pathlib import Path
from datetime import datetime, timedelta

# Configurações unificadas
CONFIG = {
    'arquivo_observa': 'docs/Observadores.xlsx',
    'sheet_url_observa': "https://docs.google.com/spreadsheets/d/1X1b8U2gsiSqIcGIcDO1kvlPAoD1xg3c4tY7v4l-1N6Y/edit?rtpof=true&sd=true&gid=1996667901#gid=1996667901",
    'geojson_ma': '../docs/dados.geojson'  # Arquivo GeoJSON com os municípios do MA
}

def extrair_dados_guia(url, nome_guia="Respostas ao formulário 2"):
    """Extrai dados de uma guia específica da planilha Google Sheets"""
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    
    if not match:
        raise ValueError("Não foi possível extrair o ID da planilha da URL")
    
    sheet_id = match.group(1)
    xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    response = requests.get(xlsx_url, timeout=30)
    response.raise_for_status()

    xlsx_file = io.BytesIO(response.content)
    
    try:
        dados_guia = pd.read_excel(xlsx_file, sheet_name=nome_guia)
        print(f"Guia '{nome_guia}' carregada com sucesso: {len(dados_guia)} linhas")
        return dados_guia
    except ValueError as e:
        todas_guias = pd.read_excel(xlsx_file, sheet_name=None)
        guias_disponiveis = list(todas_guias.keys())
        raise ValueError(f"Guia '{nome_guia}' não encontrada. Guias disponíveis: {guias_disponiveis}") from e

def processar_dados_mensais(df):
    """Processa os dados para análise mensal"""
    # Converte a coluna de data/hora
    df['Carimbo de data/hora'] = pd.to_datetime(df['Carimbo de data/hora'], errors='coerce')
    
    # Extrai mês e ano
    df['Ano'] = df['Carimbo de data/hora'].dt.year
    df['Mes'] = df['Carimbo de data/hora'].dt.month
    df['Mes_Ano'] = df['Carimbo de data/hora'].dt.to_period('M')
    
    # Remove linhas com datas inválidas
    df = df.dropna(subset=['Carimbo de data/hora'])
    
    return df

def gerar_dados_mapa_mensal(df):
    """Gera dados para o mapa mensal"""
    # Agrupa por município e mês
    municipios_mes = df.groupby(['Município', 'Mes_Ano']).size().reset_index(name='Respostas')
    
    # Pega o último mês disponível (ou você pode especificar um mês específico)
    ultimo_mes = df['Mes_Ano'].max()
    
    # Filtra apenas o último mês
    dados_ultimo_mes = municipios_mes[municipios_mes['Mes_Ano'] == ultimo_mes]
    
    # Lista de todos os municípios que responderam no último mês
    municipios_responderam = dados_ultimo_mes['Município'].unique().tolist()
    
    print(f"Municípios que responderam no mês {ultimo_mes}: {len(municipios_responderam)}")
    print(f"Lista: {municipios_responderam}")
    
    return {
        'municipios_responderam': municipios_responderam,
        'mes_referencia': str(ultimo_mes),
        'total_respostas': len(df[df['Mes_Ano'] == ultimo_mes])
    }

def salvar_dados_mapa(dados_mapa):
    """Salva os dados do mapa em arquivo JSON para uso no frontend"""
    caminho_json = 'docs/dados_mapa_observadores.json'
    
    caminho = Path(caminho_json)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    
    import json
    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(dados_mapa, f, ensure_ascii=False, indent=2)
    
    print(f"Dados do mapa salvos em: {caminho_json}")

def main():
    """Função principal do sistema integrado"""
    try:
        url_planilha = CONFIG['sheet_url_observa']
        caminho_salvar = CONFIG['arquivo_observa']
        
        # Busca os dados da guia específica
        dados_formulario = extrair_dados_guia(url_planilha, "Respostas ao formulário 2")

        print("\nDados da guia 'Respostas ao formulário 2':")
        print(f"Total de linhas: {len(dados_formulario)}")
        
        # Filtra apenas as colunas desejadas
        dados_filtrados = dados_formulario.loc[:, ['Carimbo de data/hora', 'Município', 'Instituição']]
        
        # Processa dados mensais
        dados_processados = processar_dados_mensais(dados_filtrados)
        
        # Gera dados para o mapa
        dados_mapa = gerar_dados_mapa_mensal(dados_processados)
        
        # Salva dados do mapa
        salvar_dados_mapa(dados_mapa)
        
        # SALVA A PLANILHA COMPLETA
        sucesso = salvar_planilha_local(dados_filtrados, caminho_salvar)
        
        if sucesso:
            print("\n✅ Processo concluído!")
            print(f"📊 Mês de referência: {dados_mapa['mes_referencia']}")
            print(f"🏙️ Municípios que responderam: {len(dados_mapa['municipios_responderam'])}")
            print(f"📝 Total de respostas: {dados_mapa['total_respostas']}")
        else:
            print("\n❌ Erro ao salvar o arquivo")
        
    except Exception as e:
        print(f"Erro: {e}")
        sys.exit(1)

def salvar_planilha_local(df, caminho_arquivo):
    """Salva o DataFrame como arquivo Excel local"""
    try:
        caminho = Path(caminho_arquivo)
        caminho.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(caminho_arquivo, index=False)
        print(f"Planilha salva com sucesso em: {caminho_arquivo}")
        return True
    except Exception as e:
        print(f"Erro ao salvar planilha: {e}")
        return False

if __name__ == "__main__":
    main()

Guia 'Respostas ao formulário 2' carregada com sucesso: 937 linhas

Dados da guia 'Respostas ao formulário 2':
Total de linhas: 937
Municípios que responderam no mês 2025-11: 69
Lista: ['Alcântara', 'Alto Alegre do Maranhão', 'Amapá do Maranhão', 'Anapurus', 'Arari', 'Axixá', 'Açailândia', 'Bacabal', 'Barreirinhas', 'Barão de Grajaú', 'Bequimão', 'Bom Jardim', 'Buriti Bravo', 'Cantanhede', 'Caxias', 'Centro do Guilherme', 'Chapadinha', 'Codó', 'Colinas', 'Coroatá', 'Cururupu', 'Dom Pedro', 'Fortuna', 'Governador Nunes Freire', 'Humberto de Campos', 'Icatu', 'Igarapé Grande', 'Imperatriz', 'Lago da Pedra', 'Lagoa do Mato', 'Lima Campos', 'Mata Roma', 'Matões do Norte', 'Mirador', 'Miranda do Norte', 'Monção', 'Nina Rodrigues', 'Nova Olinda do Maranhão', "Olho d'Água das Cunhãs", 'Paraibano', 'Passagem Franca', 'Pastos Bons', 'Paulo Ramos', 'Pedreiras', 'Pinheiro', 'Pirapemas', 'Poção de Pedras', 'Presidente Dutra', 'Primeira Cruz', 'Rosário', 'Santa Inês', 'Santa Luzia', 'Santo Amaro do

In [15]:
import sys
import re
import requests
import io
import pandas as pd
import os
from pathlib import Path

# Configurações unificadas
CONFIG = {
    'arquivo_observa': 'docs/Observadores.xlsx',
    'sheet_url_observa': "https://docs.google.com/spreadsheets/d/1X1b8U2gsiSqIcGIcDO1kvlPAoD1xg3c4tY7v4l-1N6Y/edit?rtpof=true&sd=true&gid=1996667901#gid=1996667901"
}

def extrair_dados_guia(url, nome_guia="Respostas ao formulário 2"):
    """Extrai dados de uma guia específica da planilha Google Sheets"""
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    
    if not match:
        raise ValueError("Não foi possível extrair o ID da planilha da URL")
    
    sheet_id = match.group(1)
    #print(f"Sheet ID: {sheet_id}")
            
    xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    response = requests.get(xlsx_url, timeout=30)
    response.raise_for_status()

    xlsx_file = io.BytesIO(response.content)
    
    # Carrega apenas a guia específica
    try:
        dados_guia = pd.read_excel(xlsx_file, sheet_name=nome_guia)
        print(f"Guia '{nome_guia}' carregada com sucesso: {len(dados_guia)} linhas")
        return dados_guia
    except ValueError as e:
        # Lista guias disponíveis para ajudar no debug
        todas_guias = pd.read_excel(xlsx_file, sheet_name=None)
        guias_disponiveis = list(todas_guias.keys())
        raise ValueError(f"Guia '{nome_guia}' não encontrada. Guias disponíveis: {guias_disponiveis}") from e

def salvar_planilha_local(df, caminho_arquivo):
    """Salva o DataFrame como arquivo Excel local"""
    try:
        # Garante que o diretório existe
        caminho = Path(caminho_arquivo)
        caminho.parent.mkdir(parents=True, exist_ok=True)
        
        # Salva o arquivo
        df.to_excel(caminho_arquivo, index=False)
        print(f"Planilha salva com sucesso em: {caminho_arquivo}")
        print(f"Tamanho do arquivo: {os.path.getsize(caminho_arquivo) / 1024:.2f} KB")
        return True
    except Exception as e:
        print(f"Erro ao salvar planilha: {e}")
        return False

def main():
    """Função principal do sistema integrado"""
    try:
        url_planilha = CONFIG['sheet_url_observa']
        caminho_salvar = CONFIG['arquivo_observa']
        
        # Busca apenas os dados da guia específica
        dados_formulario = extrair_dados_guia(url_planilha, "Respostas ao formulário 2")

        print("\nDados da guia 'Respostas ao formulário 2':")
        print(f"Total de linhas: {len(dados_formulario)}")
        print(f"Total de colunas: {len(dados_formulario.columns)}")
        print("\nPrimeiras 5 linhas:")
        #print(dados_formulario.head())

        # Filtra apenas as colunas desejadas
        dados_filtrados = dados_formulario.loc[:, ['Carimbo de data/hora', 'Município', 'Instituição']]
        dados_filtrados['Carimbo de data/hora'] = pd.to_datetime(dados_filtrados['Carimbo de data/hora'], errors='coerce')

        print("\nDados filtrados (primeiras 5 linhas):")
        print(dados_filtrados.head())
        
        # SALVA A PLANILHA - mesma forma do código anterior
        sucesso = salvar_planilha_local(dados_filtrados, caminho_salvar)
        
        if sucesso:
            print(f"\n✅ Processo concluído! Arquivo salvo em: {caminho_salvar}")
        else:
            print("\n❌ Erro ao salvar o arquivo")
        
    except Exception as e:
        print(f"Erro: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()

Guia 'Respostas ao formulário 2' carregada com sucesso: 937 linhas

Dados da guia 'Respostas ao formulário 2':
Total de linhas: 937
Total de colunas: 17

Primeiras 5 linhas:

Dados filtrados (primeiras 5 linhas):
     Carimbo de data/hora            Município   Instituição
0 2025-11-04 12:08:07.292             Pinheiro  DEFESA CIVIL
1 2025-11-04 12:08:29.902  São José de Ribamar         SENAR
2 2025-11-04 12:11:06.092            Paraibano         AGERP
3 2025-11-05 14:25:43.663           Cantanhede         AGERP
4 2025-11-05 14:57:34.907    Trizidela do Vale         AGERP
Planilha salva com sucesso em: docs/Observadores.xlsx
Tamanho do arquivo: 22.79 KB

✅ Processo concluído! Arquivo salvo em: docs/Observadores.xlsx


In [ ]:
%pip install bs4

In [5]:
from pathlib import Path

def encontrar_pastas_faltantes():
    pasta_raiz = Path("../docs/monitor_seca/")
    pastas_faltantes = []
    
    print(f"🔍 Procurando pastas faltantes em: {pasta_raiz}")
    
    if not pasta_raiz.exists():
        print("❌ Pasta não encontrada!")
        return []
    
    for subpasta in pasta_raiz.iterdir():
        if subpasta.is_dir():
            arquivo_geojson = subpasta / "seca_atributos.geojson"
            
            if not arquivo_geojson.exists():
                # Extrair mês e ano do nome da pasta
                if len(subpasta.name) == 6:  # Formato: MMAAAA
                    try:
                        mes = int(subpasta.name[:2])
                        ano = int(subpasta.name[2:])
                        pastas_faltantes.append({
                            'mes': mes, 
                            'ano': ano, 
                            'pasta': subpasta.name
                        })
                        print(f"❌ {subpasta.name} → Mês: {mes}, Ano: {ano}")
                    except ValueError:
                        print(f"❌ {subpasta.name} → Formato inválido")
                else:
                    print(f"❌ {subpasta.name} → Formato diferente")
    
    print(f"\n📊 Total de pastas faltantes: {len(pastas_faltantes)}")
    return pastas_faltantes

# Usar
pastas_faltantes = encontrar_pastas_faltantes()

# Agora você pode usar as variáveis mes e ano
for falta in pastas_faltantes:
    mes = falta['mes']
    ano = falta['ano']
    print(f"Variáveis: mes = {mes}, ano = {ano}")
    
    # Exemplo de uso
    if mes == 4 and ano == 2025:
        print(f"  → Esta é a pasta 042025 que estamos procurando!")

🔍 Procurando pastas faltantes em: ..\docs\monitor_seca
❌ 042025 → Mês: 4, Ano: 2025

📊 Total de pastas faltantes: 1
Variáveis: mes = 4, ano = 2025
  → Esta é a pasta 042025 que estamos procurando!


In [3]:
from pathlib import Path

def verificar_todas_pastas():
    pasta_raiz = Path("../docs/monitor_seca/")
    resultados = []
    
    print(f"🔍 Verificando pasta: {pasta_raiz}")
    
    if not pasta_raiz.exists():
        print("❌ Pasta não encontrada!")
        return []
    
    # Listar todas as pastas do diretório
    todas_pastas = [p for p in pasta_raiz.iterdir() if p.is_dir()]
    
    print(f"📁 Total de pastas encontradas: {len(todas_pastas)}")
    print("-" * 50)
    
    for subpasta in todas_pastas:
        arquivo_geojson = subpasta / "seca_atributos.geojson"
        
        if arquivo_geojson.exists():
            status = "✅ ENCONTRADO"
            resultados.append({
                'pasta': subpasta.name,
                'status': 'encontrado',
                'caminho': str(arquivo_geojson)
            })
        else:
            status = "❌ NÃO ENCONTRADO"
            resultados.append({
                'pasta': subpasta.name,
                'status': 'nao_encontrado'
            })
        
        print(f"  {status} | {subpasta.name}/seca_atributos.geojson")
    
    # Estatísticas
    encontrados = sum(1 for r in resultados if r['status'] == 'encontrado')
    print("-" * 50)
    print(f"📊 RESUMO:")
    print(f"   Pastas com arquivo: {encontrados}/{len(todas_pastas)}")
    print(f"   Pastas sem arquivo: {len(todas_pastas) - encontrados}/{len(todas_pastas)}")
    
    return resultados

# Usar
resultados = verificar_todas_pastas()

🔍 Verificando pasta: ..\docs\monitor_seca
📁 Total de pastas encontradas: 7
--------------------------------------------------
  ❌ NÃO ENCONTRADO | 042025/seca_atributos.geojson
  ✅ ENCONTRADO | 052025/seca_atributos.geojson
  ✅ ENCONTRADO | 062025/seca_atributos.geojson
  ✅ ENCONTRADO | 072025/seca_atributos.geojson
  ✅ ENCONTRADO | 082025/seca_atributos.geojson
  ✅ ENCONTRADO | 092025/seca_atributos.geojson
  ✅ ENCONTRADO | 102025/seca_atributos.geojson
--------------------------------------------------
📊 RESUMO:
   Pastas com arquivo: 6/7
   Pastas sem arquivo: 1/7


In [1]:
import os
from pathlib import Path

# Método 2: usando pathlib (mais moderno)
def listar_pasta_pathlib(caminho):
    pasta = Path(caminho)
    
    if pasta.exists() and pasta.is_dir():
        print(f"Conteúdo da pasta '{caminho}':")
        
        # Listar pastas
        print("\n📁 Pastas:")
        for subpasta in pasta.iterdir():
            if subpasta.is_dir():
                print(f"  {subpasta.name}")
        
        # Listar arquivos
        print("\n📄 Arquivos:")
        for arquivo in pasta.iterdir():
            if arquivo.is_file():
                tamanho = arquivo.stat().st_size
                print(f"  {arquivo.name} ({tamanho} bytes)")
    else:
        print("Pasta não encontrada!")

# Exemplo de uso
listar_pasta_pathlib("../docs/monitor_seca/")

Conteúdo da pasta '../docs/monitor_seca/':

📁 Pastas:
  042025
  052025
  062025
  072025
  082025
  092025
  102025

📄 Arquivos:
  periodos_disponiveis.json (127 bytes)


In [34]:
import requests
import os
import zipfile
from datetime import datetime, timedelta

def tentar_todas_urls(mes, ano, nome_mes, pasta_base):
    """
    Tenta todas as combinações possíveis de URLs
    """
    # Lista de todas as URLs possíveis
    urls_possiveis = [
        # Formato: nome_mes + ano (4 dígitos) - várias capitalizações
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes}{ano}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.lower()}{ano}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.upper()}{ano}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.capitalize()}{ano}.zip",
        
        # Formato: nome_mes + ano (2 dígitos) - várias capitalizações
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes}{str(ano)[-2:]}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.lower()}{str(ano)[-2:]}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.upper()}{str(ano)[-2:]}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.capitalize()}{str(ano)[-2:]}.zip",
        
        # Formato: mês numérico + ano (4 dígitos)
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{mes:02d}{ano}.zip",
        
        # Formato: mês numérico + ano (2 dígitos)
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{mes:02d}{str(ano)[-2:]}.zip",
    ]
    
    # Nome base para o arquivo (usando minúsculo como padrão)
    nome_base = f"{nome_mes.lower()}{ano}"
    arquivo_zip = os.path.join(pasta_base, f"{nome_base}.zip")
    
    for url in urls_possiveis:
        try:
            print(f"   🔍 Tentando: {os.path.basename(url)}")
            response = requests.head(url, timeout=10)
            if response.status_code == 200:
                print(f"   ✅ Encontrado: {os.path.basename(url)}")
                
                # Fazer download completo
                response_download = requests.get(url, timeout=30)
                if response_download.status_code == 200:
                    with open(arquivo_zip, 'wb') as f:
                        f.write(response_download.content)
                    
                    tamanho = len(response_download.content) / (1024 * 1024)
                    print(f"   📥 Baixado ({tamanho:.1f} MB)")
                    return True, url, tamanho
                    
        except requests.exceptions.RequestException:
            continue
    
    return False, None, 0

def baixar_automatico_detalhado():
    # Datas

    import json

    dados = json.load(open("../docs/monitor_seca/periodos_disponiveis.json", 'r', encoding='utf-8'))
    dados['periodos_disponiveis'].sort(key=lambda x: (int(x[2:]), int(x[:2])))
    maior = max(dados['periodos_disponiveis'], key=lambda x: (int(x[2:]), int(x[:2])))
    print(f"Maior período: {maior[:2]}/{maior[2:]}")

    mes_inicio = int(maior[:2])
    ano_inicio = int(maior[2:])

    data_atual = datetime.now()
    primeiro_dia_mes_atual = data_atual.replace(day=1)
    ultimo_dia_mes_anterior = primeiro_dia_mes_atual - timedelta(days=1)
    mes_anterior = ultimo_dia_mes_anterior.month
    ano_anterior = ultimo_dia_mes_anterior.year
    
    meses_pt = {
        1: 'janeiro', 2: 'fevereiro', 3: 'marco', 4: 'abril',
        5: 'maio', 6: 'junho', 7: 'julho', 8: 'agosto',
        9: 'setembro', 10: 'outubro', 11: 'novembro', 12: 'dezembro'
    }
    
    meses_pt_maiusculo = {
        1: 'Janeiro', 2: 'Fevereiro', 3: 'Marco', 4: 'Abril',
        5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
        9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'
    }
    
    print(f"📅 Período: {mes_inicio:02d}/{ano_inicio} a {mes_anterior:02d}/{ano_anterior}")
    print("=" * 60)
    
    total_encontrados = 0
    total_nao_encontrados = 0
    
    for ano in range(ano_inicio, ano_anterior + 1):
        mes_start = mes_inicio if ano == ano_inicio else 1
        mes_end = mes_anterior if ano == ano_anterior else 12
        
        for mes in range(mes_start, mes_end + 1):
            nome_mes_min = meses_pt[mes]  # minúsculo
            nome_mes_mai = meses_pt_maiusculo[mes]  # maiúsculo
            
            # Nome base padrão (minúsculo)
            nome_base = f"{nome_mes_min}{ano}"
            pasta_base = f"../docs/monitor_seca/{mes:02d}{ano}"
            arquivo_zip = os.path.join(pasta_base, f"{nome_base}.zip")
            pasta_extraida = os.path.join(pasta_base, nome_base)
            
            # Criar pasta base se não existir
            os.makedirs(pasta_base, exist_ok=True)
            
            print(f"\n📅 {mes:02d}/{ano}")
            
            # Verificar se arquivo já existe
            if os.path.exists(arquivo_zip):
                tamanho = os.path.getsize(arquivo_zip) / (1024 * 1024)
                print(f"   ✅ ZIP existe ({tamanho:.1f} MB)")
                
                # Verificar se já foi extraído
                if os.path.exists(pasta_extraida) and os.listdir(pasta_extraida):
                    print(f"   📂 Já extraído em: {nome_base}/")
                else:
                    print(f"   📦 Extraindo para {nome_base}/...")
                    try:
                        # Criar pasta de extração
                        os.makedirs(pasta_extraida, exist_ok=True)
                        
                        # Extrair ZIP diretamente para a pasta
                        with zipfile.ZipFile(arquivo_zip, 'r') as zip_ref:
                            zip_ref.extractall(pasta_extraida)
                        
                        # Listar arquivos extraídos
                        arquivos = os.listdir(pasta_extraida)
                        print(f"   ✅ Extraído: {len(arquivos)} arquivos em {nome_base}/")
                        
                    except Exception as e:
                        print(f"   ❌ Erro ao extrair: {e}")
                
                total_encontrados += 1
                
            else:
                # Tentar encontrar o arquivo com todas as URLs possíveis
                print(f"   🔍 Procurando arquivo...")
                
                # Primeiro tentar com nome minúsculo
                sucesso, url_encontrada, tamanho = tentar_todas_urls(mes, ano, nome_mes_min, pasta_base)
                
                # Se não encontrou, tentar com nome maiúsculo
                if not sucesso:
                    sucesso, url_encontrada, tamanho = tentar_todas_urls(mes, ano, nome_mes_mai, pasta_base)
                
                if sucesso:
                    # Extrair o arquivo baixado
                    print(f"   📦 Extraindo para {nome_base}/...")
                    try:
                        # Criar pasta de extração
                        os.makedirs(pasta_extraida, exist_ok=True)
                        
                        # Extrair ZIP diretamente para a pasta
                        with zipfile.ZipFile(arquivo_zip, 'r') as zip_ref:
                            zip_ref.extractall(pasta_extraida)
                        
                        # Listar arquivos extraídos
                        arquivos = os.listdir(pasta_extraida)
                        print(f"   ✅ Extraído: {len(arquivos)} arquivos em {nome_base}/")
                        
                    except Exception as e:
                        print(f"   ❌ Erro ao extrair: {e}")
                    
                    total_encontrados += 1
                else:
                    print(f"   ❌ Nenhuma URL funcionou")
                    total_nao_encontrados += 1
    
    print("\n" + "=" * 60)
    print("📊 RESUMO FINAL:")
    print(f"   ✅ Arquivos encontrados: {total_encontrados}")
    print(f"   ❌ Arquivos não encontrados: {total_nao_encontrados}")

# Executar versão detalhada
baixar_automatico_detalhado()

Maior período: 10/2025
📅 Período: 10/2025 a 10/2025

📅 10/2025
   ✅ ZIP existe (1.5 MB)
   📂 Já extraído em: outubro2025/

📊 RESUMO FINAL:
   ✅ Arquivos encontrados: 1
   ❌ Arquivos não encontrados: 0


In [35]:
import requests, os, zipfile, json
from datetime import datetime, timedelta

def tentar_todas_urls(mes, ano, nome_mes, pasta_base):
    urls_possiveis = [
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes}{ano}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.lower()}{ano}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.upper()}{ano}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.capitalize()}{ano}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes}{str(ano)[-2:]}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.lower()}{str(ano)[-2:]}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.upper()}{str(ano)[-2:]}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.capitalize()}{str(ano)[-2:]}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{mes:02d}{ano}.zip",
        f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{mes:02d}{str(ano)[-2:]}.zip",
    ]
    
    arquivo_zip = os.path.join(pasta_base, f"{nome_mes.lower()}{ano}.zip")
    for url in urls_possiveis:
        try:
            print(f"   🔍 Tentando: {os.path.basename(url)}")
            response = requests.head(url, timeout=10)
            if response.status_code == 200:
                print(f"   ✅ Encontrado: {os.path.basename(url)}")
                response_download = requests.get(url, timeout=30)
                if response_download.status_code == 200:
                    with open(arquivo_zip, 'wb') as f: f.write(response_download.content)
                    print(f"   📥 Baixado ({len(response_download.content) / (1024 * 1024):.1f} MB)")
                    return True, url, len(response_download.content) / (1024 * 1024)
        except requests.exceptions.RequestException: continue
    return False, None, 0

def baixar_automatico_detalhado():
    dados = json.load(open("../docs/monitor_seca/periodos_disponiveis.json", 'r', encoding='utf-8'))
    dados['periodos_disponiveis'].sort(key=lambda x: (int(x[2:]), int(x[:2])))
    maior = max(dados['periodos_disponiveis'], key=lambda x: (int(x[2:]), int(x[:2])))
    mes_inicio, ano_inicio = int(maior[:2]), int(maior[2:])
    
    data_atual = datetime.now()
    ultimo_dia_mes_anterior = data_atual.replace(day=1) - timedelta(days=1)
    mes_anterior, ano_anterior = ultimo_dia_mes_anterior.month, ultimo_dia_mes_anterior.year
    
    meses_pt = {1:'janeiro',2:'fevereiro',3:'marco',4:'abril',5:'maio',6:'junho',7:'julho',8:'agosto',9:'setembro',10:'outubro',11:'novembro',12:'dezembro'}
    meses_pt_maiusculo = {1:'Janeiro',2:'Fevereiro',3:'Marco',4:'Abril',5:'Maio',6:'Junho',7:'Julho',8:'Agosto',9:'Setembro',10:'Outubro',11:'Novembro',12:'Dezembro'}
    
    print(f"📅 Período: {mes_inicio:02d}/{ano_inicio} a {mes_anterior:02d}/{ano_anterior}\n" + "=" * 60)
    
    total_encontrados = total_nao_encontrados = 0
    
    for ano in range(ano_inicio, ano_anterior + 1):
        for mes in range(mes_inicio if ano == ano_inicio else 1, (mes_anterior if ano == ano_anterior else 12) + 1):
            nome_mes_min, nome_mes_mai = meses_pt[mes], meses_pt_maiusculo[mes]
            nome_base, pasta_base = f"{nome_mes_min}{ano}", f"../docs/monitor_seca/{mes:02d}{ano}"
            arquivo_zip, pasta_extraida = os.path.join(pasta_base, f"{nome_base}.zip"), os.path.join(pasta_base, nome_base)
            
            os.makedirs(pasta_base, exist_ok=True)
            print(f"\n📅 {mes:02d}/{ano}")
            
            if os.path.exists(arquivo_zip):
                print(f"   ✅ ZIP existe ({os.path.getsize(arquivo_zip) / (1024 * 1024):.1f} MB)")
                if os.path.exists(pasta_extraida) and os.listdir(pasta_extraida):
                    print(f"   📂 Já extraído em: {nome_base}/")
                else:
                    print(f"   📦 Extraindo para {nome_base}/...")
                    try:
                        os.makedirs(pasta_extraida, exist_ok=True)
                        with zipfile.ZipFile(arquivo_zip, 'r') as zip_ref: zip_ref.extractall(pasta_extraida)
                        print(f"   ✅ Extraído: {len(os.listdir(pasta_extraida))} arquivos em {nome_base}/")
                    except Exception as e: print(f"   ❌ Erro ao extrair: {e}")
                total_encontrados += 1
            else:
                print(f"   🔍 Procurando arquivo...")
                sucesso, url_encontrada, tamanho = tentar_todas_urls(mes, ano, nome_mes_min, pasta_base)
                if not sucesso: sucesso, url_encontrada, tamanho = tentar_todas_urls(mes, ano, nome_mes_mai, pasta_base)
                
                if sucesso:
                    print(f"   📦 Extraindo para {nome_base}/...")
                    try:
                        os.makedirs(pasta_extraida, exist_ok=True)
                        with zipfile.ZipFile(arquivo_zip, 'r') as zip_ref: zip_ref.extractall(pasta_extraida)
                        print(f"   ✅ Extraído: {len(os.listdir(pasta_extraida))} arquivos em {nome_base}/")
                    except Exception as e: print(f"   ❌ Erro ao extrair: {e}")
                    total_encontrados += 1
                else:
                    print(f"   ❌ Nenhuma URL funcionou")
                    total_nao_encontrados += 1
    
    print(f"\n{'=' * 60}\n📊 RESUMO FINAL:\n   ✅ Arquivos encontrados: {total_encontrados}\n   ❌ Arquivos não encontrados: {total_nao_encontrados}")

baixar_automatico_detalhado()

📅 Período: 10/2025 a 10/2025

📅 10/2025
   ✅ ZIP existe (1.5 MB)
   📂 Já extraído em: outubro2025/

📊 RESUMO FINAL:
   ✅ Arquivos encontrados: 1
   ❌ Arquivos não encontrados: 0


In [53]:
import zipfile
import os
from pathlib import Path

def extrair_zip_simplificado(caminho_zip, pasta_destino=None):
    
    caminho_zip = Path(caminho_zip)
    
    if not caminho_zip.exists():
        print(f"❌ Arquivo ZIP não encontrado: {caminho_zip}")
        return None
    
    # Definir pasta de destino
    if pasta_destino is None:
        nome_zip = caminho_zip.stem  # Nome sem extensão .zip
        pasta_destino = caminho_zip.parent / nome_zip
    else:
        pasta_destino = Path(pasta_destino)
    
    # Criar pasta de destino
    pasta_destino.mkdir(parents=True, exist_ok=True)
    
    print(f"📦 Extraindo: {caminho_zip.name}")
    print(f"📁 Destino: {pasta_destino}")
    
    try:
        with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
            # Lista todos os itens no ZIP
            itens_zip = zip_ref.namelist()
            
            # Verificar se há apenas uma pasta raiz no ZIP
            pastas_raiz = set()
            for item in itens_zip:
                primeira_pasta = item.split('/')[0]
                if primeira_pasta and not item.endswith('/'):
                    pastas_raiz.add(primeira_pasta)
            
            print(f"📊 Itens no ZIP: {len(itens_zip)}")
            print(f"📂 Pastas raiz detectadas: {list(pastas_raiz)}")
            
            # Se há apenas uma pasta raiz e ela contém todos os arquivos
            if len(pastas_raiz) == 1:
                pasta_raiz = list(pastas_raiz)[0]
                print(f"🔄 Detectada estrutura com subpasta: {pasta_raiz}")
                
                # Extrair todos os arquivos, ignorando a pasta raiz
                for item in itens_zip:
                    if not item.endswith('/'):  # Ignorar pastas vazias
                        # Remover a pasta raiz do caminho
                        caminho_relativo = item.replace(pasta_raiz + '/', '', 1)
                        
                        # Se o caminho ainda tem subpastas, criar estrutura
                        if '/' in caminho_relativo:
                            subpasta = pasta_destino / os.path.dirname(caminho_relativo)
                            subpasta.mkdir(parents=True, exist_ok=True)
                        
                        # Extrair arquivo
                        caminho_final = pasta_destino / caminho_relativo
                        with open(caminho_final, 'wb') as f:
                            f.write(zip_ref.read(item))
                        
                        print(f"   ✅ Extraído: {caminho_relativo}")
            else:
                # Estrutura plana - extrair normalmente
                print("🔄 Estrutura plana detectada")
                zip_ref.extractall(pasta_destino)
                for item in itens_zip:
                    if not item.endswith('/'):
                        print(f"   ✅ Extraído: {item}")
            
            # Contar arquivos extraídos
            arquivos_extraidos = list(pasta_destino.rglob('*'))
            arquivos_extraidos = [f for f in arquivos_extraidos if f.is_file()]
            
            print(f"🎯 Total de arquivos extraídos: {len(arquivos_extraidos)}")
            print(f"📁 Pasta final: {pasta_destino}")
            
            return str(pasta_destino)
            
    except zipfile.BadZipFile:
        print(f"❌ Arquivo ZIP corrompido: {caminho_zip}")
        return None
    except Exception as e:
        print(f"❌ Erro ao extrair {caminho_zip}: {e}")
        return None

# Exemplo de uso
if __name__ == "__main__":
    # Exemplo 1: Extrair um ZIP específico
    caminho_zip = "../docs/monitor_seca/012023/janeiro2023.zip"
    pasta_extraida = extrair_zip_simplificado(caminho_zip)
    
    if pasta_extraida:
        print(f"✅ Extraído em: {pasta_extraida}")

📦 Extraindo: janeiro2023.zip
📁 Destino: ..\docs\monitor_seca\012023\janeiro2023
📊 Itens no ZIP: 31
📂 Pastas raiz detectadas: ['janeiro2023']
🔄 Detectada estrutura com subpasta: janeiro2023
   ✅ Extraído: 202301_MS_IMPACTOS.qml
   ✅ Extraído: janeiro23.dbf
   ✅ Extraído: 202301_MS_IMPACTOS.shp
   ✅ Extraído: janeiro23.cpg
   ✅ Extraído: janeiro23.shp
   ✅ Extraído: janeiro23.qgs~
   ✅ Extraído: 202301_MS_IMPACTOS_TIPO.dbf
   ✅ Extraído: 202301_MS_IMPACTOS.qpj
   ✅ Extraído: 202301_MS_IMPACTOS.dbf
   ✅ Extraído: janeiro23.qml
   ✅ Extraído: 202301_MS_IMPACTOS_TIPO.cpg
   ✅ Extraído: 202301_MS_IMPACTOS.cpg
   ✅ Extraído: 202301_MS_IMPACTOS_TIPO.prj
   ✅ Extraído: janeiro23.shx
   ✅ Extraído: janeiro2023.qgs~
   ✅ Extraído: janeiro23.prj
   ✅ Extraído: 202301_MS_IMPACTOS.prj
   ✅ Extraído: 202301_MS_IMPACTOS_TIPO.qpj
   ✅ Extraído: 202301_MS_IMPACTOS_TIPO.qml
   ✅ Extraído: 202301_MS_IMPACTOS_TIPO.shx
   ✅ Extraído: 202301_MS_IMPACTOS_TIPO.shp
   ✅ Extraído: 202301_MS_IMPACTOS.shx
   ✅ Ext

In [15]:
import geopandas as gpd
import pandas as pd
import os
import json
import requests
from shapely.ops import unary_union
from pyproj import Geod
from pathlib import Path
import zipfile
from datetime import datetime, timedelta

print("=== ISOLANDO MANCHAS DE SECA POR CATEGORIA ===")

caminho_shp_seca = '../docs/monitor_seca/072022/julho2022/julho22.shp'
caminho_shp_maranhao = "../docs/Shapefile/MA_Municipios_2022_limite/Limite_MA_2022.shp"

# Verifique se os arquivos existem
print(f"Arquivo seca existe: {os.path.exists(caminho_shp_seca)}")
print(f"Arquivo MA existe: {os.path.exists(caminho_shp_maranhao)}")

try:
    gdf_seca = gpd.read_file(caminho_shp_seca)
    print("✓ Shapefile de seca carregado com sucesso")
except Exception as e:
    print(f"✗ Erro ao carregar shapefile de seca: {e}")
    exit()

try:
    gdf_maranhao = gpd.read_file(caminho_shp_maranhao)
    print("✓ Shapefile do MA carregado com sucesso")
except Exception as e:
    print(f"✗ Erro ao carregar shapefile do MA: {e}")
    exit()

print(f"CRS seca: {gdf_seca.crs}")
print(f"CRS Maranhão: {gdf_maranhao.crs}")
print(f"Features originais: {len(gdf_seca)}")
print(f"Colunas disponíveis: {list(gdf_seca.columns)}")
print(f"Valores únicos de seca: {sorted(gdf_seca['Valor'].unique())}")

# Verifique a estrutura dos dados
print("\nAmostra dos dados de seca:")
print(gdf_seca.head(3))

print(f"\n=== VERIFICAÇÃO ESPECÍFICA JULHO/2022 ===")
print(f"Total de geometrias válidas: {gdf_seca.geometry.is_valid.sum()}/{len(gdf_seca)}")
print(f"Geometrias vazias: {gdf_seca.geometry.is_empty.sum()}")

# Verifique se há geometrias inválidas
if not gdf_seca.geometry.is_valid.all():
    print("Corrigindo geometrias inválidas...")
    gdf_seca.geometry = gdf_seca.geometry.buffer(0)
    print(f"Após correção - Geometrias válidas: {gdf_seca.geometry.is_valid.sum()}/{len(gdf_seca)}")

# Agora tente o overlay com tratamento de erro
print(f"\n=== TENTANDO OVERLAY ===")
try:
    if gdf_seca.crs != gdf_maranhao.crs:
        print("Convertendo CRS...")
        gdf_seca = gdf_seca.to_crs(gdf_maranhao.crs)
    
    gdf_seca_maranhao = gpd.overlay(
        gdf_seca, 
        gdf_maranhao, 
        how='intersection', 
        keep_geom_type=False,
        make_valid=True
    )
    print(f"✓ Overlay concluído - Features após recorte: {len(gdf_seca_maranhao)}")
    
    # DEBUG: Mostre os dados resultantes do overlay
    print(f"\n=== DADOS DO OVERLAY ===")
    print(f"Colunas do overlay: {list(gdf_seca_maranhao.columns)}")
    print(f"Tipos de dados das colunas:")
    print(gdf_seca_maranhao.dtypes)
    print(f"\nAmostra dos dados do overlay:")
    print(gdf_seca_maranhao.head())
    
    if len(gdf_seca_maranhao) == 0:
        print("⚠️ AVISO: Nenhuma feature resultou do overlay!")
        
except Exception as e:
    print(f"✗ Erro no overlay: {e}")
    exit()

# =============================================================================
# CORREÇÃO DO PROBLEMA - CONVERSÃO DE TIPOS DE DADOS
# =============================================================================

print(f"\n=== PROCESSANDO MANCHAS DE SECA ===")

# CORREÇÃO: Converta a coluna 'Valor' para inteiro para garantir comparação correta
print("Convertendo coluna 'Valor' para inteiro...")
gdf_seca_maranhao['Valor'] = gdf_seca_maranhao['Valor'].astype(int)

# Verifique os valores que realmente têm interseção
valores_com_intersecao = gdf_seca_maranhao['Valor'].unique()
print(f"Valores com interseção no MA: {sorted(valores_com_intersecao)}")
print(f"Tipo dos valores: {type(valores_com_intersecao[0]) if len(valores_com_intersecao) > 0 else 'N/A'}")

# Ordem de gravidade (mais grave primeiro)
ordem_gravidade = [5, 4, 3, 2, 1, 0]
geometrias_por_categoria = {}

print(f"\n=== PROCESSANDO POR CATEGORIA DE SECA ===")

for valor in ordem_gravidade:
    print(f"\nProcessando seca Valor {valor}...")
    
    # Filtre as features para este valor - AGORA COM INTEIROS
    seca_filtrada = gdf_seca_maranhao[gdf_seca_maranhao['Valor'] == valor].copy()
    
    print(f"  Tentando filtrar Valor {valor} (tipo: {type(valor)})")
    print(f"  Valores únicos na coluna: {gdf_seca_maranhao['Valor'].unique()}")
    print(f"  Tipos na coluna: {gdf_seca_maranhao['Valor'].dtype}")
    
    if len(seca_filtrada) == 0:
        print(f"  Nenhuma feature encontrada para Valor {valor} no MA")
        geometrias_por_categoria[valor] = None
        continue
    
    print(f"  Encontradas {len(seca_filtrada)} features para Valor {valor}")
    
    try:
        # Una as geometrias deste valor
        geometria_unida = unary_union(seca_filtrada.geometry)
        
        # Se a geometria unida for vazia, pule
        if geometria_unida.is_empty:
            print(f"  Geometria unida vazia para Valor {valor}")
            geometrias_por_categoria[valor] = None
            continue
            
        geometria_final = geometria_unida
        
        # Subtraia as categorias mais graves
        for valor_mais_grave in [v for v in ordem_gravidade if v > valor]:
            if (valor_mais_grave in geometrias_por_categoria and 
                geometrias_por_categoria[valor_mais_grave] is not None):
                
                try:
                    geometria_final = geometria_final.difference(geometrias_por_categoria[valor_mais_grave])
                    
                    if geometria_final.is_empty:
                        print(f"  Geometria tornou-se vazia após subtrair Valor {valor_mais_grave}")
                        break
                        
                except Exception as e:
                    print(f"  Erro na diferença para Valor {valor} - {valor_mais_grave}: {e}")
                    continue
        
        # Só armazene se não estiver vazia
        if not geometria_final.is_empty:
            geometrias_por_categoria[valor] = geometria_final
            print(f"  ✓ Geometria processada para Valor {valor} - Área: {geometria_final.area:.2f}")
        else:
            geometrias_por_categoria[valor] = None
            print(f"  ✗ Geometria vazia para Valor {valor}")
        
    except Exception as e:
        print(f"  Erro ao processar Valor {valor}: {e}")
        geometrias_por_categoria[valor] = None

# Crie o GeoDataFrame final
print(f"\n=== CRIANDO GEODataFrame FINAL ===")
features_finais = []

for valor, geometria in geometrias_por_categoria.items():
    if geometria is not None and not geometria.is_empty:
        # Para geometrias MultiPolygon, crie uma feature para cada polígono
        if geometria.geom_type == 'MultiPolygon':
            for poly in geometria.geoms:
                features_finais.append({
                    'Valor': valor,
                    'geometry': poly
                })
        else:
            features_finais.append({
                'Valor': valor,
                'geometry': geometria
            })

if features_finais:
    gdf_final = gpd.GeoDataFrame(features_finais, crs=gdf_maranhao.crs)
    print(f"✓ GeoDataFrame final criado com {len(gdf_final)} features")
    
    # Mostre resumo
    print(f"\n=== RESUMO FINAL ===")
    for valor in ordem_gravidade:
        count = len([f for f in features_finais if f['Valor'] == valor])
        area_total = sum([f['geometry'].area for f in features_finais if f['Valor'] == valor])
        if count > 0:
            print(f"Valor {valor}: {count} feature(s) - Área total: {area_total:.2f}")
    
    # Salve o resultado se desejar
    caminho_saida = "../docs/monitor_seca/072022/manchas_seca_ma_julho2022.shp"
    
    # Certifique-se de que o diretório existe
    os.makedirs(os.path.dirname(caminho_saida), exist_ok=True)
    
    gdf_final.to_file(caminho_saida)
    print(f"✓ Arquivo salvo em: {caminho_saida}")
    
    # Mostre estatísticas finais
    print(f"\n=== ESTATÍSTICAS FINAIS ===")
    print(f"Total de features originais: {len(gdf_seca)}")
    print(f"Features no MA após overlay: {len(gdf_seca_maranhao)}")
    print(f"Features finais processadas: {len(gdf_final)}")
    
else:
    print("⚠️ Nenhuma feature final foi criada")
    print("\n=== DIAGNÓSTICO DO PROBLEMA ===")
    print("Possíveis causas:")
    print("1. As geometrias resultantes do overlay são muito pequenas/vazias")
    print("2. Problema na união das geometrias")
    print("3. Todas as áreas foram subtraídas por categorias mais graves")
    
    # Debug adicional
    print(f"\nGeometrias por categoria: {geometrias_por_categoria}")
    gdf_final = gpd.GeoDataFrame()

print(f"\nFeatures finais: {len(features_finais)}")

=== ISOLANDO MANCHAS DE SECA POR CATEGORIA ===
Arquivo seca existe: True
Arquivo MA existe: True
✓ Shapefile de seca carregado com sucesso
✓ Shapefile do MA carregado com sucesso
CRS seca: EPSG:4326
CRS Maranhão: EPSG:4674
Features originais: 6
Colunas disponíveis: ['uf_codigo', 'Valor', 'geometry']
Valores únicos de seca: ['0', '1', '2', '3', '4', '5']

Amostra dos dados de seca:
  uf_codigo Valor                                           geometry
0        si     0  MULTIPOLYGON (((-50.22475 -9.84116, -50.22337 ...
1        s0     1  MULTIPOLYGON (((-50.22475 -9.84116, -50.22337 ...
2        s1     2  MULTIPOLYGON (((-37.32475 -8.40614, -37.34913 ...

=== VERIFICAÇÃO ESPECÍFICA JULHO/2022 ===
Total de geometrias válidas: 6/6
Geometrias vazias: 0

=== TENTANDO OVERLAY ===
Convertendo CRS...
✓ Overlay concluído - Features após recorte: 2

=== DADOS DO OVERLAY ===
Colunas do overlay: ['uf_codigo', 'Valor', 'CD_UF', 'NM_UF', 'SIGLA_UF', 'NM_REGIAO', 'AREA_KM2', 'geometry']
Tipos de dados 

In [ ]:
#caminho_shp_seca = '../docs/monitor_seca/072022/julho2022/julho22.shp'
caminho_shp_seca = '../docs/monitor_seca/102025/outubro2025/outubro2025.shp'
caminho_shp_maranhao = "../docs/Shapefile/MA_Municipios_2022_limite/Limite_MA_2022.shp"

gdf_seca = gpd.read_file(caminho_shp_seca)
gdf_maranhao = gpd.read_file(caminho_shp_maranhao)


# DEBUG: Mostre os dados resultantes do overlay
print("\n=== DADOS DO OVERLAY ===")
print(f"Colunas do overlay: {list(gdf_seca.columns)}")

print("Tipos de dados das colunas:")
print(gdf_seca.dtypes)
print("\nAmostra dos dados do overlay:")
print(gdf_seca.head())


=== DADOS DO OVERLAY ===
Colunas do overlay: ['uf_codigo', 'Valor', 'geometry']
Tipos de dados das colunas:
uf_codigo      object
Valor          object
geometry     geometry
dtype: object

Amostra dos dados do overlay:
  uf_codigo Valor                                           geometry
0        si     0  MULTIPOLYGON (((-50.22475 -9.84116, -50.22337 ...
1        s0     1  MULTIPOLYGON (((-50.22475 -9.84116, -50.22337 ...
2        s1     2  MULTIPOLYGON (((-37.32475 -8.40614, -37.34913 ...
3        s2     3  MULTIPOLYGON (((-56.3398 -22.19259, -56.34444 ...
4        s3     4  MULTIPOLYGON (((-52.16367 -21.75737, -52.20544...


In [20]:
#caminho_shp_seca = '../docs/monitor_seca/072022/julho2022/julho22.shp'
caminho_shp_seca = '../docs/monitor_seca/102025/outubro2025/outubro25.shp'
caminho_shp_maranhao = "../docs/Shapefile/MA_Municipios_2022_limite/Limite_MA_2022.shp"

gdf_seca = gpd.read_file(caminho_shp_seca)
gdf_maranhao = gpd.read_file(caminho_shp_maranhao)


# DEBUG: Mostre os dados resultantes do overlay
print("\n=== DADOS DO OVERLAY ===")
print(f"Colunas do overlay: {list(gdf_seca.columns)}")

print("Tipos de dados das colunas:")
print(gdf_seca.dtypes)
print("\nAmostra dos dados do overlay:")
print(gdf_seca.head())


=== DADOS DO OVERLAY ===
Colunas do overlay: ['uf_codigo', 'Valor', 'geometry']
Tipos de dados das colunas:
uf_codigo      object
Valor           int64
geometry     geometry
dtype: object

Amostra dos dados do overlay:
  uf_codigo  Valor                                           geometry
0        si      0  MULTIPOLYGON (((-60.82852 -13.62951, -60.84103...
1        s0      1  MULTIPOLYGON (((-48.5643 -27.69824, -48.56393 ...
2        s1      2  MULTIPOLYGON (((-46.90521 -24.39653, -46.90667...
3        s2      3  MULTIPOLYGON (((-48.94165 -24.53139, -48.94696...
4        s3      4  POLYGON ((-45.2948 -12.66429, -45.26215 -12.66...


In [ ]:
import geopandas as gpd
import pandas as pd
import os
import json
import requests
from shapely.ops import unary_union
from pyproj import Geod
from pathlib import Path
import zipfile
from datetime import datetime, timedelta
import numpy as np
import time
import logging
import sqlite3
import concurrent.futures
import sys
import schedule
from typing import Tuple
from bs4 import BeautifulSoup
import io
import re
import threading
from urllib.parse import unquote

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('config/sistema_integrado.log', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# ========== CONFIGURAÇÕES UNIFICADAS ==========
CONFIG = {
    # Configurações do coletor de dados
    'caminho_db': 'database/BD_SIMA_MA.db',
    'arquivo_estacoes': 'docs/SIMA - Cotas de referencia.xlsx',
    'intervalo_execucao': 10,  # minutos - AMBOS os sistemas
    'max_workers': 1,
    'timeout_requisicao': 30,
    'dias_retrocesso_fallback': 5,
    
    # Configurações do sistema de notícias
    'config_dir': 'config',
    'file_id_noticias': '11GddAKmH8-J0bd15dxy4fxt2mwtj_68y',
    'caminho_base_imagens': 'assets/images/noticias',
    'imagem_padrao': '00',
    'file_id_imagens': '1Ekhthm3iTyHVbiBqr9NVjpggtigNV1_Q',
    'sheet_url': "https://docs.google.com/spreadsheets/d/1DPYeK-GiAFs9dMStVoYsYqnkdpewlPXz/edit?usp=drive_link&ouid=117844771079452586&rtpof=true&sd=true",
    
    # Configurações do sistema de seca
    'condicao_download': "sim"  # Altere para "nao" se não quiser fazer download
}

# ============================================================================
# SISTEMA DE MONITORAMENTO DE SECA (CÓDIGO ORIGINAL)
# ============================================================================

# Variável condicional - altere para "sim" ou "nao"
CONDICAO_DOWNLOAD = CONFIG['condicao_download']  # Altere para "nao" se não quiser fazer download

# ============================================================================
# FUNÇÃO PARA CONVERSÃO INTELIGENTE DA COLUNA VALOR
# ============================================================================
def converter_coluna_valor_inteligente(gdf_seca):
    """
    Analisa e converte a coluna 'Valor' para inteiro de forma inteligente
    """
    print("🔍 Analisando coluna 'Valor'...")
    
    # Verificar se a coluna 'Valor' existe
    if 'Valor' not in gdf_seca.columns:
        print("❌ Coluna 'Valor' não encontrada no shapefile de seca")
        print(f"   Colunas disponíveis: {list(gdf_seca.columns)}")
        return gdf_seca
    
    # Analisar o tipo atual da coluna
    tipo_atual = gdf_seca['Valor'].dtype
    print(f"   Tipo atual da coluna 'Valor': {tipo_atual}")
    
    # Mostrar valores únicos antes da conversão
    valores_unicos = sorted(gdf_seca['Valor'].unique())
    print(f"   Valores únicos antes da conversão: {valores_unicos}")
    
    # Verificar se já é numérico
    if pd.api.types.is_numeric_dtype(gdf_seca['Valor']):
        print("   ✅ Coluna 'Valor' já é numérica")
        # Garantir que seja inteiro
        gdf_seca['Valor'] = gdf_seca['Valor'].astype(int)
        print(f"   Valores únicos após garantir inteiro: {sorted(gdf_seca['Valor'].unique())}")
        return gdf_seca
    
    # Se for string/object, tentar conversão
    if gdf_seca['Valor'].dtype == 'object':
        print("   🔄 Convertendo coluna 'Valor' de string para inteiro...")
        
        # Tentar conversão direta
        try:
            gdf_seca['Valor'] = gdf_seca['Valor'].astype(int)
            print(f"   ✅ Conversão direta bem-sucedida")
            print(f"   Valores únicos após conversão: {sorted(gdf_seca['Valor'].unique())}")
            return gdf_seca
        except (ValueError, TypeError) as e:
            print(f"   ⚠️ Conversão direta falhou: {e}")
            print("   🔄 Tentando conversão com tratamento de erros...")
            
            # Tentar conversão com tratamento de valores problemáticos
            try:
                # Substituir valores vazios/NaN por 0
                gdf_seca['Valor'] = gdf_seca['Valor'].fillna('0')
                
                # Remover espaços em branco
                gdf_seca['Valor'] = gdf_seca['Valor'].astype(str).str.strip()
                
                # Converter para inteiro
                gdf_seca['Valor'] = gdf_seca['Valor'].astype(int)
                
                print(f"   ✅ Conversão com tratamento bem-sucedida")
                print(f"   Valores únicos após conversão: {sorted(gdf_seca['Valor'].unique())}")
                return gdf_seca
                
            except (ValueError, TypeError) as e2:
                print(f"   ❌ Todas as tentativas de conversão falharam: {e2}")
                print("   ⚠️ Mantendo coluna como string")
    
    return gdf_seca

# ============================================================================
# FUNÇÃO DE EXTRAÇÃO DE ZIP MELHORADA
# ============================================================================
def extrair_zip_simplificado(caminho_zip, pasta_destino=None):
    """
    Extrai arquivos ZIP para uma pasta com o mesmo nome do arquivo ZIP.
    Se houver subpastas dentro do ZIP, move todos os arquivos para a pasta raiz.
    """
    caminho_zip = Path(caminho_zip)
    
    if not caminho_zip.exists():
        print(f"❌ Arquivo ZIP não encontrado: {caminho_zip}")
        return None
    
    # Definir pasta de destino
    if pasta_destino is None:
        nome_zip = caminho_zip.stem
        pasta_destino = caminho_zip.parent / nome_zip
    else:
        pasta_destino = Path(pasta_destino)
    
    # Criar pasta de destino
    pasta_destino.mkdir(parents=True, exist_ok=True)
    
    print(f"📦 Extraindo: {caminho_zip.name}")
    print(f"📁 Destino: {pasta_destino}")
    
    try:
        with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
            # Lista todos os itens no ZIP
            itens_zip = zip_ref.namelist()
            
            # Verificar se há apenas uma pasta raiz no ZIP
            pastas_raiz = set()
            for item in itens_zip:
                primeira_pasta = item.split('/')[0]
                if primeira_pasta and not item.endswith('/'):
                    pastas_raiz.add(primeira_pasta)
            
            print(f"📊 Itens no ZIP: {len(itens_zip)}")
            print(f"📂 Pastas raiz detectadas: {list(pastas_raiz)}")
            
            # Se há apenas uma pasta raiz e ela contém todos os arquivos
            if len(pastas_raiz) == 1:
                pasta_raiz = list(pastas_raiz)[0]
                print(f"🔄 Detectada estrutura com subpasta: {pasta_raiz}")
                
                # Extrair todos os arquivos, ignorando a pasta raiz
                for item in itens_zip:
                    if not item.endswith('/'):
                        # Remover a pasta raiz do caminho
                        caminho_relativo = item.replace(pasta_raiz + '/', '', 1)
                        
                        # Se o caminho ainda tem subpastas, criar estrutura
                        if '/' in caminho_relativo:
                            subpasta = pasta_destino / os.path.dirname(caminho_relativo)
                            subpasta.mkdir(parents=True, exist_ok=True)
                        
                        # Extrair arquivo
                        caminho_final = pasta_destino / caminho_relativo
                        with open(caminho_final, 'wb') as f:
                            f.write(zip_ref.read(item))
                        
                        print(f"   ✅ Extraído: {caminho_relativo}")
            else:
                # Estrutura plana - extrair normalmente
                print("🔄 Estrutura plana detectada")
                zip_ref.extractall(pasta_destino)
                for item in itens_zip:
                    if not item.endswith('/'):
                        print(f"   ✅ Extraído: {item}")
            
            # Contar arquivos extraídos
            arquivos_extraidos = list(pasta_destino.rglob('*'))
            arquivos_extraidos = [f for f in arquivos_extraidos if f.is_file()]
            
            print(f"🎯 Total de arquivos extraídos: {len(arquivos_extraidos)}")
            print(f"📁 Pasta final: {pasta_destino}")
            
            return str(pasta_destino)
            
    except zipfile.BadZipFile:
        print(f"❌ Arquivo ZIP corrompido: {caminho_zip}")
        return None
    except Exception as e:
        print(f"❌ Erro ao extrair {caminho_zip}: {e}")
        return None

# ============================================================================
# FUNÇÕES DO SISTEMA DE SECA
# ============================================================================

def executar_download_seca():
    """Executa o download dos arquivos de seca se condição for atendida"""
    if CONFIG['condicao_download'] != "sim":
        print("⏭️  Download de seca pulou (CONDICAO_DOWNLOAD = 'nao')")
        return False

    print("🚀 INICIANDO DOWNLOAD DOS ARQUIVOS DE SECA...")

    def tentar_todas_urls(mes, ano, nome_mes, pasta_base):
        urls_possiveis = [
            f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes}{ano}.zip",
            f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.lower()}{ano}.zip",
            f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.upper()}{ano}.zip",
            f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.capitalize()}{ano}.zip",
            f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes}{str(ano)[-2:]}.zip",
            f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.lower()}{str(ano)[-2:]}.zip",
            f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.upper()}{str(ano)[-2:]}.zip",
            f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{nome_mes.capitalize()}{str(ano)[-2:]}.zip",
            f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{mes:02d}{ano}.zip",
            f"https://ana-monitor-secas-files.s3.sa-east-1.amazonaws.com/uploads/mapas/{mes:02d}{str(ano)[-2:]}.zip",
        ]
        
        arquivo_zip = os.path.join(pasta_base, f"{nome_mes.lower()}{ano}.zip")
        for url in urls_possiveis:
            try:
                print(f"   🔍 Tentando: {os.path.basename(url)}")
                response = requests.head(url, timeout=10)
                if response.status_code == 200:
                    print(f"   ✅ Encontrado: {os.path.basename(url)}")
                    response_download = requests.get(url, timeout=30)
                    if response_download.status_code == 200:
                        with open(arquivo_zip, 'wb') as f: 
                            f.write(response_download.content)
                        print(f"   📥 Baixado ({len(response_download.content) / (1024 * 1024):.1f} MB)")
                        return True, url, len(response_download.content) / (1024 * 1024)
            except requests.exceptions.RequestException:
                continue
        return False, None, 0

    def baixar_automatico_detalhado():
        dados = json.load(open("docs/monitor_seca/periodos_disponiveis.json", 'r', encoding='utf-8'))
        dados['periodos_disponiveis'].sort(key=lambda x: (int(x[2:]), int(x[:2])))
        maior = max(dados['periodos_disponiveis'], key=lambda x: (int(x[2:]), int(x[:2])))
        mes_inicio, ano_inicio = int(maior[:2]), int(maior[2:])
        #mes_inicio, ano_inicio = 1, 2022
        
        data_atual = datetime.now()
        ultimo_dia_mes_anterior = data_atual.replace(day=1) - timedelta(days=1)
        mes_anterior, ano_anterior = ultimo_dia_mes_anterior.month, ultimo_dia_mes_anterior.year
        
        # VERIFICAR SE ANOS SÃO IGUAIS - SE FOREM, PULA DOWNLOAD
        if ano_inicio == ano_anterior:
            print("📅 Anos iguais detectados - Pulando download")
            print(f"📍 Ano início: {ano_inicio}, Ano anterior: {ano_anterior}")
            return False
        
        meses_pt = {1:'janeiro',2:'fevereiro',3:'marco',4:'abril',5:'maio',6:'junho',7:'julho',8:'agosto',9:'setembro',10:'outubro',11:'novembro',12:'dezembro'}
        meses_pt_maiusculo = {1:'Janeiro',2:'Fevereiro',3:'Marco',4:'Abril',5:'Maio',6:'Junho',7:'Julho',8:'Agosto',9:'Setembro',10:'Outubro',11:'Novembro',12:'Dezembro'}
        
        print(f"📅 Período: {mes_inicio:02d}/{ano_inicio} a {mes_anterior:02d}/{ano_anterior}\n" + "=" * 60)
        
        total_encontrados = total_nao_encontrados = 0
        
        for ano in range(ano_inicio, ano_anterior + 1):
            for mes in range(mes_inicio if ano == ano_inicio else 1, (mes_anterior if ano == ano_anterior else 12) + 1):
                nome_mes_min, nome_mes_mai = meses_pt[mes], meses_pt_maiusculo[mes]
                nome_base, pasta_base = f"{nome_mes_min}{ano}", f"docs/monitor_seca/{mes:02d}{ano}"
                arquivo_zip, pasta_extraida = os.path.join(pasta_base, f"{nome_base}.zip"), os.path.join(pasta_base, nome_base)
                
                os.makedirs(pasta_base, exist_ok=True)
                print(f"\n📅 {mes:02d}/{ano}")
                
                if os.path.exists(arquivo_zip):
                    print(f"   ✅ ZIP existe ({os.path.getsize(arquivo_zip) / (1024 * 1024):.1f} MB)")
                    if os.path.exists(pasta_extraida) and os.listdir(pasta_extraida):
                        print(f"   📂 Já extraído em: {nome_base}/")
                    else:
                        print(f"   📦 Extraindo para {nome_base}/...")
                        try:
                            # NOVA EXTRAÇÃO - usando a função melhorada
                            pasta_final = extrair_zip_simplificado(arquivo_zip, pasta_extraida)
                            if pasta_final:
                                arquivos = len([f for f in Path(pasta_final).rglob('*') if f.is_file()])
                                print(f"   ✅ Extraído: {arquivos} arquivos em {nome_base}/")
                            else:
                                print("   ❌ Falha na extração")
                        except Exception as e: 
                            print(f"   ❌ Erro ao extrair: {e}")
                    total_encontrados += 1
                else:
                    print("   🔍 Procurando arquivo...")
                    sucesso, url_encontrada, tamanho = tentar_todas_urls(mes, ano, nome_mes_min, pasta_base)
                    if not sucesso: 
                        sucesso, url_encontrada, tamanho = tentar_todas_urls(mes, ano, nome_mes_mai, pasta_base)
                    
                    if sucesso:
                        print(f"   📦 Extraindo para {nome_base}/...")
                        try:
                            # NOVA EXTRAÇÃO - usando a função melhorada
                            pasta_final = extrair_zip_simplificado(arquivo_zip, pasta_extraida)
                            if pasta_final:
                                arquivos = len([f for f in Path(pasta_final).rglob('*') if f.is_file()])
                                print(f"   ✅ Extraído: {arquivos} arquivos em {nome_base}/")
                            else:
                                print("   ❌ Falha na extração")
                        except Exception as e: 
                            print(f"   ❌ Erro ao extrair: {e}")
                        total_encontrados += 1
                    else:
                        print("   ❌ Nenhuma URL funcionou")
                        total_nao_encontrados += 1
        
        print(f"\n{'=' * 60}\n📊 RESUMO FINAL:\n   ✅ Arquivos encontrados: {total_encontrados}\n   ❌ Arquivos não encontrados: {total_nao_encontrados}")
        return True

    # Executar o download (só se anos forem diferentes)
    fez_download = baixar_automatico_detalhado()
    if fez_download:
        print("✅ DOWNLOAD DE SECA CONCLUÍDO!")
    else:
        print("⏭️  Download de seca pulou (anos iguais)")
    return fez_download

def encontrar_pastas_faltantes():
    pasta_raiz = Path("docs/monitor_seca/")
    pastas_faltantes = []
    
    print(f"🔍 Procurando pastas faltantes em: {pasta_raiz}")
    
    if not pasta_raiz.exists():
        print("❌ Pasta não encontrada!")
        return []
    
    for subpasta in pasta_raiz.iterdir():
        if subpasta.is_dir():
            arquivo_geojson = subpasta / "seca_atributos.geojson"
            
            if not arquivo_geojson.exists():
                if len(subpasta.name) == 6:
                    try:
                        mes = int(subpasta.name[:2])
                        ano = int(subpasta.name[2:])
                        pastas_faltantes.append({'mes': mes, 'ano': ano, 'pasta': subpasta.name})
                        print(f"❌ {subpasta.name} → Mês: {mes}, Ano: {ano}")
                    except ValueError:
                        print(f"❌ {subpasta.name} → Formato inválido")
                else:
                    print(f"❌ {subpasta.name} → Formato diferente")
    
    print(f"\n📊 Total de pastas faltantes: {len(pastas_faltantes)}")
    return pastas_faltantes

def isolar_manchas_seca_por_categoria(caminho_shp_seca, caminho_shp_maranhao, caminho_saida):
    print("=== ISOLANDO MANCHAS DE SECA POR CATEGORIA ===")
    
    gdf_seca = gpd.read_file(caminho_shp_seca)
    gdf_maranhao = gpd.read_file(caminho_shp_maranhao)
    
    print(f"CRS seca: {gdf_seca.crs}")
    print(f"CRS Maranhão: {gdf_maranhao.crs}")
    print(f"Features originais: {len(gdf_seca)}")
    
    # NOVA FUNÇÃO: CONVERSÃO INTELIGENTE DA COLUNA VALOR
    gdf_seca = converter_coluna_valor_inteligente(gdf_seca)
    
    print(f"Valores únicos de seca após conversão: {sorted(gdf_seca['Valor'].unique())}")
    
    if gdf_seca.crs != gdf_maranhao.crs:
        gdf_seca = gdf_seca.to_crs(gdf_maranhao.crs)
    
    gdf_seca_maranhao = gpd.overlay(gdf_seca, gdf_maranhao, how='intersection', keep_geom_type=False)
    
    print(f"Features após recorte: {len(gdf_seca_maranhao)}")
    
    ordem_gravidade = [5, 4, 3, 2, 1, 0]
    geometrias_por_categoria = {}
    
    for valor in ordem_gravidade:
        print(f"\nProcessando seca Valor {valor}...")
        
        seca_filtrada = gdf_seca_maranhao[gdf_seca_maranhao['Valor'] == valor].copy()
        
        if len(seca_filtrada) == 0:
            print(f"  Nenhuma feature encontrada para Valor {valor}")
            geometrias_por_categoria[valor] = None
            continue
        
        try:
            geometria_unida = unary_union(seca_filtrada.geometry)
            geometria_final = geometria_unida
            
            for valor_mais_grave in [v for v in ordem_gravidade if v > valor]:
                if valor_mais_grave in geometrias_por_categoria and geometrias_por_categoria[valor_mais_grave] is not None:
                    try:
                        geometria_final = geometria_final.difference(geometrias_por_categoria[valor_mais_grave])
                        if geometria_final.is_empty:
                            # Geometria vazia, continuar para próximo valor
                            pass
                    except Exception as e:
                        print(f"  Erro na diferença para Valor {valor}: {e}")
                        continue
            
            geometrias_por_categoria[valor] = geometria_final if not geometria_final.is_empty else None
            print(f"  Geometria processada para Valor {valor}")
            
        except Exception as e:
            print(f"  Erro ao processar Valor {valor}: {e}")
            geometrias_por_categoria[valor] = None
    
    features_finais = []
    
    for valor in ordem_gravidade:
        if geometrias_por_categoria[valor] is not None and not geometrias_por_categoria[valor].is_empty:
            linha_base = gdf_seca_maranhao[gdf_seca_maranhao['Valor'] == valor].iloc[0].copy()
            
            nova_feature = {
                'geometry': geometrias_por_categoria[valor],
                'Valor': valor,
                'uf_codigo': f"s{valor}" if valor > 0 else "si",
                'NM_UF': linha_base.get('NM_UF', 'Maranhão'),
                'SIGLA_UF': linha_base.get('SIGLA_UF', 'MA'),
                'tem_seca': valor > 0
            }
            
            for col in ['CD_UF', 'NM_REGIAO', 'AREA_KM2']:
                if col in linha_base: 
                    nova_feature[col] = linha_base[col]
            
            features_finais.append(nova_feature)
    
    if features_finais:
        gdf_final = gpd.GeoDataFrame(features_finais, crs=gdf_maranhao.crs)
        gdf_final.to_file(caminho_saida, driver='ESRI Shapefile')
        print(f"\nShapefile com manchas isoladas salvo em: {caminho_saida}")
        print(f"Features no arquivo final: {len(gdf_final)}")
        return gdf_final
    else:
        print("Nenhuma feature válida processada!")
        return None

def calcular_area_elipsoidal(geometry):
    if geometry.is_empty: 
        return 0.0
    geod = Geod(ellps="GRS80")
    area, _ = geod.geometry_area_perimeter(geometry)
    return abs(area) / 1000000

def processar_municipios_seca(gdf_manchas_isoladas, gdf_municipios, gdf_maranhao, area_total_maranhao_km2):
    print("\n=== PROCESSANDO SECA POR MUNICÍPIO ===")
    
    crs_projetado = 'EPSG:5880'
    crs_geografico = 'EPSG:4674'
    
    try:
        gdf_seca_proj = gdf_manchas_isoladas.to_crs(crs_projetado)
        gdf_maranhao_proj = gdf_maranhao.to_crs(crs_projetado)
        gdf_municipios_proj = gdf_municipios.to_crs(crs_projetado)
        gdf_municipios_geo = gdf_municipios.to_crs(crs_geografico)
    except:
        try:
            crs_projetado = 'EPSG:31983'
            gdf_seca_proj = gdf_manchas_isoladas.to_crs(crs_projetado)
            gdf_maranhao_proj = gdf_maranhao.to_crs(crs_projetado)
            gdf_municipios_proj = gdf_municipios.to_crs(crs_projetado)
            gdf_municipios_geo = gdf_municipios.to_crs(crs_geografico)
        except:
            crs_projetado = 'EPSG:3857'
            gdf_seca_proj = gdf_manchas_isoladas.to_crs(crs_projetado)
            gdf_maranhao_proj = gdf_maranhao.to_crs(crs_projetado)
            gdf_municipios_proj = gdf_municipios.to_crs(crs_projetado)
            gdf_municipios_geo = gdf_municipios.to_crs(crs_geografico)
    
    print(f"CRS projetado usado para municípios: {crs_projetado}")
    gdf_municipios_geo['area_municipio_km2'] = gdf_municipios_geo.geometry.apply(calcular_area_elipsoidal)
    gdf_seca_maranhao_proj = gpd.overlay(gdf_seca_proj, gdf_maranhao_proj, how='intersection', keep_geom_type=False)
    
    municipios_com_seca = []
    
    for idx, municipio in gdf_municipios_proj.iterrows():
        municipio_nome = municipio.get('NM_MUN', f"Município_{idx}")
        area_municipio = gdf_municipios_geo.loc[idx, 'area_municipio_km2']
        
        secas_no_municipio = gdf_seca_maranhao_proj[gdf_seca_maranhao_proj.intersects(municipio.geometry)].copy()
        
        seca_por_nivel = {}
        area_total_seca_municipio = 0
        area_sem_seca_municipio = 0

        for _, seca in secas_no_municipio.iterrows():
            try:
                interseccao = seca.geometry.intersection(municipio.geometry)
                if not interseccao.is_empty:
                    interseccao_geo = gpd.GeoSeries([interseccao], crs=crs_projetado).to_crs(crs_geografico).iloc[0]
                    area_interseccao_km2 = calcular_area_elipsoidal(interseccao_geo)
                    valor_seca = seca.get('Valor', 0)
                    
                    if valor_seca > 0:
                        if valor_seca not in seca_por_nivel: 
                            seca_por_nivel[valor_seca] = 0
                        seca_por_nivel[valor_seca] += area_interseccao_km2
                        area_total_seca_municipio += area_interseccao_km2
                    else: 
                        area_sem_seca_municipio += area_interseccao_km2
            except Exception as e:
                print(f"Erro em {municipio_nome}: {e}")
                continue

        area_sem_classificacao = max(0, area_municipio - area_total_seca_municipio - area_sem_seca_municipio)
        tem_seca_real = area_total_seca_municipio > 0
        
        if tem_seca_real:
            seca_por_nivel_com_perc = {}
            for nivel, area in seca_por_nivel.items():
                seca_por_nivel_com_perc[nivel] = {
                    'area_km2': area,
                    'perc_municipio': (area / area_municipio) * 100,
                    'perc_area_seca': (area / area_total_seca_municipio) * 100
                }

            municipios_com_seca.append({
                'NM_MUN': municipio_nome,
                'area_municipio_km2': area_municipio,
                'area_com_seca_km2': area_total_seca_municipio,
                'area_sem_seca_km2': area_sem_seca_municipio,
                'area_sem_classificacao_km2': area_sem_classificacao,
                'perc_area_com_seca': (area_total_seca_municipio / area_municipio) * 100,
                'perc_area_sem_seca': (area_sem_seca_municipio / area_municipio) * 100,
                'perc_area_sem_classificacao': (area_sem_classificacao / area_municipio) * 100,
                'nivel_seca_predominante': max(seca_por_nivel.items(), key=lambda x: x[1])[0] if seca_por_nivel else 0,
                'tem_seca': True,
                'seca_por_nivel': seca_por_nivel_com_perc,
                'num_tipos_seca': len(seca_por_nivel_com_perc)
            })
        else:
            municipios_com_seca.append({
                'NM_MUN': municipio_nome,
                'area_municipio_km2': area_municipio,
                'area_com_seca_km2': 0,
                'area_sem_seca_km2': area_sem_seca_municipio,
                'area_sem_classificacao_km2': area_sem_classificacao,
                'perc_area_com_seca': 0,
                'perc_area_sem_seca': (area_sem_seca_municipio / area_municipio) * 100,
                'perc_area_sem_classificacao': (area_sem_classificacao / area_municipio) * 100,
                'nivel_seca_predominante': 0,
                'tem_seca': False,
                'seca_por_nivel': {},
                'num_tipos_seca': 0
            })
    
    df_municipios_seca = pd.DataFrame(municipios_com_seca)
    gdf_municipios_com_seca = gdf_municipios.merge(df_municipios_seca, on=['NM_MUN'], how='left')
    gdf_municipios_com_seca = gdf_municipios_com_seca.drop(['CD_MUN', 'Dados'], axis=1, errors='ignore')
    
    total_municipios_com_seca = df_municipios_seca['tem_seca'].sum()
    area_total_com_seca_estado = df_municipios_seca['area_com_seca_km2'].sum()
    
    print(f"Municípios com seca: {total_municipios_com_seca} de {len(gdf_municipios_com_seca)}")
    print(f"Área total com seca nos municípios: {area_total_com_seca_estado:,.2f} km²")
    
    return gdf_municipios_com_seca

def processar_impactos(caminho_impactos_tipo, caminho_impactos, gdf_maranhao, pasta_base):
    print("\n=== PROCESSANDO CAMADAS DE IMPACTOS ===")
    
    resultados_impactos = {}
    
    try:
        gdf_impactos_tipo = gpd.read_file(caminho_impactos_tipo)
        gdf_impactos_tipo_recortado = gpd.overlay(gdf_impactos_tipo.to_crs(gdf_maranhao.crs), gdf_maranhao, how='intersection', keep_geom_type=False)
        caminho_impactos_tipo_saida = os.path.join(pasta_base, "impactos_tipo_recortado.geojson")
        gdf_impactos_tipo_recortado.to_file(caminho_impactos_tipo_saida, driver='GeoJSON')
        resultados_impactos['impactos_tipo'] = {
            'caminho': caminho_impactos_tipo_saida,
            'features': len(gdf_impactos_tipo_recortado),
            'gdf': gdf_impactos_tipo_recortado
        }
        print(f"Impactos Tipo: {len(gdf_impactos_tipo_recortado)} features")
    except Exception as e:
        print(f"Erro em Impactos Tipo: {e}")
        resultados_impactos['impactos_tipo'] = None
    
    try:
        gdf_impactos = gpd.read_file(caminho_impactos)
        gdf_impactos_recortado = gpd.overlay(gdf_impactos.to_crs(gdf_maranhao.crs), gdf_maranhao, how='intersection', keep_geom_type=False)
        caminho_impactos_saida = os.path.join(pasta_base, "impactos_recortado.geojson")
        gdf_impactos_recortado.to_file(caminho_impactos_saida, driver='GeoJSON')
        resultados_impactos['impactos'] = {
            'caminho': caminho_impactos_saida,
            'features': len(gdf_impactos_recortado),
            'gdf': gdf_impactos_recortado
        }
        print(f"Impactos: {len(gdf_impactos_recortado)} features")
    except Exception as e:
        print(f"Erro em Impactos: {e}")
        resultados_impactos['impactos'] = None
    
    return resultados_impactos

def atualizar_json_periodos_disponiveis(periodo_analise, pasta_base="docs/monitor_seca"):
    print("\n=== ATUALIZANDO JSON DE PERÍODOS DISPONÍVEIS ===")
    
    caminho_json = os.path.join(pasta_base, "periodos_disponiveis.json")
    
    if os.path.exists(caminho_json):
        try:
            with open(caminho_json, 'r', encoding='utf-8') as f:
                dados_existentes = json.load(f)
            
            periodos = dados_existentes.get('periodos_disponiveis', [])
            
            if periodo_analise not in periodos:
                periodos.append(periodo_analise)
                periodos.sort()
                print(f"Período {periodo_analise} adicionado ao JSON existente")
            else: 
                print(f"Período {periodo_analise} já existe no JSON")
                
        except Exception as e:
            print(f"Erro ao carregar JSON existente: {e}")
            periodos = [periodo_analise]
    else:
        periodos = [periodo_analise]
        print(f"Novo JSON criado com período: {periodo_analise}")
    
    json_data = {"periodos_disponiveis": periodos}
    
    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=2, ensure_ascii=False)
    
    print(f"JSON de períodos disponíveis salvo em: {caminho_json}")
    print(f"Total de períodos: {len(periodos)}")
    print(f"Períodos disponíveis: {periodos}")
    
    return json_data

def processar_seca_completa(mes, ano):
    meses = ['janeiro', 'fevereiro', 'marco', 'abril', 'maio', 'junho','julho', 'agosto', 'setembro', 'outubro', 'novembro', 'dezembro']

    nome_mes = meses[mes-1]
    mes_str = f"{mes:02d}"
    ano_short = str(ano)[-2:]
    periodo_analise = f"{mes_str}{ano}"

    Pasta = 'docs/monitor_seca'
    pasta_base = os.path.join(Pasta, f"{mes_str}{ano}")

    caminho_shp_seca_original = os.path.join(pasta_base, f"{nome_mes}{ano}", f"{nome_mes}{ano_short}.shp")
    caminho_shp_impactos_tipo = os.path.join(pasta_base, f"{nome_mes}{ano}", f"{ano}{mes_str}_MS_IMPACTOS_TIPO.shp")
    caminho_shp_impactos = os.path.join(pasta_base, f"{nome_mes}{ano}", f"{ano}{mes_str}_MS_IMPACTOS.shp")
    caminho_shp_maranhao = "docs/Shapefile/MA_Municipios_2022_limite/Limite_MA_2022.shp"
    caminho_shp_maranhao_mun = "docs/Shapefile/MA_Municipios_2022/MA_Municipios_2022.shp"
    
    caminho_shp_seca_isolado = os.path.join(pasta_base, f"{nome_mes}{ano_short}_isolado.shp")
    caminho_geojson_principal = os.path.join(pasta_base, "seca_atributos.geojson")
    caminho_geojson_municipios = os.path.join(pasta_base, "seca_por_municipio.geojson")
    
    print(f"\n=== INICIANDO PROCESSAMENTO PARA {mes_str}/{ano} ===")
    
    os.makedirs(os.path.dirname(caminho_shp_seca_isolado), exist_ok=True)
    os.makedirs(pasta_base, exist_ok=True)
    
    gdf_manchas_isoladas = isolar_manchas_seca_por_categoria(
        caminho_shp_seca_original, 
        caminho_shp_maranhao, 
        caminho_shp_seca_isolado
    )
    
    if gdf_manchas_isoladas is None:
        print(f"Erro: Não foi possível isolar as manchas de seca para {mes_str}/{ano}.")
        return None
    
    try:
        gdf_maranhao = gpd.read_file(caminho_shp_maranhao)
        gdf_municipios = gpd.read_file(caminho_shp_maranhao_mun)
        
        crs_geografico = 'EPSG:4674'
        
        gdf_manchas_geo = gdf_manchas_isoladas.to_crs(crs_geografico)
        gdf_maranhao_geo = gdf_maranhao.to_crs(crs_geografico)
        
        geometry_union = gdf_maranhao_geo.union_all()
        geod = Geod(ellps="GRS80")
        area, _ = geod.geometry_area_perimeter(geometry_union)
        area_total_maranhao_km2 = abs(area) / 1000000
        
        print(f"\nÁrea total do Maranhão: {area_total_maranhao_km2:,.2f} km²")
        
        gdf_manchas_geo['area_seca_km2'] = gdf_manchas_geo.geometry.apply(calcular_area_elipsoidal)
        
        gdf_seca_afetada = gdf_manchas_geo[gdf_manchas_geo['Valor'] > 0]
        area_total_seca = gdf_seca_afetada['area_seca_km2'].sum()
        area_sem_seca = gdf_manchas_geo[gdf_manchas_geo['Valor'] == 0]['area_seca_km2'].sum() if len(gdf_manchas_geo[gdf_manchas_geo['Valor'] == 0]) > 0 else 0
        total_manchas_seca = len(gdf_seca_afetada)
        perc_total_afetado = (area_total_seca / area_total_maranhao_km2) * 100
        
        gdf_manchas_geo['area_maranhao_total_km2'] = area_total_maranhao_km2
        gdf_manchas_geo['perc_area_afetada'] = (gdf_manchas_geo['area_seca_km2'] / area_total_maranhao_km2) * 100
        gdf_manchas_geo['total_manchas_seca'] = total_manchas_seca
        gdf_manchas_geo['area_total_seca_km2'] = area_total_seca
        gdf_manchas_geo['perc_total_afetado'] = perc_total_afetado
        gdf_manchas_geo['area_sem_seca_km2'] = area_sem_seca
        gdf_manchas_geo['tem_seca'] = gdf_manchas_geo['Valor'] > 0
        
        gdf_manchas_geo.loc[gdf_manchas_geo['Valor'] == 0, 'perc_area_afetada'] = 0
        
        gdf_manchas_geo = gdf_manchas_geo.sort_values('Valor', ascending=True)
        
        if 'uf_codigo' not in gdf_manchas_geo.columns:
            gdf_manchas_geo['uf_codigo'] = gdf_manchas_geo['Valor'].apply(lambda x: f"s{x}" if x > 0 else "si")
        
        colunas_finais = [
            'uf_codigo', 'Valor', 'NM_UF', 'SIGLA_UF', 'area_seca_km2', 
            'area_maranhao_total_km2', 'perc_area_afetada', 'total_manchas_seca',
            'area_total_seca_km2', 'perc_total_afetado', 'area_sem_seca_km2', 'tem_seca'
        ]
        
        colunas_existentes = [col for col in colunas_finais if col in gdf_manchas_geo.columns]
        gdf_resultado_final = gdf_manchas_geo[colunas_existentes + ['geometry']]
        
        gdf_resultado_final.to_file(caminho_geojson_principal, driver='GeoJSON')
        
        gdf_municipios_seca = processar_municipios_seca(
            gdf_manchas_isoladas, 
            gdf_municipios, 
            gdf_maranhao, 
            area_total_maranhao_km2
        )
        
        gdf_export = gdf_municipios_seca.copy()
        gdf_export['seca_por_nivel_json'] = gdf_export['seca_por_nivel'].apply(json.dumps)
        gdf_export.to_file(caminho_geojson_municipios, driver='GeoJSON')
        
        resultados_impactos = processar_impactos(
            caminho_shp_impactos_tipo,
            caminho_shp_impactos,
            gdf_maranhao,
            pasta_base
        )
        
        json_periodos = atualizar_json_periodos_disponiveis(periodo_analise, Pasta)
        
        print("\n" + "="*50)
        print(f"RELATÓRIO FINAL DO PROCESSAMENTO - {mes_str}/{ano}")
        print("="*50)
        
        print(f"\n📅 PERÍODO ANALISADO: {periodo_analise}")
        
        print("\n📁 ARQUIVOS GERADOS:")
        print(f"✅ GeoJSON Principal: {caminho_geojson_principal}")
        print(f"✅ GeoJSON Municípios: {caminho_geojson_municipios}")
        print(f"✅ Shapefile Manchas Isoladas: {caminho_shp_seca_isolado}")
        print(f"✅ JSON Períodos Disponíveis: {os.path.join(Pasta, 'periodos_disponiveis.json')}")
        
        if resultados_impactos.get('impactos_tipo'):
            print(f"✅ Impactos Tipo: {resultados_impactos['impactos_tipo']['caminho']}")
        if resultados_impactos.get('impactos'):
            print(f"✅ Impactos: {resultados_impactos['impactos']['caminho']}")
        
        print("\n📊 ESTATÍSTICAS:")
        print(f"Área total do Maranhão: {area_total_maranhao_km2:,.2f} km²")
        print(f"Área com seca: {area_total_seca:,.2f} km² ({perc_total_afetado:.2f}%)")
        print(f"Área sem seca: {area_sem_seca:,.2f} km²")
        print(f"Categorias de seca processadas: {len(gdf_resultado_final)}")
        
        print("\n🏙️ MUNICÍPIOS:")
        total_mun_seca = gdf_municipios_seca['tem_seca'].sum()
        print(f"Municípios com seca: {total_mun_seca} de {len(gdf_municipios_seca)}")
        
        print("\n🔍 DETALHES POR CATEGORIA:")
        for idx, row in gdf_resultado_final.iterrows():
            categoria = "Si (Sem seca)" if row['Valor'] == 0 else f"S{row['Valor']-1}"
            print(f"  {categoria} (Valor {row['Valor']}): {row['area_seca_km2']:,.2f} km² ({row['perc_area_afetada']:.2f}%)")
        
        print("\n📅 PERÍODOS DISPONÍVEIS:")
        print(f"Total de períodos: {len(json_periodos['periodos_disponiveis'])}")
        print(f"Períodos: {json_periodos['periodos_disponiveis']}")
        
        print(f"\n✅ PROCESSAMENTO PARA {mes_str}/{ano} CONCLUÍDO COM SUCESSO!")
        
        return {
            'geojson_principal': gdf_resultado_final,
            'geojson_municipios': gdf_municipios_seca,
            'impactos': resultados_impactos,
            'periodos_disponiveis': json_periodos,
            'estatisticas': {
                'area_total_maranhao': area_total_maranhao_km2,
                'area_total_seca': area_total_seca,
                'perc_total_afetado': perc_total_afetado,
                'municipios_com_seca': total_mun_seca,
                'total_municipios': len(gdf_municipios_seca)
            }
        }
        
    except Exception as e:
        print(f"❌ Erro no processamento principal para {mes_str}/{ano}: {e}")
        import traceback
        traceback.print_exc()
        return None

def processar_todas_pastas_faltantes():
    """
    Função principal que encontra pastas faltantes e processa cada uma
    """
    print("🚀 INICIANDO PROCESSAMENTO DE PASTAS FALTANTES")
    print("="*60)
    
    # Encontrar pastas que não possuem o arquivo seca_atributos.geojson
    pastas_faltantes = encontrar_pastas_faltantes()
    
    if not pastas_faltantes:
        print("🎉 Nenhuma pasta faltante encontrada!")
        return
    
    print(f"\n🔄 Iniciando processamento de {len(pastas_faltantes)} pasta(s) faltante(s)...")
    
    resultados = []
    
    for falta in pastas_faltantes:
        mes = falta['mes']
        ano = falta['ano']
        pasta_nome = falta['pasta']
        
        print(f"\n{'='*50}")
        print(f"PROCESSANDO: {pasta_nome} (Mês: {mes}, Ano: {ano})")
        print(f"{'='*50}")
        
        # Executar o processamento completo para esta pasta
        resultado = processar_seca_completa(mes, ano)
        
        if resultado:
            resultados.append({
                'pasta': pasta_nome,
                'mes': mes,
                'ano': ano,
                'status': 'sucesso',
                'resultado': resultado
            })
            print(f"✅ {pasta_nome} processada com sucesso!")
        else:
            resultados.append({
                'pasta': pasta_nome,
                'mes': mes,
                'ano': ano,
                'status': 'erro'
            })
            print(f"❌ Erro ao processar {pasta_nome}")
    
    # Relatório final
    print(f"\n{'='*60}")
    print("📊 RELATÓRIO FINAL DO PROCESSAMENTO")
    print(f"{'='*60}")
    
    sucessos = sum(1 for r in resultados if r['status'] == 'sucesso')
    erros = sum(1 for r in resultados if r['status'] == 'erro')
    
    print(f"✅ Pastas processadas com sucesso: {sucessos}")
    print(f"❌ Pastas com erro: {erros}")
    print(f"📁 Total de pastas processadas: {len(resultados)}")
    
    if sucessos > 0:
        print(f"\n🎉 Processamento concluído! {sucessos} pasta(s) foram processadas com sucesso.")
    else:
        print("\n⚠️ Nenhuma pasta foi processada.")
    
    return resultados

# ============================================================================
# SISTEMA DE DADOS HIDROLÓGICOS E NOTÍCIAS (CÓDIGO PRINCIPAL)
# ============================================================================

class ColetorDadosHidrologicos:
    def __init__(self):
        self.caminho_db = CONFIG['caminho_db']
        self.arquivo_estacoes = CONFIG['arquivo_estacoes']
        self.df_cadastro = None
        self.estacoes_dados = None
        self.lista_estacoes = []
        self.progress_lock = threading.Lock()
        self.ultimas_datas_cache = {}
        
        # Criar pasta database se não existir
        os.makedirs('database', exist_ok=True)
        
        # Carregar dados iniciais
        self.carregar_estacoes()
        self.criar_tabelas_se_nao_existirem()
        self.atualizar_cadastro_estacoes()
        self.carregar_ultimas_datas()

    def carregar_estacoes(self):
        """Carrega o arquivo de estações"""
        try:
            self.df_cadastro = pd.read_excel(self.arquivo_estacoes)
            logger.info(f"✅ Coletor - Arquivo de estações carregado: {len(self.df_cadastro)} estações")
            
            self.estacoes_dados = self.df_cadastro[self.df_cadastro['Codigo_Origin'].notna()].copy()
            self.estacoes_dados['Codigo_Origin'] = self.estacoes_dados['Codigo_Origin'].astype(str).str.replace('.0', '', regex=False)
            self.lista_estacoes = self.estacoes_dados['Codigo_Origin'].tolist()
            
            logger.info(f"🏞️  Coletor - Total de estações para processar: {len(self.lista_estacoes)}")
            
        except Exception as e:
            logger.error(f"❌ Coletor - Erro ao carregar arquivo de estações: {e}")
            raise

    def criar_tabelas_se_nao_existirem(self):
        """Cria as tabelas necessárias se não existirem"""
        conn = sqlite3.connect(self.caminho_db)
        cursor = conn.cursor()
        
        try:
            cursor.execute('''
            CREATE TABLE IF NOT EXISTS dados_diarios (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                codigo_estacao TEXT NOT NULL,
                data_completa TEXT NOT NULL,
                nivel REAL,
                vazao REAL,
                precipitacao REAL,
                data_insercao TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                UNIQUE(codigo_estacao, data_completa)
            )
            ''')
            
            cursor.execute('''
            CREATE TABLE IF NOT EXISTS cadastro_estacoes (
                codigo_origin TEXT PRIMARY KEY,
                estacao TEXT NOT NULL,
                municipio TEXT,
                rio TEXT,
                bacia TEXT,
                latitude REAL,
                longitude REAL,
                r_cota TEXT,
                s_emergencia REAL,
                s_alerta REAL,
                s_atencao REAL,
                c_normal REAL,
                c_atencao REAL,
                c_alerta REAL,
                c_emergencia REAL,
                data_atualizacao TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
            ''')
            
            cursor.execute('''
            CREATE TABLE IF NOT EXISTS log_execucoes (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                data_execucao TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                estacoes_processadas INTEGER,
                registros_coletados INTEGER,
                registros_inseridos INTEGER,
                tempo_execucao REAL,
                status TEXT
            )
            ''')
            
            cursor.execute('CREATE INDEX IF NOT EXISTS idx_codigo_data ON dados_diarios(codigo_estacao, data_completa)')
            cursor.execute('CREATE INDEX IF NOT EXISTS idx_data ON dados_diarios(data_completa)')
            
            conn.commit()
            logger.info("✅ Coletor - Tabelas e índices verificados/criados")
            
        except Exception as e:
            logger.error(f"❌ Coletor - Erro ao criar tabelas: {e}")
            raise
        finally:
            conn.close()

    def atualizar_cadastro_estacoes(self):
        """Atualiza o cadastro de estações no banco"""
        conn = sqlite3.connect(self.caminho_db)
        cursor = conn.cursor()
        
        try:
            if 'Codigo_Origin' in self.df_cadastro.columns and 'Estacao' in self.df_cadastro.columns:
                estacoes_para_inserir = []
                
                for _, row in self.df_cadastro.iterrows():
                    if pd.notna(row['Codigo_Origin']):
                        codigo = str(row['Codigo_Origin']).replace('.0', '')
                        
                        estacao_data = (
                            codigo,
                            row['Estacao'] if pd.notna(row['Estacao']) else f"Estacao_{codigo}",
                            row['Municipio'] if 'Municipio' in self.df_cadastro.columns and pd.notna(row['Municipio']) else None,
                            row['Rio'] if 'Rio' in self.df_cadastro.columns and pd.notna(row['Rio']) else None,
                            row['Bacia'] if 'Bacia' in self.df_cadastro.columns and pd.notna(row['Bacia']) else None,
                            row['Latitude'] if 'Latitude' in self.df_cadastro.columns and pd.notna(row['Latitude']) else None,
                            row['Longitude'] if 'Longitude' in self.df_cadastro.columns and pd.notna(row['Longitude']) else None,
                            row['r_cota'] if 'r_cota' in self.df_cadastro.columns and pd.notna(row['r_cota']) else None,
                            row['s_emergencia'] if 's_emergencia' in self.df_cadastro.columns and pd.notna(row['s_emergencia']) else None,
                            row['s_alerta'] if 's_alerta' in self.df_cadastro.columns and pd.notna(row['s_alerta']) else None,
                            row['s_atencao'] if 's_atencao' in self.df_cadastro.columns and pd.notna(row['s_atencao']) else None,
                            row['c_normal'] if 'c_normal' in self.df_cadastro.columns and pd.notna(row['c_normal']) else None,
                            row['c_atencao'] if 'c_atencao' in self.df_cadastro.columns and pd.notna(row['c_atencao']) else None,
                            row['c_alerta'] if 'c_alerta' in self.df_cadastro.columns and pd.notna(row['c_alerta']) else None,
                            row['c_emergencia'] if 'c_emergencia' in self.df_cadastro.columns and pd.notna(row['c_emergencia']) else None,
                        )
                        estacoes_para_inserir.append(estacao_data)
                
                cursor.executemany('''
                INSERT OR REPLACE INTO cadastro_estacoes 
                (codigo_origin, estacao, municipio, rio, bacia, latitude, longitude, 
                r_cota, s_emergencia, s_alerta, s_atencao, c_normal, c_atencao, c_alerta, c_emergencia)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ''', estacoes_para_inserir)
                
                conn.commit()
                logger.info(f"✅ Coletor - Cadastro atualizado: {len(estacoes_para_inserir)} estações")
                    
        except Exception as e:
            logger.error(f"⚠️  Coletor - Aviso ao atualizar cadastro: {e}")
        finally:
            conn.close()

    def carregar_ultimas_datas(self):
        """Carrega as últimas datas de cada estação do banco"""
        conn = sqlite3.connect(self.caminho_db)
        cursor = conn.cursor()
        
        try:
            cursor.execute('''
            SELECT codigo_estacao, MAX(data_completa) as ultima_data 
            FROM dados_diarios 
            GROUP BY codigo_estacao
            ''')
            
            resultados = cursor.fetchall()
            
            for codigo, ultima_data in resultados:
                if ultima_data:
                    try:
                        data_obj = datetime.strptime(ultima_data, '%Y-%m-%d %H:%M:%S')
                        data_inicio = data_obj.strftime('%Y-%m-%d')
                        self.ultimas_datas_cache[codigo] = data_inicio
                    except:
                        self.ultimas_datas_cache[codigo] = ultima_data.split(' ')[0]
            
            logger.info(f"📅 Coletor - Últimas datas carregadas para {len(self.ultimas_datas_cache)} estações")
            
        except Exception as e:
            logger.error(f"❌ Coletor - Erro ao carregar últimas datas: {e}")
        finally:
            conn.close()

    def obter_data_inicio_estacao(self, codigo_estacao: str) -> str:
        """Obtém a data de início para uma estação específica"""
        data_fim = datetime.now().strftime('%Y-%m-%d')
        data_fim_obj = datetime.strptime(data_fim, '%Y-%m-%d')
        
        if codigo_estacao in self.ultimas_datas_cache:
            data_cache = self.ultimas_datas_cache[codigo_estacao]
            try:
                data_obj = datetime.strptime(data_cache, '%Y-%m-%d')
                data_inicio = (data_obj + timedelta(days=1)).strftime('%Y-%m-%d')
                
                if datetime.strptime(data_inicio, '%Y-%m-%d') > data_fim_obj:
                    return data_fim
                    
                return data_inicio
            except Exception as e:
                logger.warning(f"⚠️  Coletor - Erro ao processar data do cache para {codigo_estacao}: {e}")
        
        data_inicio = (data_fim_obj - timedelta(days=CONFIG['dias_retrocesso_fallback'])).strftime('%Y-%m-%d')
        logger.info(f"🔄 Coletor - Usando fallback para {codigo_estacao}: {data_inicio} a {data_fim}")
        return data_inicio

    def processar_estacao(self, codigo_estacao: str) -> Tuple[str, int, int]:
        """Processa uma estação individual"""
        conn_thread = None
        try:
            conn_thread = sqlite3.connect(self.caminho_db)
            cursor_thread = conn_thread.cursor()
            
            nome_estacao = "Desconhecida"
            try:
                nome_estacao = self.estacoes_dados[self.estacoes_dados['Codigo_Origin'] == codigo_estacao]['Estacao'].iloc[0]
            except:
                nome_estacao = f"Estacao_{codigo_estacao}"
            
            data_inicio = self.obter_data_inicio_estacao(codigo_estacao)
            data_fim = datetime.now().strftime('%Y-%m-%d') 

            logger.info(f"🔍 Coletor - Processando {nome_estacao} ({codigo_estacao}) - Período: {data_inicio} a {data_fim}")

            data_inicio_obj = datetime.strptime(data_inicio, '%Y-%m-%d')
            data_fim_obj = datetime.strptime(data_fim, '%Y-%m-%d')
            
            if data_inicio_obj > data_fim_obj:
                logger.info("   ⏩ Coletor - Estação já atualizada, pulando...")
                return codigo_estacao, 0, 0

            url = f"https://telemetriaws1.ana.gov.br/ServiceANA.asmx/DadosHidrometeorologicos?codEstacao={codigo_estacao}&dataInicio={data_inicio}&dataFim={data_fim}"

            with requests.Session() as session:
                response = session.get(url, timeout=CONFIG['timeout_requisicao'])
                response.raise_for_status()

            soup = BeautifulSoup(response.content, 'xml')
            
            if soup.find('error') or soup.find('fault'):
                logger.warning(f"   ⚠️  Coletor - Erro no XML para {codigo_estacao}")
                return codigo_estacao, 0, 0
                
            dados = soup.find_all('DadosHidrometereologicos')

            if not dados:
                logger.warning(f"   ⚠️  Coletor - Nenhum dado encontrado para {codigo_estacao} no período {data_inicio} a {data_fim}")
                return codigo_estacao, 0, 0

            dados_processados = []
            registros_validos = 0

            for dado in dados:
                try:
                    data_hora = dado.find('DataHora')
                    nivel = dado.find('Nivel')
                    vazao = dado.find('Vazao')
                    chuva = dado.find('Chuva')

                    if not data_hora or not data_hora.text.strip():
                        continue

                    data_hora_text = data_hora.text.strip()
                    nivel_text = nivel.text if nivel and nivel.text else ''
                    vazao_text = vazao.text if vazao and vazao.text else ''
                    chuva_text = chuva.text if chuva and chuva.text else ''

                    nivel_float = self._processar_valor(nivel_text, 'nivel')
                    vazao_float = self._processar_valor(vazao_text, 'vazao')
                    chuva_float = self._processar_valor(chuva_text, 'chuva')

                    if nivel_float is not None or vazao_float is not None or chuva_float is not None:
                        registros_validos += 1

                    dados_estacao = (codigo_estacao, data_hora_text, nivel_float, vazao_float, chuva_float)
                    dados_processados.append(dados_estacao)

                except Exception as e:
                    logger.warning(f"   ⚠️  Coletor - Erro ao processar registro para {codigo_estacao}: {e}")
                    continue

            if not dados_processados:
                logger.warning(f"   ⚠️  Coletor - Nenhum dado válido para {codigo_estacao}")
                return codigo_estacao, 0, 0

            logger.info(f"   📋 Coletor - {len(dados_processados)} registros processados, {registros_validos} com dados válidos")

            inseridos = 0
            try:
                cursor_thread.executemany('''
                INSERT OR IGNORE INTO dados_diarios
                (codigo_estacao, data_completa, nivel, vazao, precipitacao)
                VALUES (?, ?, ?, ?, ?)
                ''', dados_processados)

                conn_thread.commit()
                inseridos = cursor_thread.rowcount
                logger.info(f"   ✅ Coletor - {nome_estacao}: {inseridos}/{len(dados_processados)} registros inseridos")

                if inseridos > 0:
                    self._atualizar_cache_ultima_data(codigo_estacao, dados_processados)

            except Exception as e:
                logger.error(f"   ❌ Coletor - Erro ao inserir {codigo_estacao}: {e}")
                conn_thread.rollback()
                return codigo_estacao, len(dados_processados), 0

            return codigo_estacao, len(dados_processados), inseridos

        except requests.exceptions.RequestException as e:
            logger.error(f"   ❌ Coletor - Erro de requisição {codigo_estacao}: {e}")
            return codigo_estacao, 0, 0
        except Exception as e:
            logger.error(f"   ❌ Coletor - Erro geral {codigo_estacao}: {e}")
            return codigo_estacao, 0, 0
        finally:
            if conn_thread:
                conn_thread.close()

    def _atualizar_cache_ultima_data(self, codigo_estacao: str, dados_processados: list):
        """Atualiza o cache da última data para uma estação"""
        try:
            datas = []
            for dado in dados_processados:
                if dado[1]:
                    try:
                        data_obj = datetime.strptime(dado[1], '%Y-%m-%d %H:%M:%S')
                        datas.append(data_obj)
                    except:
                        continue
            
            if datas:
                ultima_data = max(datas)
                self.ultimas_datas_cache[codigo_estacao] = ultima_data.strftime('%Y-%m-%d')
        except Exception as e:
            logger.warning(f"⚠️  Coletor - Não foi possível atualizar cache para {codigo_estacao}: {e}")

    def _processar_valor(self, valor_text: str, tipo: str):
        """Processa e valida valores numéricos"""
        if not valor_text or not valor_text.strip() or valor_text == '0':
            return None
        
        try:
            valor = float(valor_text.replace(',', '.'))
            
            if tipo == 'nivel' and 0 < valor <= 1000:
                return valor
            elif tipo == 'vazao' and 0 < valor <= 10000:
                return valor
            elif tipo == 'chuva' and valor >= 0:
                return valor
                
        except (ValueError, TypeError):
            pass
            
        return None

    def executar_coleta(self):
        """Executa uma coleta completa - SEMPRE EXECUTA"""
        logger.info("🚀 COLETOR - INICIANDO COLETA AUTOMÁTICA...")
        inicio = time.time()

        self.carregar_ultimas_datas()
        self.atualizar_cadastro_estacoes()

        total_dados_coletados = 0
        total_dados_inseridos = 0

        try:
            with concurrent.futures.ThreadPoolExecutor(max_workers=CONFIG['max_workers']) as executor:
                resultados = list(executor.map(self.processar_estacao, self.lista_estacoes))
                
                for i, (codigo_estacao, coletados, inseridos) in enumerate(resultados, 1):
                    total_dados_coletados += coletados
                    total_dados_inseridos += inseridos
                    
                    with self.progress_lock:
                        percent_estacoes = (i / len(self.lista_estacoes)) * 100
                        bar_estacoes = '█' * int(percent_estacoes/5) + '_' * (20 - int(percent_estacoes/5))
                        sys.stdout.write(f'\r📊 Coletor - Estações: [{bar_estacoes}] {percent_estacoes:.1f}% | 📈 Dados: {total_dados_coletados} coletados, {total_dados_inseridos} inseridos')
                        sys.stdout.flush()

            sys.stdout.write('\n')
            tempo_total = time.time() - inicio

            self._salvar_log_execucao(len(self.lista_estacoes), total_dados_coletados, 
                                     total_dados_inseridos, tempo_total, "SUCESSO")

            self._exibir_estatisticas(tempo_total, total_dados_coletados, total_dados_inseridos)
            return True

        except Exception as e:
            logger.error(f"❌ Coletor - Erro durante la coleta: {e}")
            self._salvar_log_execucao(0, 0, 0, 0, "ERRO")
            return False

    def _salvar_log_execucao(self, estacoes: int, coletados: int, inseridos: int, tempo: float, status: str):
        """Salva log da execução no banco"""
        conn = sqlite3.connect(self.caminho_db)
        cursor = conn.cursor()
        
        try:
            cursor.execute('''
            INSERT INTO log_execucoes 
            (estacoes_processadas, registros_coletados, registros_inseridos, tempo_execucao, status)
            VALUES (?, ?, ?, ?, ?)
            ''', (estacoes, coletados, inseridos, tempo, status))
            
            conn.commit()
            logger.info(f"📝 Coletor - Log de execução salvo: {status}")
        except Exception as e:
            logger.error(f"❌ Coletor - Erro ao salvar log: {e}")
        finally:
            conn.close()

    def _exibir_estatisticas(self, tempo_total: float, coletados: int, inseridos: int):
        """Exibe estatísticas finais"""
        logger.info("\n" + "=" * 60)
        logger.info("🎉 COLETOR - COLETA CONCLUÍDA!")
        logger.info(f"📈 Estações processadas: {len(self.lista_estacoes)}")
        logger.info(f"📊 Registros coletados: {coletados}")
        logger.info(f"💾 Registros inseridos: {inseridos}")
        
        if coletados > 0:
            taxa_sucesso = (inseridos / coletados) * 100
            logger.info(f"📈 Taxa de sucesso: {taxa_sucesso:.1f}%")
        
        logger.info(f"⏱️  Tempo total: {tempo_total:.2f} segundos")
        
        if tempo_total > 0:
            velocidade = inseridos / tempo_total
            logger.info(f"⚡ Velocidade: {velocidade:.1f} registros/segundo")

class SistemaNoticiasCompleto:
    def __init__(self):
        # Configurações do coletor de notícias
        self.config_dir = Path(CONFIG['config_dir'])
        self.noticias_path = self.config_dir / 'noticias.json'
        self.file_id_noticias = CONFIG['file_id_noticias']
        self.caminho_base_imagens = CONFIG['caminho_base_imagens']
        self.imagem_padrao = CONFIG['imagem_padrao']
        
        # Configurações do baixador de imagens
        self.pasta_destino_imagens = Path(self.caminho_base_imagens)
        self.session = requests.Session()
        self.file_id_imagens = CONFIG['file_id_imagens']
        
        # Configurações da planilha de controle
        self.sheet_url = CONFIG['sheet_url']
        
        # Headers para simular navegador
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        # Formatos de imagem suportados (em ordem de prioridade)
        self.formatos_suportados = ['.jpg', '.jpeg', '.png', '.webp', '.gif']
        
        # Criar diretórios se não existirem
        self.config_dir.mkdir(exist_ok=True)
        self.pasta_destino_imagens.mkdir(parents=True, exist_ok=True)
        
        logger.info(f"📁 Sistema Notícias - Diretório de configuração: {self.config_dir.absolute()}")
        logger.info(f"📁 Sistema Notícias - Diretório de imagens: {self.pasta_destino_imagens.absolute()}")

    # ========== MÉTODOS DA PLANILHA DE CONTROLE ==========

    def verificar_condicao_atualizacao(self):
        """Verifica na planilha se deve atualizar o sistema
        Retorna: True se deve atualizar, False caso contrário"""
        try:
            logger.info("📋 Sistema Notícias - VERIFICANDO CONDIÇÃO NA PLANILHA")
            
            cabecalho_b = self.obter_cabecalho_segunda_coluna(self.sheet_url)
            
            if cabecalho_b is None:
                logger.warning("⚠️  Sistema Notícias - Não foi possível obter o valor da planilha, mantendo sistema atual")
                return False
            
            logger.info(f"🎯 Sistema Notícias - VALOR ENCONTRADO: '{cabecalho_b}'")
            
            # Verificar se o valor é 'sim' (case insensitive)
            if cabecalho_b.lower().strip() == 'sim':
                logger.info("✅ Sistema Notícias - CONDIÇÃO ATENDIDA: Sistema deve ser atualizado!")
                return True
            else:
                logger.info("⏸️  Sistema Notícias - CONDIÇÃO NÃO ATENDIDA: Mantendo sistema atual")
                return False
                
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao verificar condição de atualização: {e}")
            return False

    def obter_cabecalho_segunda_coluna(self, sheet_url):
        """Retorna o valor do cabeçalho da segunda coluna da segunda guia"""
        try:
            # Extrair Sheet ID
            match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', sheet_url)
            if not match:
                logger.error("❌ Sistema Notícias - Não encontrou Sheet ID")
                return None
            
            sheet_id = match.group(1)
            logger.info(f"🔗 Sistema Notícias - Sheet ID: {sheet_id}")
            
            # Baixar planilha como XLSX
            xlsx_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
            response = requests.get(xlsx_url, timeout=30)
            response.raise_for_status()
            
            logger.info("✅ Sistema Notícias - Planilha baixada com sucesso")
            
            # Ler arquivo XLSX
            xlsx_file = io.BytesIO(response.content)
            todas_guias = pd.read_excel(xlsx_file, sheet_name=None)
            
            logger.info(f"📑 Sistema Notícias - Guias encontradas: {list(todas_guias.keys())}")
            
            # Pegar a segunda guia
            nomes_guias = list(todas_guias.keys())
            if len(nomes_guias) < 2:
                logger.error("❌ Sistema Notícias - Menos de 2 guias encontradas")
                return None
            
            segunda_guia_nome = nomes_guias[1]
            segunda_guia = todas_guias[segunda_guia_nome]
            
            logger.info(f"📊 Sistema Notícias - Segunda guia '{segunda_guia_nome}': {segunda_guia.shape}")
            logger.info(f"🔍 Sistema Notícias - Cabeçalhos da segunda guia: {list(segunda_guia.columns)}")
            
            # PEGAR O CABEÇALHO DA SEGUNDA COLUNA (coluna B)
            if segunda_guia.shape[1] > 1:
                cabecalho_segunda_coluna = segunda_guia.columns[1]
                logger.info(f"✅ Sistema Notícias - Cabeçalho da segunda coluna: '{cabecalho_segunda_coluna}'")
                return str(cabecalho_segunda_coluna) if pd.notna(cabecalho_segunda_coluna) else None
            else:
                logger.error("❌ Sistema Notícias - Segunda guia não tem segunda coluna")
                return None
                
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao obter cabeçalho: {e}")
            return None

    # ========== MÉTODOS DO BAIXADOR DE IMAGENS ==========

    def extrair_file_id(self, url):
        """Extrai o file ID da URL do Google Drive"""
        try:
            padroes = [
                r'[\/=]([a-zA-Z0-9_-]{25,})',
                r'folders\/([a-zA-Z0-9_-]+)',
                r'drive\.google\.com\/drive\/folders\/([a-zA-Z0-9_-]+)'
            ]
            
            for padrao in padroes:
                match = re.search(padrao, url)
                if match:
                    return match.group(1)
            
            return None
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao extrair file ID: {e}")
            return None

    def obter_info_arquivos_pasta(self, pasta_url):
        """Obtém informações dos arquivos (ID, nome, URL) de uma pasta pública"""
        try:
            file_id = self.extrair_file_id(pasta_url)
            if not file_id:
                logger.error("❌ Sistema Notícias - Não foi possível extrair o file ID da URL")
                return []
            
            logger.info(f"🔗 Sistema Notícias - File ID extraído: {file_id}")
            
            url = f"https://drive.google.com/drive/folders/{file_id}"
            
            logger.info(f"🌐 Sistema Notícias - Acessando pasta: {url}")
            response = self.session.get(url, headers=self.headers, timeout=30)
            response.raise_for_status()
            
            arquivos_info = self._extrair_info_arquivos_pagina(response.text)
            
            if not arquivos_info:
                logger.warning("⚠️  Sistema Notícias - Nenhum arquivo encontrado na página")
                return []
            
            logger.info(f"📷 Sistema Notícias - Encontrados {len(arquivos_info)} arquivos")
            return arquivos_info
            
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao acessar pasta: {e}")
            return []

    def _extrair_info_arquivos_pagina(self, html):
        """Extrai informações dos arquivos do HTML"""
        try:
            arquivos_info = []
            
            file_ids = set()
            padroes_ids = [
                r'data-id="([a-zA-Z0-9_-]{25,})"',
                r'/file/d/([a-zA-Z0-9_-]{25,})',
            ]
            
            for padrao in padroes_ids:
                matches = re.findall(padrao, html)
                for file_id in matches:
                    file_ids.add(file_id)
            
            logger.info(f"🔍 Sistema Notícias - File IDs encontrados: {len(file_ids)}")
            
            for file_id in file_ids:
                nome_arquivo = self._encontrar_nome_por_file_id(html, file_id)
                
                if nome_arquivo:
                    download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
                    arquivos_info.append({
                        'id': file_id,
                        'nome_original': nome_arquivo,
                        'download_url': download_url
                    })
                    logger.info(f"✅ Sistema Notícias - Arquivo identificado: {nome_arquivo} (ID: {file_id})")
                else:
                    download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
                    arquivos_info.append({
                        'id': file_id,
                        'nome_original': f"arquivo_{file_id}.jpg",
                        'download_url': download_url
                    })
                    logger.info(f"⚠️  Sistema Notícias - Arquivo sem nome: {file_id} (usando fallback)")
            
            return arquivos_info
            
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao extrair informações: {e}")
            return []

    def _encontrar_nome_por_file_id(self, html, file_id):
        """Tenta encontrar o nome do arquivo correspondente a um file ID"""
        try:
            padrao1 = f'data-id="{file_id}"[^>]*data-title="([^"]+)"'
            match1 = re.search(padrao1, html)
            if match1:
                nome = match1.group(1)
                if nome and '.' in nome:
                    return nome
            
            padrao2 = f'data-id="{file_id}"[^>]*>([^<]+\.(jpg|jpeg|png|gif|webp))'
            match2 = re.search(padrao2, html, re.IGNORECASE)
            if match2:
                return match2.group(1)
            
            contexto = 500
            idx = html.find(f'data-id="{file_id}"')
            if idx != -1:
                inicio = max(0, idx - contexto)
                fim = min(len(html), idx + contexto)
                area_contexto = html[inicio:fim]
                
                padrao_nome = r'([a-zA-Z0-9_\-\s]+\.(jpg|jpeg|png|gif|webp))'
                matches_nome = re.findall(padrao_nome, area_contexto, re.IGNORECASE)
                if matches_nome:
                    return matches_nome[0][0]
            
            return None
            
        except Exception as e:
            logger.warning(f"⚠️  Sistema Notícias - Erro ao buscar nome para {file_id}: {e}")
            return None
        
    def _limpar_pasta_imagens(self):
        """Limpa todas as imagens da pasta de destino antes do download"""
        try:
            diretorio_imagens = Path(self.caminho_base_imagens)
            
            if not diretorio_imagens.exists():
                logger.info("📁 Sistema Notícias - Pasta de imagens não existe, criando...")
                diretorio_imagens.mkdir(parents=True, exist_ok=True)
                return 0
            
            # Contador de imagens removidas
            imagens_removidas = 0
            
            # Lista de extensões de imagem suportadas
            extensoes_imagem = ['.jpg', '.jpeg', '.png', '.gif', '.webp', '.bmp', '.tiff']
            
            # Remover todas as imagens
            for extensao in extensoes_imagem:
                for arquivo in diretorio_imagens.glob(f"*{extensao}"):
                    try:
                        arquivo.unlink()
                        imagens_removidas += 1
                        logger.info(f"🗑️  Sistema Notícias - Removido: {arquivo.name}")
                    except Exception as e:
                        logger.warning(f"⚠️  Sistema Notícias - Não foi possível remover {arquivo.name}: {e}")
            
            # Também remover arquivos com extensões em maiúsculo
            for extensao in extensoes_imagem:
                for arquivo in diretorio_imagens.glob(f"*{extensao.upper()}"):
                    try:
                        arquivo.unlink()
                        imagens_removidas += 1
                        logger.info(f"🗑️  Sistema Notícias - Removido: {arquivo.name}")
                    except Exception as e:
                        logger.warning(f"⚠️  Sistema Notícias - Não foi possível remover {arquivo.name}: {e}")
            
            logger.info(f"✅ Sistema Notícias - Limpeza concluída: {imagens_removidas} arquivos removidos")
            return imagens_removidas
            
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao limpar pasta de imagens: {e}")
            return 0

    def verificar_tipo_imagem(self, download_url):
        """Verifica se a URL realmente aponta para uma imagem"""
        try:
            response = self.session.head(download_url, headers=self.headers, timeout=10, allow_redirects=True)
            
            content_type = response.headers.get('Content-Type', '').lower()
            content_length = response.headers.get('Content-Length', '0')
            
            is_image = any(img_type in content_type for img_type in ['image/jpeg', 'image/jpg', 'image/png', 'image/gif', 'image/webp'])
            
            if is_image:
                logger.info(f"✅ Sistema Notícias - URL válida: {content_type} ({content_length} bytes)")
                return True
            else:
                logger.info(f"❌ Sistema Notícias - URL não é imagem: {content_type}")
                return False
                
        except Exception as e:
            logger.warning(f"⚠️  Sistema Notícias - Não foi possível verificar URL: {e}")
            return True

    def baixar_imagem_com_nome_original(self, arquivo_info):
        """Baixa uma imagem mantendo o nome original"""
        try:
            download_url = arquivo_info['download_url']
            nome_original = arquivo_info['nome_original']
            
            response = self.session.get(download_url, headers=self.headers, timeout=30, stream=True)
            response.raise_for_status()
            
            if "confirm=" in response.url or "google.com" in response.url:
                for hist in response.history:
                    if "download" in hist.url:
                        download_url = hist.url
                        break
                response = self.session.get(download_url, headers=self.headers, timeout=30, stream=True)
            
            nome_final = nome_original
            if 'Content-Disposition' in response.headers:
                content_disposition = response.headers['Content-Disposition']
                filename_match = re.findall('filename="(.+)"', content_disposition)
                if filename_match:
                    nome_header = unquote(filename_match[0])
                    logger.info(f"📄 Sistema Notícias - Nome do header: {nome_header}")
                    if '.' in nome_header and any(ext in nome_header.lower() for ext in ['.jpg', '.jpeg', '.png', '.gif', '.webp']):
                        nome_final = nome_header
            
            caminho_destino = self.pasta_destino_imagens / nome_final
            
            if caminho_destino.exists():
                logger.info(f"⚠️  Sistema Notícias - Imagem já existe: {nome_final}")
                return True
            
            logger.info(f"📥 Sistema Notícias - Baixando: {nome_final}")
            
            with open(caminho_destino, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            
            if caminho_destino.exists() and caminho_destino.stat().st_size > 0:
                tamanho_kb = caminho_destino.stat().st_size / 1024
                logger.info(f"✅ Sistema Notícias - Salvo: {nome_final} ({tamanho_kb:.1f} KB)")
                return True
            else:
                logger.error(f"❌ Sistema Notícias - Arquivo vazio ou corrompido: {nome_final}")
                if caminho_destino.exists():
                    caminho_destino.unlink()
                return False
            
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao baixar {arquivo_info['nome_original']}: {e}")
            return False

    def baixar_imagens_pasta_publica(self, pasta_url=None, limite=10):
        """Baixa todas as imagens de uma pasta pública mantendo nomes originais"""
        try:
            if pasta_url is None:
                pasta_url = f"https://drive.google.com/drive/folders/{self.file_id_imagens}"
            
            logger.info(f"📂 Sistema Notícias - Iniciando download da pasta: {pasta_url}")
            
            # Limpar pasta antes de baixar
            logger.info("🗑️  Sistema Notícias - Limpando pasta de imagens antes do download...")
            imagens_removidas = self._limpar_pasta_imagens()
            logger.info(f"✅ Sistema Notícias - {imagens_removidas} imagens antigas removidas")
            
            arquivos_info = self.obter_info_arquivos_pasta(pasta_url)
            
            if not arquivos_info:
                logger.error("❌ Sistema Notícias - Nenhum arquivo encontrado na pasta")
                return False
            
            arquivos_validos = []
            for arquivo in arquivos_info:
                if self.verificar_tipo_imagem(arquivo['download_url']):
                    arquivos_validos.append(arquivo)
            
            if not arquivos_validos:
                logger.error("❌ Sistema Notícias - Nenhuma imagem válida encontrada após verificação")
                return False
            
            logger.info(f"📷 Sistema Notícias - Imagens válidas encontradas: {len(arquivos_validos)}")
            
            arquivos_validos = arquivos_validos[:limite]
            logger.info(f"📥 Sistema Notícias - Baixando {len(arquivos_validos)} imagens...")
            
            sucessos = 0
            for i, arquivo_info in enumerate(arquivos_validos):
                logger.info(f"--- Sistema Notícias - [{i+1}/{len(arquivos_validos)}] ---")
                
                if self.baixar_imagem_com_nome_original(arquivo_info):
                    sucessos += 1
                
                time.sleep(1)
            
            logger.info(f"🎉 Sistema Notícias - Download concluído: {sucessos}/{len(arquivos_validos)} imagens baixadas")
            
            return sucessos > 0
            
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao baixar imagens da pasta: {e}")
            return False

    # ========== MÉTODOS DO COLETOR DE NOTÍCIAS ==========

    def detectar_formato_imagem(self, numero_imagem):
        """Detecta automaticamente o formato da imagem"""
        try:
            for formato in self.formatos_suportados:
                caminho_padrao = f"{self.caminho_base_imagens}/{self.imagem_padrao}{formato}"
                if Path(caminho_padrao).exists():
                    logger.info(f"🔄 Sistema Notícias - Usando imagem padrão: {self.imagem_padrao}{formato}")
                    return formato
            
            for formato in self.formatos_suportados:
                caminho_teste = f"{self.caminho_base_imagens}/{numero_imagem:02d}{formato}"
                caminho_absoluto = Path(caminho_teste)
                
                if caminho_absoluto.exists():
                    logger.info(f"🔍 Sistema Notícias - Imagem específica encontrada: {caminho_teste}")
                    return formato
            
            logger.info(f"⚠️  Sistema Notícias - Nenhuma imagem encontrada, usando padrão {self.imagem_padrao}.jpg")
            return '.jpg'
            
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao detectar formato da imagem {numero_imagem}: {e}")
            return '.jpg'

    def processar_imagens_noticias(self, dados_noticias):
        """Processa os números das imagens para criar caminhos completos com formato correto"""
        try:
            for noticia in dados_noticias['noticias']:
                if 'image' in noticia:
                    image_value = noticia['image']
                    
                    if isinstance(image_value, int) or (isinstance(image_value, str) and image_value.isdigit()):
                        numero = int(image_value)
                        
                        formato = self.detectar_formato_imagem(numero)
                        
                        caminho_padrao = f"{self.caminho_base_imagens}/{self.imagem_padrao}{formato}"
                        caminho_especifico = f"{self.caminho_base_imagens}/{numero:02d}{formato}"
                        
                        if Path(caminho_especifico).exists():
                            noticia['image'] = caminho_especifico
                            logger.info(f"🖼️  Sistema Notícias - Imagem específica: {noticia['image']}")
                        else:
                            noticia['image'] = caminho_padrao
                            logger.info(f"🔄 Sistema Notícias - Imagem padrão: {noticia['image']}")
                    
                    elif isinstance(image_value, str) and '.' in image_value:
                        partes = image_value.split('.')
                        if partes[0].isdigit():
                            numero = int(partes[0])
                            formato = f".{partes[1].lower()}"
                            
                            caminho_padrao = f"{self.caminho_base_imagens}/{self.imagem_padrao}{formato}"
                            caminho_especifico = f"{self.caminho_base_imagens}/{numero:02d}{formato}"
                            
                            if Path(caminho_especifico).exists():
                                noticia['image'] = caminho_especifico
                                logger.info(f"🖼️  Sistema Notícias - Imagem específica com formato: {noticia['image']}")
                            else:
                                noticia['image'] = caminho_padrao
                                logger.info(f"🔄 Sistema Notícias - Imagem padrão com formato: {noticia['image']}")
                    
                    elif isinstance(image_value, str) and image_value.startswith(('http://', 'https://')):
                        logger.info(f"🌐 Sistema Notícias - Imagem é URL: {noticia['image']}")
                    
                    elif isinstance(image_value, str) and ('/' in image_value or '\\' in image_value):
                        caminho_imagem = Path(image_value)
                        if not caminho_imagem.exists():
                            formato = self.detectar_formato_imagem(0)
                            caminho_padrao = f"{self.caminho_base_imagens}/{self.imagem_padrao}{formato}"
                            noticia['image'] = caminho_padrao
                            logger.info(f"🔄 Sistema Notícias - Caminho local não existe, usando padrão: {noticia['image']}")
                        else:
                            logger.info(f"📁 Sistema Notícias - Imagem é caminho local: {noticia['image']}")
                    else:
                        logger.warning(f"⚠️  Sistema Notícias - Formato de imagem não reconhecido, usando padrão: {type(image_value)} - {image_value}")
                        formato = self.detectar_formato_imagem(0)
                        caminho_padrao = f"{self.caminho_base_imagens}/{self.imagem_padrao}{formato}"
                        noticia['image'] = caminho_padrao
            
            return dados_noticias
            
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao processar imagens: {e}")
            return dados_noticias

    def verificar_imagens_existentes(self):
        """Verifica quais imagens existem no diretório (para debug)"""
        try:
            diretorio_imagens = Path(self.caminho_base_imagens)
            if diretorio_imagens.exists():
                imagens = list(diretorio_imagens.glob('*.*'))
                logger.info(f"📂 Sistema Notícias - Imagens encontradas no diretório ({len(imagens)}):")
                for img in imagens:
                    logger.info(f"   - {img.name}")
                
                imagem_padrao_encontrada = any(img.name.startswith(self.imagem_padrao) for img in imagens)
                if imagem_padrao_encontrada:
                    logger.info(f"✅ Sistema Notícias - Imagem padrão '{self.imagem_padrao}' encontrada")
                else:
                    logger.warning(f"⚠️  Sistema Notícias - Imagem padrão '{self.imagem_padrao}' NÃO encontrada")
            else:
                logger.warning(f"📂 Sistema Notícias - Diretório de imagens não existe: {diretorio_imagens}")
                diretorio_imagens.mkdir(parents=True, exist_ok=True)
                logger.info(f"📁 Sistema Notícias - Diretório de imagens criado: {diretorio_imagens}")
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro ao verificar imagens existentes: {e}")

    def baixar_noticias_google_drive(self):
        """Baixa o arquivo noticias.json do Google Drive"""
        try:
            logger.info("📥 Sistema Notícias - Iniciando download de notícias do Google Drive...")
            logger.info(f"🔗 Sistema Notícias - File ID: {self.file_id_noticias}")
            
            self.verificar_imagens_existentes()
            
            download_url = f"https://drive.google.com/uc?export=download&id={self.file_id_noticias}"
            logger.info(f"🌐 Sistema Notícias - URL: {download_url}")
            
            response = requests.get(download_url, timeout=30, allow_redirects=True)
            response.raise_for_status()
            
            logger.info(f"✅ Sistema Notícias - Download realizado - Status: {response.status_code}")
            logger.info(f"📦 Sistema Notícias - Tamanho do conteúdo: {len(response.content)} bytes")
            
            try:
                dados_noticias = response.json()
                logger.info("✅ Sistema Notícias - JSON decodificado com sucesso")
                
                logger.info("🔍 Sistema Notícias - Estrutura do JSON recebido:")
                for i, noticia in enumerate(dados_noticias['noticias']):
                    logger.info(f"   Noticia {i+1}: image={noticia['image']} (tipo: {type(noticia['image']).__name__})")
                
                if 'noticias' not in dados_noticias:
                    raise ValueError("Estrutura do JSON inválida - campo 'noticias' não encontrado")
                
                logger.info(f"📊 Sistema Notícias - Estrutura válida: {len(dados_noticias['noticias'])} notícias encontradas")
                
                dados_noticias = self.processar_imagens_noticias(dados_noticias)
                
                with open(self.noticias_path, 'w', encoding='utf-8') as f:
                    json.dump(dados_noticias, f, ensure_ascii=False, indent=2)
                
                logger.info(f"💾 Sistema Notícias - Arquivo salvo em: {self.noticias_path.absolute()}")
                logger.info(f"✅ Sistema Notícias - Notícias atualizadas com sucesso: {len(dados_noticias['noticias'])} notícias")
                return True
                
            except json.JSONDecodeError as e:
                logger.error(f"❌ Sistema Notícias - Erro ao decodificar JSON: {e}")
                logger.error(f"📄 Sistema Notícias - Conteúdo recebido: {response.text[:200]}...")
                return False
                
        except requests.exceptions.RequestException as e:
            logger.error(f"❌ Sistema Notícias - Erro de conexão ao baixar notícias: {e}")
            return False
        except Exception as e:
            logger.error(f"❌ Sistema Notícias - Erro inesperado ao baixar notícias: {e}")
            return False

    def _criar_noticias_padrao(self):
        """Cria arquivo de notícias padrão"""
        noticias_padrao = {
            "noticias": [
                {
                    "image": "1",
                    "category": "Sistema",
                    "title": "Sistema SIMA MA em Operação - TESTE",
                    "excerpt": "Sistema de Monitoramento Ambiental do Maranhão está coletando dados automaticamente. (Arquivo local)",
                    "date": datetime.now().strftime('%d/%m/%Y'),
                    "link": "#sistema"
                }
            ]
        }
        
        noticias_padrao = self.processar_imagens_noticias(noticias_padrao)
        
        with open(self.noticias_path, 'w', encoding='utf-8') as f:
            json.dump(noticias_padrao, f, ensure_ascii=False, indent=2)
        
        logger.info("📄 Sistema Notícias - Arquivo de notícias padrão criado")

    # ========== MÉTODOS DE INTEGRAÇÃO ==========

    def executar_sistema_condicional(self):
        """Executa o sistema completo apenas se a condição da planilha for 'sim'"""
        try:
            logger.info("🔄 SISTEMA NOTÍCIAS - INICIANDO VERIFICAÇÃO CONDICIONAL")
            logger.info("=" * 50)
            
            # Passo 1: Verificar condição na planilha
            logger.info("\n📋 SISTEMA NOTÍCIAS - VERIFICANDO CONDIÇÃO NA PLANILHA")
            deve_atualizar = self.verificar_condicao_atualizacao()
            
            if not deve_atualizar:
                logger.info("⏸️  SISTEMA NOTÍCIAS - MANTIDO: Condição não atendida, mantendo estado atual")
                return False
            
            # Passo 2: Baixar imagens da pasta pública
            logger.info("\n📥 SISTEMA NOTÍCIAS - BAIXANDO IMAGENS DA PASTA PÚBLICA")
            sucesso_imagens = self.baixar_imagens_pasta_publica()
            
            if not sucesso_imagens:
                logger.warning("⚠️  SISTEMA NOTÍCIAS - Falha no download de imagens, continuando com as existentes")
            
            # Passo 3: Baixar notícias
            logger.info("\n📰 SISTEMA NOTÍCIAS - BAIXANDO NOTÍCIAS ATUALIZADAS")
            sucesso_noticias = self.baixar_noticias_google_drive()
            
            if not sucesso_noticias:
                if not self.noticias_path.exists():
                    self._criar_noticias_padrao()
                    logger.info("📄 SISTEMA NOTÍCIAS - Usando notícias padrão (download falhou e não há arquivo local)")
                else:
                    logger.info("📄 SISTEMA NOTÍCIAS - Usando notícias locais (download falhou)")
            
            # Passo 4: Ler e exibir resultado final
            logger.info("\n📊 SISTEMA NOTÍCIAS - VERIFICANDO RESULTADO FINAL")
            noticias_finais = self.ler_noticias_atual()
            
            logger.info("\n🎉 SISTEMA NOTÍCIAS - ATUALIZAÇÃO COMPLETE CONCLUÍDA")
            return True
            
        except Exception as e:
            logger.error(f"❌ SISTEMA NOTÍCIAS - Erro na execução condicional: {e}")
            return False

    def ler_noticias_atual(self):
        """Lê e exibe as notícias atuais para verificação"""
        try:
            if self.noticias_path.exists():
                with open(self.noticias_path, 'r', encoding='utf-8') as f:
                    noticias = json.load(f)
                
                logger.info("=" * 50)
                logger.info("📰 SISTEMA NOTÍCIAS - NOTÍCIAS ATUAIS (COM IMAGENS PROCESSADAS):")
                logger.info("=" * 50)
                for noticia in noticias['noticias']:
                    logger.info(f"📌 {noticia['title']}")
                    logger.info(f"   🖼️  Imagem: {noticia['image']}")
                    logger.info(f"   🏷️  Categoria: {noticia['category']}")
                    logger.info(f"   📅 Data: {noticia['date']}")
                    logger.info(f"   📝 {noticia['excerpt']}")
                    logger.info(f"   🔗 Link: {noticia['link']}")
                    logger.info("-" * 40)
                return noticias
            else:
                logger.warning("❌ SISTEMA NOTÍCIAS - Arquivo de notícias não encontrado")
                return None
        except Exception as e:
            logger.error(f"❌ SISTEMA NOTÍCIAS - Erro ao ler notícias: {e}")
            return None

# ============================================================================
# SISTEMA INTEGRADO PRINCIPAL
# ============================================================================

class SistemaIntegrado:
    def __init__(self):
        self.coletor = ColetorDadosHidrologicos()
        self.sistema_noticias = SistemaNoticiasCompleto()
        logger.info("🎯 SISTEMA INTEGRADO INICIALIZADO")

    def executar_coletor_dados(self):
        """Executa o coletor de dados - SEMPRE executa"""
        try:
            logger.info("🚀 INICIANDO COLETOR DE DADOS (EXECUÇÃO AUTOMÁTICA)")
            return self.coletor.executar_coleta()
        except Exception as e:
            logger.error(f"❌ Erro no coletor de dados: {e}")
            return False

    def executar_sistema_noticias(self):
        """Executa o sistema de notícias - SOMENTE se condição for atendida"""
        try:
            logger.info("📰 INICIANDO VERIFICAÇÃO DO SISTEMA DE NOTÍCIAS")
            return self.sistema_noticias.executar_sistema_condicional()
        except Exception as e:
            logger.error(f"❌ Erro no sistema de notícias: {e}")
            return False

    def executar_sistema_seca(self):
        """Executa o sistema de monitoramento de seca"""
        try:
            logger.info("🌵 INICIANDO SISTEMA DE MONITORAMENTO DE SECA")
            
            # Executar download (condicional)
            download_executado = executar_download_seca()
            
            # Executar processamento (sempre)
            logger.info("\n🔄 INICIANDO PROCESSAMENTO DOS DADOS DE SECA...")
            resultados = processar_todas_pastas_faltantes()
            
            return resultados is not None
            
        except Exception as e:
            logger.error(f"❌ Erro no sistema de seca: {e}")
            return False

    def executar_ciclo_completo(self):
        """Executa um ciclo completo: PRIMEIRO notícias, DEPOIS coletor de dados, DEPOIS seca"""
        logger.info("🔄 INICIANDO CICLO COMPLETO DO SISTEMA INTEGRADO")
        logger.info("=" * 60)
        
        # PRIMEIRO: Sistema de Notícias (condicional)
        logger.info("\n📰 PRIMEIRO: EXECUTANDO SISTEMA DE NOTÍCIAS")
        inicio_noticias = time.time()
        noticias_executado = self.executar_sistema_noticias()
        tempo_noticias = time.time() - inicio_noticias
        
        # SEGUNDO: Coletor de Dados (SEMPRE executa)
        logger.info("\n📊 SEGUNDO: EXECUTANDO COLETOR DE DADOS")
        inicio_coletor = time.time()
        coletor_executado = self.executar_coletor_dados()
        tempo_coletor = time.time() - inicio_coletor
        
        # TERCEIRO: Sistema de Seca (condicional + processamento)
        logger.info("\n🌵 TERCEIRO: EXECUTANDO SISTEMA DE SECA")
        inicio_seca = time.time()
        seca_executado = self.executar_sistema_seca()
        tempo_seca = time.time() - inicio_seca
        
        # Estatísticas do ciclo
        logger.info("\n📈 ESTATÍSTICAS DO CICLO COMPLETO:")
        logger.info(f"   📰 Sistema de Notícias: {tempo_noticias:.2f} segundos")
        logger.info(f"   📊 Coletor de Dados: {tempo_coletor:.2f} segundos")
        logger.info(f"   🌵 Sistema de Seca: {tempo_seca:.2f} segundos")
        logger.info(f"   ⏱️  Tempo Total: {tempo_noticias + tempo_coletor + tempo_seca:.2f} segundos")
        logger.info("✅ CICLO COMPLETO FINALIZADO")
        logger.info("=" * 60)

    def iniciar_agendamento(self):
        """Inicia o agendamento automático com ciclo único a cada intervalo configurado"""
        logger.info("⏰ INICIANDO AGENDADOR DO SISTEMA INTEGRADO")
        
        # Agendar ciclo completo a cada intervalo configurado
        schedule.every(CONFIG['intervalo_execucao']).minutes.do(self.executar_ciclo_completo)
        logger.info(f"🔄 Agendado ciclo completo: a cada {CONFIG['intervalo_execucao']} minutos")
        
        # Executar uma vez imediatamente ao iniciar
        logger.info("🚀 Executando primeiro ciclo...")
        self.executar_ciclo_completo()
        
        # Manter o agendador rodando
        while True:
            try:
                schedule.run_pending()
                time.sleep(1)
            except KeyboardInterrupt:
                logger.info("🛑 Sistema integrado interrompido pelo usuário")
                break
            except Exception as e:
                logger.error(f"❌ Erro no agendador: {e}")
                time.sleep(60)

def main():
    """Função principal do sistema integrado"""
    try:
        sistema = SistemaIntegrado()
        
        print("\n" + "="*60)
        print("🌊 SISTEMA INTEGRADO SIMA MA")
        print("="*60)
        print(f"🔄 CICLO COMPLETO: A cada {CONFIG['intervalo_execucao']} minutos")
        print("   📰 PRIMEIRO: Sistema de Notícias (condicional)")
        print("   📊 SEGUNDO: Coletor de Dados (sempre executa)")
        print("   🌵 TERCEIRO: Sistema de Seca (condicional + processamento)")
        print("🎯 CONDIÇÃO de atualização")
        print(f"🏞️  Estações: {len(sistema.coletor.lista_estacoes)}")
        print("="*60)
        print("Pressione Ctrl+C para parar o sistema")
        print("="*60 + "\n")
        
        sistema.iniciar_agendamento()
        
    except Exception as e:
        logger.error(f"❌ Erro fatal ao iniciar sistema integrado: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()